# MACE-MP-0b2 small — DIMER E2E interatomic-potential fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/mace-materials-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/mace-materials-pipeline/blob/main/tutorials/mace_materials_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-mace--foundations%2Fmace--mp--0-ffcc4d?style=flat)](https://huggingface.co/mace-foundations/mace-mp-0) [![Upstream](https://img.shields.io/badge/Upstream-ACEsuit%2Fmace-181717?style=flat&logo=github&logoColor=white)](https://github.com/ACEsuit/mace) [![Paper](https://img.shields.io/badge/arXiv-2401.00096-b31b1b.svg)](https://arxiv.org/abs/2401.00096)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.2 — **standalone** (§4)  
**Capability:** energy, force and stress prediction for atomic structures and bounded fine-tuning of the foundation interatomic potential

**This notebook is standalone.** It carries the repository's package (3 modules under `src/mace_materials_pipeline/`, at revision `4101558ae1ad`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `e291ace2bfae073c3ebc7ae2f9479a525989baa7` (~68 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2.1); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime builds an isolated Python 3.12.12 environment from the hash-locked pins (torch, mace-torch, e3nn, ase, numpy, safetensors, huggingface-hub and their dependencies; the kernel's own packages are left alone, so no restart is needed), stages and digest-verifies the pinned MACE-MP-0b2 small source asset (67.6 MB pickled module from the Hub), statically audits every global and generated source string that pickle would execute and refuses anything outside the torch / e3nn / mace allow-lists, unpickles it exactly once to write a code-free JSON config + safetensors pair whose digests are pinned in the module, rebuilds the model from the installed mace-torch and loads it strictly, generates a deterministic 48-structure dataset in code (rattled, strained fcc Cu, Al and Cu₁₆Al₁₆ supercells labelled by ASE's EMT potential — no download), validates the structures and the dataset contract, splits them into train/validation/test sets (stratified by composition, since every generated structure is an independent draw), checks the frozen model for rotation equivariance, permutation invariance and analytic-versus-finite-difference forces, measures its zero-shot energy and force errors against a composition baseline and a zero-force baseline, calibrates the per-element reference energies and runs a bounded Adam fine-tuning of the readouts plus the last interaction block, evaluates energy and force errors on the held-out test split, predicts three freshly generated structures, exports the adapter as safetensors with a manifest, reloads that artifact into a fresh pipeline to verify prediction parity, and runs the Section 10 activity (a second, separate fine-tuning of the readouts only, also exported and reloaded). The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.2 §5). Building the isolated environment takes a few minutes; after it, the model work takes about four minutes on CPU and well under a minute on a GPU.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4, select that cell and choose **Runtime → Run after** (it re-runs Section 4 and every later cell). Supply one extended-XYZ file of labelled structures (one frame per structure, a `Lattice` and `pbc` when periodic, an `energy` in eV and a `forces` array in eV/Å per frame, and optionally `group=<id>` or `trajectory=<id>` on each frame's comment line) through the upload dialog on Colab, or by path in `BYOD_PATH` on any runtime (a path, when set, is used instead of the dialog). By default (`BYOD_SPLIT = 'auto'`) whole groups — your trajectory tags, or else whole compositions — go to one split, so near-duplicate frames never straddle train and test; the grouping is printed. Sections 5–7 return to the pinned foundation weights before they run, so the frozen-model rows and the adaptation start from the foundation model, not from the sample adaptation. BYOD outputs are written under `outputs/mace_materials_byod_*` and never overwrite the sample's. The expected schema, the size limits and the element support are stated in the Prerequisites and in Section 4, and the file stays inside this runtime. BYOD is optional and never part of the default path.

MACE-MP-0 is a universal machine-learned interatomic potential (Batatia et al., 2023): an equivariant message-passing model trained on Materials Project PBE/PBE+U trajectories that predicts a total energy, per-atom energies, forces and stress for structures of 89 elements. The *0b2 small* checkpoint is the 8.2 M-parameter, two-interaction, 128x0e variant with Agnesi radial transform and ZBL pair repulsion, released November 2024. Forces are the analytic gradient of the energy, so the model needs autograd even at inference — this notebook checks that the forces it returns really are that gradient.

The upstream artifact is a **pickled torch module** (`.model`), which the fleet asset specification treats as executable serialization. So this pipeline never serves it: Section 3 downloads the pinned file, verifies its digest, disassembles it statically (every global it would import, every fx-generated source it would `exec`, every TorchScript source it would compile), refuses anything outside the torch / e3nn / mace allow-lists, unpickles it exactly once, and writes a code-free JSON config + safetensors pair whose digests are pinned in the carried module. The model you run is rebuilt from the installed `mace-torch` and loaded with `strict=True`; the pickle is provenance, not runtime.

The tutorial dataset is generated in code and labelled with **ASE's effective-medium-theory potential (EMT)** — a real, deterministic interatomic potential for Cu and Al, but a different level of theory from the PBE data MACE-MP-0 was trained on. That is exactly the fine-tuning situation: the frozen model already predicts sensible forces, its energies sit on a different reference and its curvature differs, and the adaptation has to close the gap on held-out structures. Every structure of one composition has the same atoms, so a per-element energy regression (the composition baseline) cannot see the displacements at all.

**Who this is for.** A learner who knows basic Python and has used Colab or Jupyter, and wants to see how a pretrained interatomic potential is checked, measured against baselines and fine-tuned to a different level of theory. You should know what an atom's position, a periodic cell, an energy and a force are; no experience with MACE, equivariant networks or fine-tuning is assumed. Per-atom energies, MAE, equivariance, E0 calibration, adapters and the other terms are explained where they are first used and again in the **Glossary** at the end. A CPU runtime is enough.

**Input → Model → Output.**

| | Inference | Adaptation |
|---|---|---|
| Input | 1 to 32 structures per call, each `{symbols, positions, cell, pbc}` in Å (1 to 512 atoms, at most 4,096 atoms per call) | at least 8 labelled structures (an `energy` in eV and `forces` in eV/Å each), split into train / validation / test |
| Model | MACE-MP-0b2 small rebuilt from the converted pair, float64, forces as the gradient of the energy | the same model after a per-element reference-energy (E0) calibration, with the readouts, E0, scale/shift and the last interaction block trained by Adam |
| Output | per structure: total energy (eV), energy per atom, forces (eV/Å), stress (eV/Å³) for periodic cells — point predictions with **no uncertainty and no out-of-domain flag** | per-atom energy and force errors on the test split beside two baselines and the frozen model, and a safetensors adapter with a manifest |

**How to use this notebook.** Choose **Runtime → Run all**. Sections 1–3 are **infrastructure** — the isolated environment, the carried code and the model verification — and can be run without study; the code of Sections 1 and 2 is collapsed. Section 3's output is worth reading: it prints the pickle audit and the conversion. The learning path starts in Section 4. Form fields (`# @param`) are the only values meant to be edited; the defaults reproduce the recorded path. Each stage asks you to **predict** before it runs, and the next cell opens with **What to notice** and a collapsible **Check your reasoning** block with a worked answer from this repository's CPU run of the same stages (GPU numbers can differ in the last decimals). Section 10 is a **change-one-thing activity**. **Troubleshooting**, a **Glossary** and a **Conclusion** template are at the end. Your notes are optional and are not required submissions.

**Roadmap:** 4 generate, validate and split the structures → 5 physics checks on the frozen model → 6 baselines and the zero-shot error → 7 calibrate and fine-tune → 8 held-out evaluation → 9 new structures, adapter export and reload → 10 **change one thing: which blocks train** → conclude. Core concepts are Sections 5 and 6 (what the model returns and what a good error is); evaluation practice is Sections 4, 6 and 8 (split, baselines, held-out errors); engineering and reproducibility are Sections 1–3 and 9.

**Learning objectives:** inspect the carried pipeline, dataset and metrics modules; stage and digest-verify an immutable source asset, read a static audit of a pickled checkpoint and see it converted into a code-free serving pair; validate atomic structures and split a labelled dataset so that related structures stay on one side; check a foundation potential for equivariance, permutation invariance and gradient-consistent forces; measure zero-shot energy and force errors against two trivial baselines; calibrate reference energies and run a bounded fine-tuning with explicit hyperparameters; evaluate per-atom energy and force errors on an independent test split; predict new structures; export a safetensors adapter that reloads against the pinned base with verified parity; and compare two trainable scopes from the same starting point.

**This notebook does not demonstrate:** molecular-dynamics driving, geometry optimisation, phonons, the published MACE-MP-0 benchmarks, the medium and large checkpoints, multi-head replay fine-tuning against the Materials Project data, LAMMPS or CUDA-equivariance deployment, per-prediction uncertainty or out-of-domain detection, and any claim that an EMT-labelled fcc supercell stands in for a DFT dataset. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh Linux x86_64 runtime (Google Colab, Kaggle or Linux Jupyter); the notebook builds its own Python 3.12.12 environment, so the kernel's Python version does not matter. CPU is enough — a 32-atom cell is scored in under 0.1 s and each default fine-tuning takes about a minute — and CUDA is used automatically when present. The model runs in float64, as upstream ships it.
- **Knowledge:** what an interatomic potential, a periodic cell and a force are; why energies are compared per atom; and what MAE and RMSE mean (all repeated in the Glossary).
- **Executable serialization handled explicitly:** the pinned `.model` file is a pickle. It is digest-verified, statically audited against allow-lists (the audit digest is pinned) and unpickled **once** to produce the safetensors pair the model is actually loaded from. No Hub-hosted Python module is imported; `mace-torch` is installed from PyPI at a pinned version. One process-wide side effect to know: importing `mace` (Section 1) sets the environment variable `TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1`, which `e3nn` needs to load its own constants under current torch. For the rest of the session every `torch.load` call that does not pass `weights_only` explicitly unpickles arbitrary objects (`weights_only=False`) — including any you write yourself. This notebook loads its own files through safetensors only and unpickles the source asset exactly once, after the audit; if you load other checkpoints in this runtime, pass `weights_only=True` yourself.
- **Data contract:** structures are `{symbols, positions, cell, pbc}` in Å with optional `energy` (eV) and `forces` (eV/Å); elements limited to the 89 MACE-MP-0 supports (Z 1–83, 89–94); 1..512 atoms per structure, at most 32 structures and 4,096 atoms per prediction call (evaluation and adaptation process longer lists in chunks of that size); no two atoms closer than 0.5 Å (periodic images included); cell vectors under 200 Å and periodic cell heights of at least 1 Å. A dataset needs 8 to 1,000 labelled structures with unique names, and its split must leave at least 4 training, 1 validation and 1 test structure, with every validation and test element present in training; a grouped split also needs at least 3 groups. Section 4 refuses a dataset that breaks any of these, naming the rule. BYOD accepts extended XYZ.
- **Validation is structural, not chemical:** nothing checks that a structure is charge-neutral, near equilibrium or physically meaningful — a random cloud of supported atoms is scored without complaint, and EMT labels are only meaningful for the metals it parameterises.
- **No uncertainty, no domain flag:** every energy, force and stress is a point prediction. Nothing in this notebook estimates its uncertainty or flags a structure far from the training data; an unfamiliar chemistry is scored with the same confidence as a familiar one.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — proprietary alloy compositions or unpublished DFT datasets are exactly that. The default path uploads nothing.
- **External access:** the Hugging Face Hub only, to fetch the pinned `mace-foundations/mace-mp-0` snapshot (~68 MB in total) at revision `e291ace2bfae…`. No repository access and no credentials are required; nothing is installed from this repository. The isolated environment also needs PyPI (`pypi.org`, `files.pythonhosted.org`) for the pinned `uv` wheel and the 73 hash-locked packages, and the managed CPython 3.12.12 build (python-build-standalone) that `uv` downloads.

## 1. Install the pinned runtime (isolated environment)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

Hosted runtimes such as Colab import some packages, NumPy among them, before the first cell runs, and Python cannot swap a module that is already loaded. Installing the pins into the notebook's own Python would therefore leave mixed versions or require a manual restart. Instead, the next cell builds a separate environment (`dimer_isolated_env/`) and leaves the kernel's packages untouched: it downloads the pinned `uv` 0.12.15 wheel and refuses it unless its size and SHA-256 match, has `uv` install the managed CPython **3.12.12** (whatever Python the kernel itself runs), and installs the 73 packages of the carried hash-locked requirements (`tutorials/requirements-colab.lock.txt`, compiled from the pins below) with `--require-hashes --only-binary :all:`, so every package, direct or transitive, is the exact file that was locked. No repository code is installed. The lock holds manylinux x86_64 wheels, so the notebook supports **Linux x86_64 runtimes only** (Google Colab, Kaggle or Linux Jupyter); on any other platform the cell stops with that message. The printed dictionary names the isolated Python version and the kernel's.

In [ ]:
# @title Infrastructure: install the locked runtime into an isolated environment
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import hashlib
import io
import os
import platform
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'mace-torch==0.3.16',
    'e3nn==0.4.4',
    'ase==3.29.0',
    'numpy==2.5.3',
    'safetensors==0.8.0',
    'huggingface-hub==1.32.0',
]
MANAGED_PYTHON = '3.12.12'
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
LOCK_NAME = 'requirements-colab.lock.txt'
LOCK_SHA256 = 'c16a2f71e3656dbbe63e66b7455ebf197fcc28b10da2acb68a82728347295f4c'
LOCKED_PACKAGES = 73
# The hash-locked requirements, compiled from the PINS above with `uv pip compile --generate-hashes` for manylinux x86_64.
LOCK_TEXT = r'''# This file was autogenerated by uv via the following command:
#    uv pip compile pyproject.toml -c <ast-audio-classification-pipeline tutorials/requirements-colab.lock.txt @ 16eee39, versions only, without huggingface-hub> --override tutorials/requirements-colab-overrides.txt --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt
anyio==4.15.1 \
    --hash=sha256:6152fdbbf9a77fdec97731721bebf7c4c44f7c29b424b0065826173efc7ed101 \
    --hash=sha256:9f28306018cbd6d329e64a36d58256edff76dd996fe423bc957326e578b82a94
    # via httpx
ase==3.29.0 \
    --hash=sha256:7b9dd103f007810339c24acfee2f6b677c0c48443b21d3c98e52959246cf4ebf \
    --hash=sha256:ef4e2caa38169e3fbbc4164764a060d1877a6692519a4bed82521328eeb0d9aa
    # via
    #   mace-materials-pipeline (pyproject.toml)
    #   mace-torch
    #   matscipy
certifi==2026.7.22 \
    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \
    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   httpcore
    #   httpx
click==8.5.0 \
    --hash=sha256:255bc9599cf7748b4b1a446ccc735421bd08a2ae529a8b88597d3de5664ee360 \
    --hash=sha256:ba0d2089de75ea0310e2dde03160e6ca10009947fb95a182f9b54021bb272e34
    # via huggingface-hub
configargparse==1.8.0 \
    --hash=sha256:22a417f4d7b00149f0af82ef7c491f8ecc4b1d5454633fd319b386f5eb806f92 \
    --hash=sha256:bb25b307c3cd46a3e868e7f7aa51487eb8003a9f01cac3ae129b66ec26eb794c
    # via mace-torch
contourpy==1.4.0 \
    --hash=sha256:018227e134b73090b06f911e773a526c77c9af426be29065500ce096a5accd4c \
    --hash=sha256:072a702e2e178f4fcf96f04775d0c059e4c917925abf0d3208ebb6865df3c23d \
    --hash=sha256:0c7a4c2716a4e98342221954416a836cca77c996c14ddbf22b5f01d5d93ca09c \
    --hash=sha256:107ec46f7aa1664266d69b1181aadc953d58d39f7d8d648fa5b795d4a060b0de \
    --hash=sha256:181bea01bc742734ae672fa00c717d855dd35e1f029536406538c52e7b0cd73d \
    --hash=sha256:186ba929df36d61b6127da2e89cd1357e4cb79aca647381ab1a6feb0b152877b \
    --hash=sha256:196759a3e4db60e1546167e361090ebb6d199aa3aafc9e34ea17a5ac3b23e814 \
    --hash=sha256:20156f5a1ac4f8ce02656e39a61e82164a3d359796dc8026f75b062783d500e1 \
    --hash=sha256:20da2f86bfaed60dbba729fe738c8241b1dd22b6cf7e8cbbf30df290bd04ba28 \
    --hash=sha256:2deb580178ca19437a84bd77e4bc2cd91a8ccad212413a83c273900868b5978c \
    --hash=sha256:2e741a39dfc96babe1722561e81351aeec104526c57dbe91c167c1db606c2d55 \
    --hash=sha256:3863ef2e2b13fe93f8c0ebb08ee400cb07153b8b0e91c5acb26e4537f283634f \
    --hash=sha256:3e0a2392533e5e2abdf277c951047516acc8d2f3c42b2dd9dc34c907c9ebabf2 \
    --hash=sha256:404dbcd9233513dfd1323b66ea593fef90e899f658f6b8a9ed8e93bd0ca669db \
    --hash=sha256:417be048f7e122cefbe2a34c51a1f2b0410f8eb35796f794d45475ddb7872d9d \
    --hash=sha256:423bd5f4b3f11d54a8c597234e513382e3454bb106b8ec7ae32ba4ed63f8d99b \
    --hash=sha256:432e89f835cf01f8d2123a130a3126b17ee6e0348e4d62c0973872b30aa873bf \
    --hash=sha256:43c3ccbb32c6294b183dcc8e8c46dacf5ecef809497e3d48a5be298eef185ad0 \
    --hash=sha256:43d80072a32299bf945de0a6dd4e034ea0162e832261de2e07a274a5adbba9c9 \
    --hash=sha256:4bffcb2e8cc5c0e837631acab59dd04265fdac376550e7c3e310523877c68cb7 \
    --hash=sha256:510f7d93d94cf6ebf4e2cd0640bea16ab8affac7547230175b6745b596ce0599 \
    --hash=sha256:5450f091ac1be0be3ad3a2a3b3f23b5e443e78c670ced4fd347d626f92a28fd2 \
    --hash=sha256:5545ff38229c15d11d822fc6f7415b932da9f4975f53e305b7b96ebfb01f889c \
    --hash=sha256:5a4c89c7f38a0d7a94356d74e3391073c4325c4d47ad2fd065c3fe7dbae16c0c \
    --hash=sha256:61624f6722480aa7e168fd746164e7bfdf48f8ccb6542f1613741200dc37e2b1 \
    --hash=sha256:618137ec5778fb76d494c9ec854b104a377fddba0d128bef7136cce2c3bf0df8 \
    --hash=sha256:630d32f06cedf37b7f1dd2b6a12e4492826bd477a4c9c7b2e2fee6ce3a2cea76 \
    --hash=sha256:64039341e2d8804f1a13bda8c69083eab935167bbd7e856cf096241f17d5bd45 \
    --hash=sha256:6507a016976a75a15aee47809a34605de6244ccc056e6164bf9fb14b27eecad4 \
    --hash=sha256:6ced1670fafee703b8277ab1645072a226e74f167166eae146fb743916119b67 \
    --hash=sha256:6e697d94e69f499ff6bebb899cae97a58d5d14f0e1fe9568b43a0248d2f9af8c \
    --hash=sha256:6f963ea9f0d68d2b6a02d861e3c0aae09ae25ba21f243fcf281e9fed468b078d \
    --hash=sha256:738c44fa71735a617f36da58e32810512407d283d5d1bb5b1cecb00d7eeba7cf \
    --hash=sha256:758e7496cec3195fc28a28945eab3ceef348c47fb95c7afc2eca2f3901209c8c \
    --hash=sha256:78ed5c5f962b3e156109f0531b174a65ba80d13cc681a70227b12524c162e184 \
    --hash=sha256:79b06c60d5e569ce12c5da8f0c51217cdc484386e2349fa717c30fafb56d2871 \
    --hash=sha256:7f861af508fca2384eef392bc1d8377cfa349b2b854cb11b45d89c75780e2561 \
    --hash=sha256:80c7fc8e7ac217777ce7ba2db3cf8478652b5a4b335283111012ff9eaac9d842 \
    --hash=sha256:86d05cec773c9507a3950122e0e40ce77c23c75ceeb2fc189514e71893cbb34b \
    --hash=sha256:875f42444c9cf48d56f724f2637e60d0f73b3b12c9041e1484580a233edf9591 \
    --hash=sha256:90feb8da006803a95ba573fddc9536357d0c3faddc3a4c9e22816ca49af52878 \
    --hash=sha256:912c6afaa106e2f74ba22b30416b77ef7eb94e3bdc5a4df51ab147845dba9b6d \
    --hash=sha256:96019f059bcb3774acc0104be3360a57b36d33eaaa2e1d7f77c800ca8e07618b \
    --hash=sha256:9c0e07c691f3b3321913ed9b8161c50ea5f77fadd006f52f5e755359b0dcbfc5 \
    --hash=sha256:a03b32c2e7eda8b17757c022162c13c86f5c3d6f937dddf9ba3c3ff7b513953b \
    --hash=sha256:a1c8a74744fa746eeaa12f0f9ed7f8b4add9d8f1b14d95e3b5e133675eac888f \
    --hash=sha256:a3e67bc1a6a4618a1dac7c3053a9ffece5ddfa2046b2670b342cf430bb0b87d1 \
    --hash=sha256:a58b5f130afec61a093d743cdda435cc4047748b6791627e34a68f41ec3a9b07 \
    --hash=sha256:a5e2bb871b90cd7a52bdee63bf80ef7acb46398d27efc66d7679240016efc5aa \
    --hash=sha256:abd0f7e51ecc52bf8c95713bd80b1c17c516a71fa391c54475308a4377903c2b \
    --hash=sha256:b7794ea07cab575633daad8d6e963b662053391022294d3aacc1d9ba9ef54114 \
    --hash=sha256:b9dc493e7924e5aa32d89d0b8e50986280f3d4b284968a792787bd7bcffd3cad \
    --hash=sha256:be85d160e7c795113c2a93254dde2fc7e86b375322570eee78b420969426973c \
    --hash=sha256:bee6ff96653e0807cd0f8eaef00174a031e3a2effa5c20f49a5a8d574b05af23 \
    --hash=sha256:c76f3a5318164db7d9401132fc1b5364b784c613c93fa506e3b5e6d1bf353ec8 \
    --hash=sha256:c8be3392b84cf43373a488874dc1ebf46f60784a70977f92586e18019d71c7ed \
    --hash=sha256:c927ec747633e68c9920bcaad48b0bcefe648812b822f18c34e2d0e2270a3d2a \
    --hash=sha256:cc87f2ffa84a49a77d76cb792f28a48ff3b05a222aa0b2bddbd839ba78e1d5ca \
    --hash=sha256:dd59df9e2fb0aff8bd7fd1adb9f349b7c835245a9d0f18c2f2deeb537190f2b1 \
    --hash=sha256:ddf5a2596d716fd3793434844caf31abc7a40b1e8718431420c89858268fb909 \
    --hash=sha256:de503609fdb71f597634ca64fb4bd2f13d27c97f140e912b1e8ed93544840df0 \
    --hash=sha256:e21d1d4a9db5b3a793649c7ab0ddb4697f2818929eec871a7b9c45fce7b2a1ea \
    --hash=sha256:e439ab450c93455feb0218532ded3e946ec7a9b5f459069f122cfa44b89229f6 \
    --hash=sha256:ea09dc704029930cbd097f75cb0cc1db9852adbcf1a151fac7e9559f7bcbfd19 \
    --hash=sha256:ef47fef9a912c77e3d84b014f701e5a9340f25703e9ed3bcebe37f91fb69dc49 \
    --hash=sha256:ef9440f6f8506246269a82734f5ff9e2e4c5e775b3996fc883cc491c5257eca6 \
    --hash=sha256:f1219a8898523cba821085f2da8a1b962cc696325763f09d7f93538ba43b2d70 \
    --hash=sha256:f863c6100bf926cf47d13f3cd75f9bb8ebd98aaab230eeb4468b91f98f39a6b3 \
    --hash=sha256:fa1b787362a3856e63dd2b89449f6c384d55ee1beb8f94f4bcc871b40f02462c \
    --hash=sha256:fc9feef8f1f001c5b87decadc67c4a5d1eebb62ca39c4763d1237ff62cf2b707
    # via matplotlib
cuda-bindings==13.4.3 \
    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \
    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \
    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \
    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \
    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \
    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \
    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \
    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \
    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \
    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \
    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \
    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \
    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \
    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \
    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \
    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \
    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \
    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \
    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \
    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \
    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \
    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \
    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   torch
cuda-pathfinder==1.8.3 \
    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   cuda-bindings
cuda-toolkit==13.0.3.0 \
    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   torch
cycler==0.12.1 \
    --hash=sha256:85cef7cff222d8644161529808465972e51340599459b8ac3ccbac5a854e0d30 \
    --hash=sha256:88bb128f02ba341da8ef447245a9e138fae777f6a23943da4540077d3601eb1c
    # via matplotlib
e3nn==0.4.4 \
    --hash=sha256:51c91a84c1fb72e7e3600000958fa8caad48f8270937090fb8d0f8bfffbb3525 \
    --hash=sha256:87d99876abb362a6e07d555d10752ff2e20c2b7731d8928ebe5d9121fb019f84
    # via
    #   mace-materials-pipeline (pyproject.toml)
    #   mace-torch
filelock==4.0.9 \
    --hash=sha256:635e7d67fa92654eed444e75e9ca18426d34e77ad9c469bf4373f75a932f7b22 \
    --hash=sha256:9287fd61b99a806e5202be29a83034c0808a1b9830e820537c2f5773981f7eeb
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   huggingface-hub
    #   torch
fonttools==4.66.1 \
    --hash=sha256:058cd823b80bac59e64dfad9e3b6fcd677852f9a3804971bbf6b48cc611e785c \
    --hash=sha256:05aeb146451f37289f782c3c861f3d0f4b86c2dd2e4620b46683544c7406640e \
    --hash=sha256:05c0fff6b4a5d872ed89cab2c4f81060b86ace263903eb4e8d0edcac47a60dfa \
    --hash=sha256:08d8956e3ec990c75230d92f1630b215e8f3738c83a003421c22b31ebfd0ce15 \
    --hash=sha256:09ae73bd219e1245debd8376077a0fa6e03175e255c4f51bae5f6a271bfe384a \
    --hash=sha256:0dc6fd99cb8c30941036308b148da9432640442a6f26f36d71dad9be24cbd0e9 \
    --hash=sha256:1801fdad5600118327171e0e8aa79f7cc48831dd55ab36998c9de03bd5ffe6cd \
    --hash=sha256:261d8dc95845e751f975fe8d6075600593ee253470d46d1b84801688051b09f6 \
    --hash=sha256:2aeb745f2664eb811026997c95628071137a777ea2ad296deec9cb393f0b23cf \
    --hash=sha256:2c7340497cf53490293e0c2b61011e0191633022ede0a0a964a68157a98b0fb4 \
    --hash=sha256:2ce4c93160535761f22c80b2afbc96cabc09855363a5d1a5554265b8a4c85901 \
    --hash=sha256:2d320483928c7831f0139ecb361954a26b2e2a8995681200155835dd8cd4a7d5 \
    --hash=sha256:2d637468dac23aac0e223bd52e66f8faa3b0dfcef57435460fa2107e830226cd \
    --hash=sha256:3087a430722aba8de429c2539fd2a58a9cf05238cdfefd8626460001052ca878 \
    --hash=sha256:34378db9a398b59de18cc79d942f0a907c6fc6301945e065ec888202f607aa3f \
    --hash=sha256:36bb24d4b98faacaff04af1d5e0a4285feba6ed1da6728cd34b6b6deb6bbb934 \
    --hash=sha256:38ce8f5fbd5c17dd2153d47d7c8d4108f3deda3f2b4a79b60ddc470a58faded3 \
    --hash=sha256:43d1284c1964666ee833f2badd3017dc138f53d4889043ffca66c5ce4188f188 \
    --hash=sha256:53e5854ea8003efec34adc0863c18ce91da923018354d27366f7fee7db928d7a \
    --hash=sha256:56d41d650cb8fc6cfe1d85ed7c62a0a56cbeed07bc65ca795475b914d401312a \
    --hash=sha256:5de5d80fbc0e50ff794c244e8fb7afd3eadfe0fa232ba8b162b8c551df22fcb4 \
    --hash=sha256:60f5ea17aed4262630afa43f26997ceabd6417fa05dcedf54c665f5a29193e18 \
    --hash=sha256:64967c6ddb0d4c610dfd8cb1485981b2d27972ddfb7d4bbbd9e199d2a089c450 \
    --hash=sha256:668f092bc0de8902167df6a0d5c5aedc3b4f9e43cf88eea92e9b46a2bd3968f5 \
    --hash=sha256:66fad3b7874062c2a2692f0ae6dea56d24f01b778c7f191950ca3ff997e25a88 \
    --hash=sha256:6946fe7bfb28590a1fd4061a17609c9a843952deb65dcf30d1fe725070c3e7a4 \
    --hash=sha256:71c7ca1b5f46f5dd549f56b47d47c0b709217675c23d3a7bc6aa1a69b6d9bbae \
    --hash=sha256:72299346b96b9244dabcc051b24e4653da4edfda6105544cfb10ce856a1afaac \
    --hash=sha256:7234ae9e28db64273fbbfa72caebd0a97e3bdba6b05064114741b9539ef339d0 \
    --hash=sha256:7b8ff9e0edbcee2fbf7dff0c41b9041c1901c26acf64e23adb67495012df11de \
    --hash=sha256:7cf4f996f9b1cb549bff9ea4c50813988a26ec922c95cfa85c7e4f1270447e06 \
    --hash=sha256:7f49f2834f5d006fe0f3bb10fec73b261806c50941f0cfbc08294074ffc32210 \
    --hash=sha256:83572afe48733bad7a4a9c11721d3a726c2e976d82b063fc9bdd049d76955abd \
    --hash=sha256:8526b2b7ec4db6b81efb83438be52b1264eda9a4994d867163cfe8c65581ce8d \
    --hash=sha256:8aed2bbcd6216253ef1b015763593365ee8084f621dfa53bb957c19d5e05f7cd \
    --hash=sha256:90de3477394c73481d27d2b86091c1c736053ee13ff52c42f0e151948e8578c6 \
    --hash=sha256:9261ef507f2dd74203443a472b65b5a26429eb378f975016dec7dc7305b24898 \
    --hash=sha256:9ea6c93091cbf83161a544388746a0911550bd98cb911faca3591cf5ead166ac \
    --hash=sha256:b13c8c541ce0b794add3211b3641cc0e113d707f73e06235e6fe9731bd7c45a9 \
    --hash=sha256:b18803cbdef248e7ee1be59cb277fbbe1da1faaa6f726fa5d3557904e6a3d967 \
    --hash=sha256:b878c78b2af11b879bd4f26bb0d8bda2a4c64543fdd3f28efe2c80f97f043885 \
    --hash=sha256:b8b71db96d605784e2c5ebf0788a406018ea8fdd80338491f4c83613d5cd1fec \
    --hash=sha256:b913b8e9f7ca9bec44d1eb919f591c596c61041aa357c96be55ff93169859e91 \
    --hash=sha256:c258eba62260beb33c110b03a6912cefa3635239c4ab5615b7225fb6f7b85238 \
    --hash=sha256:c47299bca4b5acaaeb32100f77b944feea151de9ef1773365a410dc3d49b945b \
    --hash=sha256:c666fefdd5613a0e99aa4516e6ff4ef87aa86cf1c7ba12a73550f4770e46b750 \
    --hash=sha256:c724e56213494c6695335577822b2d1628d102e71614de8b7eb8e30886d6a314 \
    --hash=sha256:d3b5403e82d0c7659ff1d9f956e29a3a68d094f043e9f5bc0442796fc3a4fb58 \
    --hash=sha256:d4f76868aea9cc4ce47fdbeaa904c02ee7d85dd0ad095071ae77f0bda6e62cf5 \
    --hash=sha256:d84ac0bf776b68396185bd919dd29e633d94300660335efc40b55b294b886903 \
    --hash=sha256:d8f0a8f16c4f3a5a87ca971de2631792d8cb4d570951f2000acf712f157d40db \
    --hash=sha256:dbb7b950f8c02deaffb6968994691e8589d671b7ef8396bc9d5b5c0dfbb7292f \
    --hash=sha256:dfba62cc93199ba62c376f90f2a9147d92730d301e44f88e013e50ff5edf6193 \
    --hash=sha256:e1cde50b3ec84ca6fe63ca815de183dbecb88e8adf8ada82d8ea130ef12b2b43 \
    --hash=sha256:e7ea7a08547a453fa000db96ed5714a3dc7e2b4255b9243f897921f8c10c169a \
    --hash=sha256:eef76d5796e604f9d6753fa6d323c4eb9f4e0e43f1dcca553f3e6914f1667b64 \
    --hash=sha256:f08ab7f8461c37ecfdd29ad97fb0c0780b50501bd664bb0f46b6e83ed2b9d2a7 \
    --hash=sha256:fdf4afd75c643e60ef4a96fe64fc8a9def27d2a542112332371a9e5066885f9a
    # via matplotlib
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   huggingface-hub
    #   torch
gitdb==4.0.12 \
    --hash=sha256:5ef71f855d191a3326fcfbc0d5da835f26b13fbcba60c32c21091c349ffdb571 \
    --hash=sha256:67073e15955400952c6565cc3e707c554a4eea2e428946f7a4c162fab9bd9bcf
    # via gitpython
gitpython==3.2.0 \
    --hash=sha256:bd70c5ec05cd2b797423e7eb312147d2458d3cca92085888fba2213f85905537 \
    --hash=sha256:fb92310af6844d96adc95ca066ed2e617c00e1dbd146a326626c81e72e18cc2e
    # via mace-torch
h11==0.16.0 \
    --hash=sha256:4e35b956cf45792e4caa5885e69fba00bdbc6ffafbfa020300e549b208ee5ff1 \
    --hash=sha256:63cf8bbe7522de3bf65932fda1d9c2772064ffb3dae62d55932da54b31cb6c86
    # via httpcore
h5py==3.16.0 \
    --hash=sha256:099f2525c9dcf28de366970a5fb34879aab20491589fa89ce2863a84218bb524 \
    --hash=sha256:0f456f556e4e2cebeebd9d66adf8dc321770a42593494a0b6f0af54a7567b242 \
    --hash=sha256:15922e485844f77c0b9d275396d435db3baa58292a9c2176a386e072e0cf2491 \
    --hash=sha256:1677ad48b703f44efc9ea0c3ab284527f81bc4f318386aaaebc5fede6bbae56f \
    --hash=sha256:171038f23bccddfc23f344cadabdfc9917ff554db6a0d417180d2747fe4c75a7 \
    --hash=sha256:17d1f1630f92ad74494a9a7392ab25982ce2b469fc62da6074c0ce48366a2999 \
    --hash=sha256:1897a771a7f40d05c262fc8f37376ec37873218544b70216872876c627640f63 \
    --hash=sha256:18f2bbcd545e6991412253b98727374c356d67caa920e68dc79eab36bf5fedad \
    --hash=sha256:2b2c02b0a160faed5fb33f1ba8a264a37ee240b22e049ecc827345d0d9043074 \
    --hash=sha256:314b6054fe0b1051c2b0cb2df5cbdab15622fb05e80f202e3b6a5eee0d6fe365 \
    --hash=sha256:346df559a0f7dcb31cf8e44805319e2ab24b8957c45e7708ce503b2ec79ba725 \
    --hash=sha256:370a845f432c2c9619db8eed334d1e610c6015796122b0e57aa46312c22617d9 \
    --hash=sha256:39c2838fb1e8d97bcf1755e60ad1f3dd76a7b2a475928dc321672752678b96db \
    --hash=sha256:3e6cb3387c756de6a9492d601553dffea3fe11b5f22b443aac708c69f3f55e16 \
    --hash=sha256:3fae9197390c325e62e0a1aa977f2f62d994aa87aab182abbea85479b791197c \
    --hash=sha256:42108e93326c50c2810025aade9eac9d6827524cdccc7d4b75a546e5ab308edb \
    --hash=sha256:42b012933a83e1a558c673176676a10ce2fd3759976a0fedee1e672d1e04fc9d \
    --hash=sha256:43259303989ac8adacc9986695b31e35dba6fd1e297ff9c6a04b7da5542139cc \
    --hash=sha256:4c6ab014ab704b4feaa719ae783b86522ed0bf1f82184704ed3c9e4e3228796e \
    --hash=sha256:656f00e4d903199a1d58df06b711cf3ca632b874b4207b7dbec86185b5c8c7d4 \
    --hash=sha256:698dd69291272642ffda44a0ecd6cd3bda5faf9621452d255f57ce91487b9794 \
    --hash=sha256:719439d14b83f74eeb080e9650a6c7aa6d0d9ea0ca7f804347b05fac6fbf18af \
    --hash=sha256:7c4dd4cf5f0a4e36083f73172f6cfc25a5710789269547f132a20975bfe2434c \
    --hash=sha256:7e420b539fb6023a259a1b14d4c9f6df8cf50d7268f48e161169987a57b737ff \
    --hash=sha256:8389e13a1fd745ad2856873e8187fd10268b2d9677877bb667b41aebd771d8b7 \
    --hash=sha256:85b9c49dd58dc44cf70af944784e2c2038b6f799665d0dcbbc812a26e0faa859 \
    --hash=sha256:86385ea895508220b8a7e45efa428aeafaa586bd737c7af9ee04661d8d84a10d \
    --hash=sha256:8975273c2c5921c25700193b408e28d6bdd0111c37468b2d4e25dcec4cd1d84d \
    --hash=sha256:8c1eff849cdd53cbc73c214c30ebdb6f1bb8b64790b4b4fc36acdb5e43570210 \
    --hash=sha256:9300ad32dea9dfc5171f94d5f6948e159ed93e4701280b0f508773b3f582f402 \
    --hash=sha256:96b422019a1c8975c2d5dadcf61d4ba6f01c31f92bbde6e4649607885fe502d6 \
    --hash=sha256:9c9d307c0ef862d1cd5714f72ecfafe0a5d7529c44845afa8de9f46e5ba8bd65 \
    --hash=sha256:a0dbaad796840ccaa67a4c144a0d0c8080073c34c76d5a6941d6818678ef2738 \
    --hash=sha256:a6fbc5367d4046801f9b7db9191b31895f22f1c6df1f9987d667854cac493538 \
    --hash=sha256:bdef06507725b455fccba9c16529121a5e1fbf56aa375f7d9713d9e8ff42454d \
    --hash=sha256:c3f0a0e136f2e95dd0b67146abb6668af4f1a69c81ef8651a2d316e8e01de447 \
    --hash=sha256:c5313566f4643121a78503a473f0fb1e6dcc541d5115c44f05e037609c565c4d \
    --hash=sha256:df02dd29bd247f98674634dfe41f89fd7c16ba3d7de8695ec958f58404a4e618 \
    --hash=sha256:dfc21898ff025f1e8e67e194965a95a8d4754f452f83454538f98f8a3fcb207e \
    --hash=sha256:e06f864bedb2c8e7c1358e6c73af48519e317457c444d6f3d332bb4e8fa6d7d9 \
    --hash=sha256:e2c04d129f180019e216ee5f9c40b78a418634091c8782e1f723a6ca3658b965 \
    --hash=sha256:e4360f15875a532bc7b98196c7592ed4fc92672a57c0a621355961cafb17a6dd \
    --hash=sha256:ec86d4fffd87a0f4cb3d5796ceb5a50123a2a6d99b43e616e5504e66a953eca3 \
    --hash=sha256:fa48993a0b799737ba7fd21e2350fa0a60701e58180fae9f2de834bc39a147ab \
    --hash=sha256:faca8fb4e4319c09d83337adc80b2ca7d5c5a343c2d6f1b6388f32cfecca13c1 \
    --hash=sha256:fb1720028d99040792bb2fb31facb8da44a6f29df7697e0b84f0d79aff2e9bd3 \
    --hash=sha256:ff24039e2573297787c3063df64b60aab0591980ac898329a08b0320e0cf2527 \
    --hash=sha256:ffbab2fedd6581f6aa31cf1639ca2cb86e02779de525667892ebf4cc9fd26434
    # via mace-torch
hf-xet==1.6.0 \
    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \
    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \
    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \
    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \
    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \
    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \
    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \
    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \
    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \
    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \
    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \
    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \
    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \
    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \
    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \
    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \
    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   huggingface-hub
httpcore==1.0.9 \
    --hash=sha256:2d400746a40668fc9dec9810239072b40b4484b640a8c38fd654a024c7a1bf55 \
    --hash=sha256:6e34463af53fd2ab5d807f399a9b45ea31c3dfa2276f15a2c3f00afff6e176e8
    # via httpx
httpx==0.28.1 \
    --hash=sha256:75e98c5f16b0f35b567856f597f06ff2270a374470a5c2392242528e3e3e42fc \
    --hash=sha256:d909fcccc110f8c7faf814ca82a9a4d816bc5a6dbfea25d6591d6985b8ba59ad
    # via huggingface-hub
huggingface-hub==1.32.0 \
    --hash=sha256:b0c7c80561969d9cdacdd55fce67ba9584cca0b9d4ea80957a3a5c1445fac5c8 \
    --hash=sha256:ed70a45498abe86039df7c2f4e5f7575de524be908d3840e8f828d5525eafd6a
    # via mace-materials-pipeline (pyproject.toml)
idna==3.20 \
    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \
    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   anyio
    #   httpx
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   torch
kiwisolver==1.5.1 \
    --hash=sha256:007a5553dfc4f4e8d184f588a0200e2cd4b63a59cc8796df3c39909e679dc7a0 \
    --hash=sha256:0324cd2567259b7a095f6cf18a52b0ffc6f3de9e69528ff1bc0e7a37bd43ff1a \
    --hash=sha256:0627b9bceb9c3cdcf12b8a18655eedfed2692b038df27423383c120d0b7dc2d6 \
    --hash=sha256:06a6917674de9e0fe3f66f5430787f59a9f2ddb64af9b714eaec547e29ef5c19 \
    --hash=sha256:072bdb15a3c19a5b5dbc8f8fb1f4e1884bf4f3507eeb4cc6334401274d37a5c0 \
    --hash=sha256:0a4faea5c6db201c6a21391d2ac926ea97acf7dacdbc3c417189e1adb1a00837 \
    --hash=sha256:0ba9527afc80ae3d7814ed98b6572d02bf85eaf48065678342c5f0c6dab7a8c7 \
    --hash=sha256:0d8924877ce22e17326a99a418c3c82037da078df3c6a260b13eca677444e6e7 \
    --hash=sha256:0ebdef3eae5336568147c39a55be6a2036ffde53faa9ca2d978989ae7c2da12c \
    --hash=sha256:1209042a623ddfda5497e4066c7b77651dde8e1d3a9dd97599dc7e97f3b9b78c \
    --hash=sha256:16895f553ee6620a827d2da56b871f835fb70b9216cca5d188e885caf6e3bd23 \
    --hash=sha256:17851e5dad4484be0cbccbde3b15331deae036de9aebd45eed964487802b172f \
    --hash=sha256:1798e83840c3f627246104c4d8a9639c60fa068adf9ce92b61791781fa8a68c1 \
    --hash=sha256:18170a77ddfecf40ec60d0928268dc95880c881864e015a8f34094ed18b9b9ad \
    --hash=sha256:186884a58486651e3c217b6acea0a53eaa9498fdd472057c46f2f0fb5c25aad5 \
    --hash=sha256:18a0cfb124546a4c2e6087c5f3029c7f44b37c85b142e0ced71f73a7599ac208 \
    --hash=sha256:1983f0974a750a6f6556f368ba11105d1d8369c735b944747c9f12ae5aea7aae \
    --hash=sha256:1a7587dc335f2c0f5bd577fd0540bd16c66006bdb60f759a1059f025e6c4f071 \
    --hash=sha256:1acc7e5b7ef05e9da8bb70cd6c7c4513090213d2e1ad9720f599f0bf6c52aec5 \
    --hash=sha256:1d852545c4d0e35a72728d072cbaa59e2fa7dd84bdf01e068d670dd0ceb58eb6 \
    --hash=sha256:1ed0f5e49d0ceff8b72190824d9e59c062fbbc02c231b853112c78474b3f5ec2 \
    --hash=sha256:1fff05e239575b1481b6ed1a782f6fad616efbf1f0b1f44e6e85c4dfe426e483 \
    --hash=sha256:21e46b23a2da695c364124817bc01d970effd5483147f8d66a6a7167e3f6b851 \
    --hash=sha256:22d5e5aaad6be121f2515765e3b1c444352cb8eb4c86510801db8f2e50757316 \
    --hash=sha256:2551cf9917af48ee7c4b29cc82320489508cf96fd26a51f6fc124de661cd44c7 \
    --hash=sha256:255605693a483db7bd5c79f60437f7bf658f7f520d61aa42722e32257c941951 \
    --hash=sha256:26e8268480be5061d509e29669d59103c067a26377a56491630ece11762e3858 \
    --hash=sha256:27add358abe374ebaa3b8763ef380bc99051b5a4b18d94878366a9e4f59efef0 \
    --hash=sha256:2ae70bc59790d2af72a3f76f24b272403e135070340281108b447cb77ea70819 \
    --hash=sha256:2e10ae1bba1899188b33557c10d73affcc12033edd18adddb57d209039976a4c \
    --hash=sha256:3221f78211074f561c44ca42eac0619828171bec15a2c4cf6f7747d07df76e8e \
    --hash=sha256:34633ecf50d16187ab8e5528b7a2530f2feb4e23f300db4672538b51cfc5cd38 \
    --hash=sha256:34ec467940442c9943016fb2d4c81d1ba84351eeca2f1a78f8bc87f1ba0d414c \
    --hash=sha256:37f801b5d7cc0e5a548921308e059fd2b057bb42972b591cfa3049f95423c4ed \
    --hash=sha256:38f6e0deb4d0a4615efe0c4efc5990b06ae450ab50a0b321c0b078b6d238c083 \
    --hash=sha256:3c24cd69455e1b00ddf770c13b6e2c33e07d6dc3f2d34add0bf9277c5c6bbd46 \
    --hash=sha256:3cc210010fd2f438a3ed430b45f1b501fd13a8618bf984dc2c5ce5b69b78752e \
    --hash=sha256:3fa5855898f6d3d01b72ccd48a2d65cbdee301251603fefe34e2025bddba219c \
    --hash=sha256:416ba7ff9f233b7036689bb5a3783537e838ad483f63558d2a800f75afe738b1 \
    --hash=sha256:431dc224a1a92a5c8f582d96e505196a3b5997a7271076678da2dfde67b77e9a \
    --hash=sha256:43844c1a7ad6d723d5b5b4c4fc7f5bd399c40e288120d16257c7c9e8765c6e85 \
    --hash=sha256:44b8faef94f1857e77fa0238f3390ff1ac51d2ea20a487e2e452a59fd2b5f5ca \
    --hash=sha256:470d420f98d368d6f010633a20659b544c5fdfa5329e6b70219f2ef08fd4a7ef \
    --hash=sha256:482676e5bd48d70ac99d9fc78863469845421e01184fa83f1f9366dc49f7e974 \
    --hash=sha256:4d4ca09bf13cff792b1884f64b98ee6c2467930d632233be25c56b442d99f10e \
    --hash=sha256:5025e36fb4fb275cef0a4e30dbb11cb4ae61d1c83deb90189cb5d7e4cafd6b55 \
    --hash=sha256:509735237ae0d849e8a843551d423d2500d2e0a9ac1611a145658b29c0fb9f85 \
    --hash=sha256:534f02c1abb31ed6dbd3515545285c330b2f12d00fdb1fdb71658b9ca5a13a6a \
    --hash=sha256:5978c3340f16a35c30f8ab2fa7bcf559973c55f1a5ef6970e1f621acf3c4db13 \
    --hash=sha256:5b973887ff782cfd6b67c9904ad8ca542e0bc5e4961503408b423b5a688b4d38 \
    --hash=sha256:5c490db2168a508088f59140dd392556a54b8bd1048fc6383c8baff13c359673 \
    --hash=sha256:5d142e352eb13facc7dd047489aebdff6ba78576c239f1ea04931979caaf0567 \
    --hash=sha256:5daa1f19e097050b9c4d9a78fcc9263cb96c9dfae08037ddc1b7c4ad1889f2a2 \
    --hash=sha256:61e9a64c7635095a6bfe483e2ff055d437c59bd45f3617a228b37277f0185d62 \
    --hash=sha256:63fb7294b768f444eb4b068965f2662f28c2fd4161e23bd60fcf3ff27b74c046 \
    --hash=sha256:685929988b208a911f1285e2f8ed54210b0d681a3dc0f03e00d599d291986e7e \
    --hash=sha256:6a797a1cefc8b9c93170db580337e1fe3d011ad18b1299943231279406342048 \
    --hash=sha256:6b92f60017dda7d877fdc546438b5e28f31c523264f49cf5a48c1d0ce1a0dfbc \
    --hash=sha256:70ed9a45c7484d2b30cdacf60d220f494a1763b9fec1ad03285c6553fa0889f2 \
    --hash=sha256:719a35fa1156db3640555f95ebb94f60a444e64d1c69626b0edef5df78eba225 \
    --hash=sha256:74ad5c3dad54a4641b4c28cd15ded70899d04459c6c7aeacafea716be97cce6d \
    --hash=sha256:74ea337e0ec3f6f342a36a4f1b5cd94dd9affddcd28ba9aae2905af932ee8c6b \
    --hash=sha256:75d9b1cf8258462dbdc1eeda718c96ea7f079324c09067f6daabfcf37712b7fe \
    --hash=sha256:77a4c8187a5948d7f8795adb765a3c7b553d07d86d88e43038fc32fc1fb9a3f3 \
    --hash=sha256:7824b5e8bdbf0bccb4ccd37bbb115849a1dc45437fb4de8351385ed07c437ee0 \
    --hash=sha256:7d38b0c279c3032e8c9cc013b405c6df8e1668dbf15465779aa7f15f61201812 \
    --hash=sha256:7e9c01d3dd7ceba4d1d436cc021d40d592466e40b9bc7f5d83dc4e98a5c9cd8c \
    --hash=sha256:7fd82debf43c6acd0a94359d232f6bb516ee13f269a7993736a9ac9f988bb5d9 \
    --hash=sha256:824c3d763a05ea9e9003610145186b0e9848c7584a5575c79bac5a8e7cd80bad \
    --hash=sha256:828f75af2b0080c8a972e75f649ab46af008e92c6104a57a759157200b835b75 \
    --hash=sha256:83f78128fa28705fa85d01c59771c72fe81c11bd0e6155edbb9f818983a7d761 \
    --hash=sha256:876bbfd276473d3daffe30e8c975df4ed9429967b41a6cb362dbb5155b6f13ad \
    --hash=sha256:886fc26012f0e8b5f69d1cfe6d711f6b11f194621539bf8e6bb1c25c5dc82724 \
    --hash=sha256:8a34616dc2521cc8dc1d7d081734da63539f021ac0450ce950908340c6e7aa2f \
    --hash=sha256:8a708a47ade1fe19e8371d5da076bac0dd4b0a5a7985ad6c637f7f7e361b6baa \
    --hash=sha256:8af9b142ad719ae3a911ebf616bc4b78b32bbab84d6a40d3ad2f129670509957 \
    --hash=sha256:8bf4df63592c2a66b4f8edc5df2544998c288aa02f96ce0acd880cd1de8c8127 \
    --hash=sha256:8de6f2a4ce7e7bd27d23dd94abf0ccafe0e0e5cc9c764b0577191f2c25f08f26 \
    --hash=sha256:8f8fddb8e323bd6eee4e54e69a39243beab22689070f4c66b472c4cc88bb89d8 \
    --hash=sha256:8fca690b00c4c48f6c2a547b0160ed511357093a4e4c9b47e0fadf3128066d89 \
    --hash=sha256:9506e892bcc3b409831d363c6f53e5985e1c8d1f6f6b0256d00358684ff85378 \
    --hash=sha256:958254518717542d02d0688d0d20cbf771da5e415e6f49543f92481c850a4540 \
    --hash=sha256:95a02752aa032eef4aed01cda6d9b687c669bd0396bf4519eef8bba22a286720 \
    --hash=sha256:96c30002424670b5e1e46495c2b8cbffef39cf77c1d79e76462029d50339785b \
    --hash=sha256:98b208a7cc42c803445ef551d6753cc42a5ea13e9cab1ee66cd8b9cb70195330 \
    --hash=sha256:9b3092d8992a1d69b7a59c3e39f35e1b9be327a17f68a7c35fc17329e337d6f2 \
    --hash=sha256:9e51c119992ea8820706871c30a4642ec76de20ae82f9b50b9a45517d8e9f810 \
    --hash=sha256:a5716a33bfabb2c6ce27b6cf03253467b3804f83e215f4d202685cf93c6c9874 \
    --hash=sha256:a5a00665d1a0e26763a7338d7e911d4598fbc1d50dd0d6b7919b7dc6c5d6569f \
    --hash=sha256:a5ca5aebae78a0bc13c1943af4af615d4966c5b650b05d5aa83b50e427196fee \
    --hash=sha256:a7b85b2cc6ea45e5f7e8c9a30bc9fabd47cda09106cbb4b967335c3e6c43b69d \
    --hash=sha256:a83ee7107df13abe42a54a6654670eef9bb39425cf2e27f65e0007465e1286ab \
    --hash=sha256:aa7d00b1700966d2917e54d278aba86897890ca9276dd8b76cf6446b6c181b92 \
    --hash=sha256:ab620eb663952455271ac37f9aaad86b73c969c02f11f53cea405b38e96a4300 \
    --hash=sha256:ad8b9671348d7c8716715652ae11f85ed0eb99e265a2df2ca490577d69860b2c \
    --hash=sha256:aefe930d113798330e9462f7874542977869c0613cba3262e2de3a8d5dee8f3a \
    --hash=sha256:b03af77d77e50edba2030fd5f7c352ff209314b09030a3cba7c14edf9a09a444 \
    --hash=sha256:b390aec180a7c054919c04898835e1c77bced23ea8383eb2c570213bf25d1a86 \
    --hash=sha256:b3d78f7bb2b9d9a30345be1474b9aaa8685430b54afb51ba3639b5c6c11e9ed6 \
    --hash=sha256:b5664603a253efd3a75716d793d1d3a6a82723b61dc6db767b2460bbbeec4c0f \
    --hash=sha256:b69602970994a2ed8bbfa78c2f0394a7435226c6040489702d9f0a0ad0c07052 \
    --hash=sha256:b6ae6a0328f0bc035741820fdeecdcd67bf4694eee03972e843663107122f450 \
    --hash=sha256:bad20d4c69c851c982a1e3606f4c293edfd5a87885786c50082412240c4b1ffd \
    --hash=sha256:bb7c99f0673c03017a3ee01e54a5c2617a05468b11eabe513b0080e063ed95b1 \
    --hash=sha256:bebb89489b279b2f5661bbbb2abcc87bcd4a46607bb4a5c966f04f1db6b8df9a \
    --hash=sha256:bfd1de989b3330420e29de39352f5c049905c9e3ee67233a50d550e3d652c148 \
    --hash=sha256:c2306e8bb53601979fcb3fa09cc65e031876d9ae01eff2fcbcd7a84ef94d5bc1 \
    --hash=sha256:c3a4e41e3096bf1f0f1b76e2ffd6d828d6547f574f702d59bdbef7acfa59db9c \
    --hash=sha256:c6834b92dd2428e2dd85ef3d85f723d3c12f20aaf43a2ddd4f944ca25d833408 \
    --hash=sha256:c90d3022d8a94778939cda8638c6c8da8fa757b8958dad7ec868ce29c87681b8 \
    --hash=sha256:ca307d6c259e5c98d3cb9ade55342b47a6839762caf2536f3d7b46ee660cc82e \
    --hash=sha256:ca7f6fe0f37ca978a1e5eb7a3a68e6413f417e78e838324947ffd420202b198b \
    --hash=sha256:cb6fae641357ed2f6e533c0d3c6504a4a5703621a50c89459e46051d56b61140 \
    --hash=sha256:cdaeeb6c350106df6bf9d873395973e5f066a9713200b72cd64f55d0a3eafab6 \
    --hash=sha256:cea20da04494e662b83c872683bf4ff2345206043d036315ed0e924b652e7294 \
    --hash=sha256:cea90547bfd93807e0013a004dc76552be44fad3bc1cc2b38610a9e889ed098f \
    --hash=sha256:d09037ca068d784ebc4aec290ef952ca27ac15dd9c0b5801a88c6e1096b83e6b \
    --hash=sha256:d27c2123977cb9269c30a49ba45f03a4323017ef693e19db4ec9dbe1299a3002 \
    --hash=sha256:d50de98e8d807dc31822fff96f50293163a62418eb65487a21b42713d72ed0b7 \
    --hash=sha256:d66a64dd5dec136040ec2ae94aa026a912ee60fdd45bc28d3db30037fd809e88 \
    --hash=sha256:d79308fa689fac89cbcfbd4dbfc80b5f95c54c5a7fd4d194be221f9d33d026e6 \
    --hash=sha256:da3275833be0edbaf4830fae08bae3dc7219f40ce0c37eaa6c25825957e06612 \
    --hash=sha256:dc1a26b8e53395a01c2c611e58602fa47461f136fba7cd5542e6db6d64be1839 \
    --hash=sha256:dc23390afe9f4ef9ac3bcc72a03a56eebbde03f4c571a32cb38f859cff9a6524 \
    --hash=sha256:e05c2f7925f1d88778e53cb44f14e0223204a3bdd09a41664750363acfb1f2ef \
    --hash=sha256:e12dfea7f5fc2a34a9080efbf79c4c44eb380ec5b9c6fea09407e08f0d1e941d \
    --hash=sha256:e4e4523d6f336708d732516e6cfca7796cf3d96c9474eb5aecf6165f2f1fefc3 \
    --hash=sha256:e4e49f7e1a4e7191bdf9dc67a974db714501b1fc52c24324103d06a86abd5c08 \
    --hash=sha256:e68e151428b5384f766cd25739bf77c7e4a3dc93b5ded7a12118d9fbfdf78ab6 \
    --hash=sha256:e8e4d953faaded9ec7ede36824e9814082d22d4c7b1eafbfa079ecba8cd0d076 \
    --hash=sha256:ee9df1f0d77b9c6e94f4ac0fec533fbddd5ea3a327807f18d7b069ae019ded80 \
    --hash=sha256:f0a887b6565bbfe80efde2b7f6e8890d7d9bbdb11bdb17028a3690c32fe0621f \
    --hash=sha256:f0f4a42db92d6ec7677ab9d12830a2a8ec145a9c6d15db2b593466bc875c78d7 \
    --hash=sha256:f1303ef2eec81262a4b708c3e858afe58d7c75ad91c1c05266eda7673369859a \
    --hash=sha256:f1d56ec54d257d05e0b50f5780d967540cd07beeaf9e5f645b26d50cce79f4d8 \
    --hash=sha256:f4167e87b397f273dc2356fcf1eaf50a6bac51e6105f45103ef7129c8efb0255 \
    --hash=sha256:f76fc85bd054c806960f917ec0f329e24e436f1712267d90588e4c39890caa63 \
    --hash=sha256:f942903fde7363d1d879057ec5de01310efda2597161784d752fa9953a01a71a \
    --hash=sha256:f9b1c4900736e489a812c529100de4b8fb617d4db075e931e213c57424b83d9b \
    --hash=sha256:fc271a6f0a2126958f4090e5507b9da5848927dae331f8f763bd4aa642b3d2cd \
    --hash=sha256:febcce10f2bcdbb80b4ea919238a6a4ac13dbc4c7cadbe8d5d75c3682f8b5404
    # via matplotlib
lightning-utilities==0.15.3 \
    --hash=sha256:6c55f1bee70084a1cbeaa41ada96e4b3a0fea5909e844dd335bd80f5a73c5f91 \
    --hash=sha256:792ae0204c79f6859721ac7f386c237a33b0ed06ba775009cb894e010a842033
    # via torchmetrics
lmdb==3.0.0 \
    --hash=sha256:066dd62c91f245483f164d6bd09ad82de5f7a43be0bc524ef94919afc21db1d9 \
    --hash=sha256:06dda0723545e14d56ac7dcd6f582d9922c2ed5e16f1c3f8d0e670a96ec2ee65 \
    --hash=sha256:075d6a6afb7a8377f6d466b6044c79ddb969d80d0b11046846fa3640b30d8e1c \
    --hash=sha256:0979febf547d8a2fc10527caa247d2d470df61b475da5046f2586814f4226fe6 \
    --hash=sha256:0b6fa807feef09618a069bf6f25f619c5dbe977846d1b2e528473e3471aba946 \
    --hash=sha256:0f3c03ad20a235efe9753015268ae954e7ca14d439fe4bbf376c4fc7d677bb63 \
    --hash=sha256:169bf3966a529beeb1b5fcaada312e6cf50a71e0f1548166da9f1c087695b588 \
    --hash=sha256:231fc9ef677eb9317cf5a31752a7968e9da9010473b48cff273587915c6dedbc \
    --hash=sha256:3389df78f05b3af9811fb9fc63040a7112e0cc571212bcbe2fd5d0bba5339c35 \
    --hash=sha256:3e6c26010ddd5473d43e543beef3bd745f4bd5e8fdc7148de10d2b86603d0479 \
    --hash=sha256:3e7b803b40ea4c3cb0fa4bc331cf8c44c7d79f91c27e92609a8e90d2bd2f25f6 \
    --hash=sha256:4a2c08b76c9a00b28f2d28bbebd08687bb8343dda99a33e807a5b4842549b6ee \
    --hash=sha256:4a9243db25b116937412da87b8000a049c5cde3377f6111c87aa141b1cf0e3b6 \
    --hash=sha256:4f9c14d86f41de5676f72377df40bdff94d31585c1ffc41532f1b8f0011bf406 \
    --hash=sha256:56c6eeabfe4ddbec29c514e8fd95195957029ae5ebbbbddef943d42c0f0c0f94 \
    --hash=sha256:5991212e70e5d7e9addc7214861fa21af78ce7b0f90fd89a5e8f9dacffa28060 \
    --hash=sha256:5badcac838f9ce06601de42d02db7d9bed9276e2d99046074304d8eda72b7700 \
    --hash=sha256:5d66373b7ce51e8362ca677dfd6cf6b252fc6d64d8cf354b0aca1dd3b5c5bf8f \
    --hash=sha256:68c324cc0582afdc96e85df62616c6a5bc96ba0d0c0dd5072011214a1106b861 \
    --hash=sha256:70fe1ebc6aa679095295dd6096a42418c062a3cd572f083716bc255d1e6dd770 \
    --hash=sha256:7661f6f410fcf9fad16e3967823576bdb8001a4405cba93b950e52b640829f25 \
    --hash=sha256:86d1f9c0194f60fa67a1782c89903d63b6ba11dafcc357a1eb794b8e616559f3 \
    --hash=sha256:88004c2b4862e5ec67cec8a766d3955e0fd31a05400a521a6c86784988915e21 \
    --hash=sha256:89d146705771f817478b4b5ab44c6e2dd5b341c2d2ea13ce78b1a1e1850ee697 \
    --hash=sha256:8db6dc44a58d3dc12867e7b5685be71e8ae442c7997572c5dc8f55d4bfae4688 \
    --hash=sha256:954238443d6cd48618817adea790a6391d96aacf8be94762efdd9c127dcea244 \
    --hash=sha256:99dab02837f6254cc045d95cc5df721a2e5c07a6fcf08826d47328e553adcf8e \
    --hash=sha256:9d063bf15d94ae444fabc1bcf65e51452e96f14c3f21d0ec42163f41c56241a6 \
    --hash=sha256:9d97a9d2349941b89d8ddb054c4a9e5a8e398009e2e838ae1d6f35b858db4fb8 \
    --hash=sha256:9feccf2fe5d7826dd745618350f58f675093093da7187b976c2fa6942a23297a \
    --hash=sha256:af4b4518071adcc1c755f381fa49652fc5b27c11b3e6539ea76d5a6c0d64547e \
    --hash=sha256:ba3644422e5abe4e012369f6e19ea77993eabba54454889f07037c7e93f55f48 \
    --hash=sha256:bba67fbf79532bc6ad19bf98a2fc179b3c712e148820c1ced42d928a7a73228b \
    --hash=sha256:d34876ba920b8f2c7b30af2cd8de94133070242fa0b42dfbd26d508044681220 \
    --hash=sha256:d7513c11b949118d3b35f1e5d47fd71d6a0068078ccbf90000afc71a1c961e31 \
    --hash=sha256:f7545d6f78117419292eea27fb2b9f7551c840a29f7c4739924cebfe396e934b \
    --hash=sha256:f8f515911ea712a5a0ae0a51e331dcb5ab2265f355b89d328183fd257fdb9bc4 \
    --hash=sha256:fb6204aed79f394c1fa3a50374ff297de262829d2aa5cb6186fbfee18fc88131
    # via mace-torch
mace-torch==0.3.16 \
    --hash=sha256:5fbc84d319a92ab15e3e686533547a0afd2632261132e31cb0705a4347508935 \
    --hash=sha256:b80407edf6b2a1ec8523668c2a36852d20927ce1c3c56b70983a9f2dc53233ad
    # via mace-materials-pipeline (pyproject.toml)
markupsafe==3.0.3 \
    --hash=sha256:0303439a41979d9e74d18ff5e2dd8c43ed6c6001fd40e5bf2e43f7bd9bbc523f \
    --hash=sha256:068f375c472b3e7acbe2d5318dea141359e6900156b5b2ba06a30b169086b91a \
    --hash=sha256:0bf2a864d67e76e5c9a34dc26ec616a66b9888e25e7b9460e1c76d3293bd9dbf \
    --hash=sha256:0db14f5dafddbb6d9208827849fad01f1a2609380add406671a26386cdf15a19 \
    --hash=sha256:0eb9ff8191e8498cca014656ae6b8d61f39da5f95b488805da4bb029cccbfbaf \
    --hash=sha256:0f4b68347f8c5eab4a13419215bdfd7f8c9b19f2b25520968adfad23eb0ce60c \
    --hash=sha256:1085e7fbddd3be5f89cc898938f42c0b3c711fdcb37d75221de2666af647c175 \
    --hash=sha256:116bb52f642a37c115f517494ea5feb03889e04df47eeff5b130b1808ce7c219 \
    --hash=sha256:12c63dfb4a98206f045aa9563db46507995f7ef6d83b2f68eda65c307c6829eb \
    --hash=sha256:133a43e73a802c5562be9bbcd03d090aa5a1fe899db609c29e8c8d815c5f6de6 \
    --hash=sha256:1353ef0c1b138e1907ae78e2f6c63ff67501122006b0f9abad68fda5f4ffc6ab \
    --hash=sha256:15d939a21d546304880945ca1ecb8a039db6b4dc49b2c5a400387cdae6a62e26 \
    --hash=sha256:177b5253b2834fe3678cb4a5f0059808258584c559193998be2601324fdeafb1 \
    --hash=sha256:1872df69a4de6aead3491198eaf13810b565bdbeec3ae2dc8780f14458ec73ce \
    --hash=sha256:1b4b79e8ebf6b55351f0d91fe80f893b4743f104bff22e90697db1590e47a218 \
    --hash=sha256:1b52b4fb9df4eb9ae465f8d0c228a00624de2334f216f178a995ccdcf82c4634 \
    --hash=sha256:1ba88449deb3de88bd40044603fafffb7bc2b055d626a330323a9ed736661695 \
    --hash=sha256:1cc7ea17a6824959616c525620e387f6dd30fec8cb44f649e31712db02123dad \
    --hash=sha256:218551f6df4868a8d527e3062d0fb968682fe92054e89978594c28e642c43a73 \
    --hash=sha256:26a5784ded40c9e318cfc2bdb30fe164bdb8665ded9cd64d500a34fb42067b1c \
    --hash=sha256:2713baf880df847f2bece4230d4d094280f4e67b1e813eec43b4c0e144a34ffe \
    --hash=sha256:2a15a08b17dd94c53a1da0438822d70ebcd13f8c3a95abe3a9ef9f11a94830aa \
    --hash=sha256:2f981d352f04553a7171b8e44369f2af4055f888dfb147d55e42d29e29e74559 \
    --hash=sha256:32001d6a8fc98c8cb5c947787c5d08b0a50663d139f1305bac5885d98d9b40fa \
    --hash=sha256:3524b778fe5cfb3452a09d31e7b5adefeea8c5be1d43c4f810ba09f2ceb29d37 \
    --hash=sha256:3537e01efc9d4dccdf77221fb1cb3b8e1a38d5428920e0657ce299b20324d758 \
    --hash=sha256:35add3b638a5d900e807944a078b51922212fb3dedb01633a8defc4b01a3c85f \
    --hash=sha256:38664109c14ffc9e7437e86b4dceb442b0096dfe3541d7864d9cbe1da4cf36c8 \
    --hash=sha256:3a7e8ae81ae39e62a41ec302f972ba6ae23a5c5396c8e60113e9066ef893da0d \
    --hash=sha256:3b562dd9e9ea93f13d53989d23a7e775fdfd1066c33494ff43f5418bc8c58a5c \
    --hash=sha256:457a69a9577064c05a97c41f4e65148652db078a3a509039e64d3467b9e7ef97 \
    --hash=sha256:4bd4cd07944443f5a265608cc6aab442e4f74dff8088b0dfc8238647b8f6ae9a \
    --hash=sha256:4e885a3d1efa2eadc93c894a21770e4bc67899e3543680313b09f139e149ab19 \
    --hash=sha256:4faffd047e07c38848ce017e8725090413cd80cbc23d86e55c587bf979e579c9 \
    --hash=sha256:509fa21c6deb7a7a273d629cf5ec029bc209d1a51178615ddf718f5918992ab9 \
    --hash=sha256:5678211cb9333a6468fb8d8be0305520aa073f50d17f089b5b4b477ea6e67fdc \
    --hash=sha256:591ae9f2a647529ca990bc681daebdd52c8791ff06c2bfa05b65163e28102ef2 \
    --hash=sha256:5a7d5dc5140555cf21a6fefbdbf8723f06fcd2f63ef108f2854de715e4422cb4 \
    --hash=sha256:69c0b73548bc525c8cb9a251cddf1931d1db4d2258e9599c28c07ef3580ef354 \
    --hash=sha256:6b5420a1d9450023228968e7e6a9ce57f65d148ab56d2313fcd589eee96a7a50 \
    --hash=sha256:722695808f4b6457b320fdc131280796bdceb04ab50fe1795cd540799ebe1698 \
    --hash=sha256:729586769a26dbceff69f7a7dbbf59ab6572b99d94576a5592625d5b411576b9 \
    --hash=sha256:77f0643abe7495da77fb436f50f8dab76dbc6e5fd25d39589a0f1fe6548bfa2b \
    --hash=sha256:795e7751525cae078558e679d646ae45574b47ed6e7771863fcc079a6171a0fc \
    --hash=sha256:7be7b61bb172e1ed687f1754f8e7484f1c8019780f6f6b0786e76bb01c2ae115 \
    --hash=sha256:7c3fb7d25180895632e5d3148dbdc29ea38ccb7fd210aa27acbd1201a1902c6e \
    --hash=sha256:7e68f88e5b8799aa49c85cd116c932a1ac15caaa3f5db09087854d218359e485 \
    --hash=sha256:83891d0e9fb81a825d9a6d61e3f07550ca70a076484292a70fde82c4b807286f \
    --hash=sha256:8485f406a96febb5140bfeca44a73e3ce5116b2501ac54fe953e488fb1d03b12 \
    --hash=sha256:8709b08f4a89aa7586de0aadc8da56180242ee0ada3999749b183aa23df95025 \
    --hash=sha256:8f71bc33915be5186016f675cd83a1e08523649b0e33efdb898db577ef5bb009 \
    --hash=sha256:915c04ba3851909ce68ccc2b8e2cd691618c4dc4c4232fb7982bca3f41fd8c3d \
    --hash=sha256:949b8d66bc381ee8b007cd945914c721d9aba8e27f71959d750a46f7c282b20b \
    --hash=sha256:94c6f0bb423f739146aec64595853541634bde58b2135f27f61c1ffd1cd4d16a \
    --hash=sha256:9a1abfdc021a164803f4d485104931fb8f8c1efd55bc6b748d2f5774e78b62c5 \
    --hash=sha256:9b79b7a16f7fedff2495d684f2b59b0457c3b493778c9eed31111be64d58279f \
    --hash=sha256:a320721ab5a1aba0a233739394eb907f8c8da5c98c9181d1161e77a0c8e36f2d \
    --hash=sha256:a4afe79fb3de0b7097d81da19090f4df4f8d3a2b3adaa8764138aac2e44f3af1 \
    --hash=sha256:ad2cf8aa28b8c020ab2fc8287b0f823d0a7d8630784c31e9ee5edea20f406287 \
    --hash=sha256:b8512a91625c9b3da6f127803b166b629725e68af71f8184ae7e7d54686a56d6 \
    --hash=sha256:bc51efed119bc9cfdf792cdeaa4d67e8f6fcccab66ed4bfdd6bde3e59bfcbb2f \
    --hash=sha256:bdc919ead48f234740ad807933cdf545180bfbe9342c2bb451556db2ed958581 \
    --hash=sha256:bdd37121970bfd8be76c5fb069c7751683bdf373db1ed6c010162b2a130248ed \
    --hash=sha256:be8813b57049a7dc738189df53d69395eba14fb99345e0a5994914a3864c8a4b \
    --hash=sha256:c0c0b3ade1c0b13b936d7970b1d37a57acde9199dc2aecc4c336773e1d86049c \
    --hash=sha256:c47a551199eb8eb2121d4f0f15ae0f923d31350ab9280078d1e5f12b249e0026 \
    --hash=sha256:c4ffb7ebf07cfe8931028e3e4c85f0357459a3f9f9490886198848f4fa002ec8 \
    --hash=sha256:ccfcd093f13f0f0b7fdd0f198b90053bf7b2f02a3927a30e63f3ccc9df56b676 \
    --hash=sha256:d2ee202e79d8ed691ceebae8e0486bd9a2cd4794cec4824e1c99b6f5009502f6 \
    --hash=sha256:d53197da72cc091b024dd97249dfc7794d6a56530370992a5e1a08983ad9230e \
    --hash=sha256:d6dd0be5b5b189d31db7cda48b91d7e0a9795f31430b7f271219ab30f1d3ac9d \
    --hash=sha256:d88b440e37a16e651bda4c7c2b930eb586fd15ca7406cb39e211fcff3bf3017d \
    --hash=sha256:de8a88e63464af587c950061a5e6a67d3632e36df62b986892331d4620a35c01 \
    --hash=sha256:df2449253ef108a379b8b5d6b43f4b1a8e81a061d6537becd5582fba5f9196d7 \
    --hash=sha256:e1c1493fb6e50ab01d20a22826e57520f1284df32f2d8601fdd90b6304601419 \
    --hash=sha256:e1cf1972137e83c5d4c136c43ced9ac51d0e124706ee1c8aa8532c1287fa8795 \
    --hash=sha256:e2103a929dfa2fcaf9bb4e7c091983a49c9ac3b19c9061b6d5427dd7d14d81a1 \
    --hash=sha256:e56b7d45a839a697b5eb268c82a71bd8c7f6c94d6fd50c3d577fa39a9f1409f5 \
    --hash=sha256:e8afc3f2ccfa24215f8cb28dcf43f0113ac3c37c2f0f0806d8c70e4228c5cf4d \
    --hash=sha256:e8fc20152abba6b83724d7ff268c249fa196d8259ff481f3b1476383f8f24e42 \
    --hash=sha256:eaa9599de571d72e2daf60164784109f19978b327a3910d3e9de8c97b5b70cfe \
    --hash=sha256:ec15a59cf5af7be74194f7ab02d0f59a62bdcf1a537677ce67a2537c9b87fcda \
    --hash=sha256:f190daf01f13c72eac4efd5c430a8de82489d9cff23c364c3ea822545032993e \
    --hash=sha256:f34c41761022dd093b4b6896d4810782ffbabe30f2d443ff5f083e0cbbb8c737 \
    --hash=sha256:f3e98bb3798ead92273dc0e5fd0f31ade220f59a266ffd8a4f6065e0a3ce0523 \
    --hash=sha256:f42d0984e947b8adf7dd6dde396e720934d12c506ce84eea8476409563607591 \
    --hash=sha256:f71a396b3bf33ecaa1626c255855702aca4d3d9fea5e051b41ac59a9c1c41edc \
    --hash=sha256:f9e130248f4462aaa8e2552d547f36ddadbeaa573879158d721bbd33dfe4743a \
    --hash=sha256:fed51ac40f757d41b7c48425901843666a6677e3e8eb0abcff09e4ba6e664f50
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   jinja2
matplotlib==3.11.2 \
    --hash=sha256:01dc8eaaab5a9fce9ff615eca82345728f289e4715b186ee10c6d85272fc26bb \
    --hash=sha256:02329432ae5c6af87cf208ee575b701d698bdf0b1a3bb28cc6d53c36e967e575 \
    --hash=sha256:07d9b9fa60cd4c393692f50d0bb03123242ddf61c99bb0e95e75feb354e7c1a8 \
    --hash=sha256:116cdb0eb0eb5644fc98eb2975d4b4dd4ad35e5c8e6b851c22e6976f371f7ab5 \
    --hash=sha256:1944895967f87c84c9b4bad29a707b31f5b36df4a3a2339ea1f8ea3ce5105539 \
    --hash=sha256:1a3040b209f3968b4e84161df7b174f07a9fad33b0f2d7e48ea3bbd3075e2863 \
    --hash=sha256:1b9a7ad579856284135e401ecc918c5f8a017ee30539298862a109f51b971710 \
    --hash=sha256:254d4ddb2fa8df3b4c689c0c306063aee10521df82cfb438185e499c75fe37c1 \
    --hash=sha256:2a8285cea8ef4d92aa041d1c33788bcae82248503300f93f1ca2136b9049452f \
    --hash=sha256:30ec15d7eefee71de16b7c689b42ba49715a4644650b76a6c7c70d79daf24e91 \
    --hash=sha256:399fef672f7046ef7d6a57572b2a6f9845f3f5afcff04e7b2df7a363a9f42190 \
    --hash=sha256:3aa4b8516fd26659e4363abbf317c703d9116496c5db2e9d0609a2866dd39dd2 \
    --hash=sha256:3da3bc0cbf7245e7db72cc6d29d12c5abef72cb73059c73b945f14e3545f3eb2 \
    --hash=sha256:3e8576f7c47e02fd4f21f44171302d2d1d58d4471d46da3d71fe8899d19539d9 \
    --hash=sha256:57b9ea60a835937c2012861923cbb91f47db8565775d326d1c42fc926aa10351 \
    --hash=sha256:5e1e923a3fc3326b99ec0a6ff1ab1338ac6c6cc62ad9d8a9c944197c7f8c6221 \
    --hash=sha256:643ff850d8e0f5b8319337f87ed3cb59506afb3df3cc48de777d85871233be7b \
    --hash=sha256:75b6d88402770e181b5d06a67dda4129c31da7d05004d63a21c310ec78d1b83c \
    --hash=sha256:79a258f58253dfa025af80a9e9bb228d75fced007f0e93ae7423fefbde81a74d \
    --hash=sha256:7d43ff8cebb50840648cb6429b2228621dbd709010f3117b3740abb20abf21c0 \
    --hash=sha256:7e5a90f8a707ebb6004a713b1cff091a40cc5df4c7c1cb165e5a505ebc11c292 \
    --hash=sha256:7ef7a53b66780e5d942923724f08577fbc5be1f7322da0f0f3f9dcaa45dd803d \
    --hash=sha256:854df8d7dfe9fdffcbaa6f39e44a6c24b409cb4d7561fbc09213b157d833f6a6 \
    --hash=sha256:894a9cbbecbe30ae6787d464df2e8fc7a8d475cfc68f87c3029f7c11152899b3 \
    --hash=sha256:8c8255de28f986d935a64c9ca71c0ec2d2f41d355691f5ea684725dc91413f71 \
    --hash=sha256:930efb28f59fda124e39265d177bab302625297bb147d2910de725a2fc2aef54 \
    --hash=sha256:a24d5fd36e4f0e742c3851dcd20810e56a95633a342e4bf6cb591c678e8fe61f \
    --hash=sha256:a6939df7567114b6bac7f4c5e06c84a67f197c1b2f2e4b234d4eecb3bfec9482 \
    --hash=sha256:a8756cc73d9af9a7fe0deb54ea2e75ef73b01d9e575877e72acad5458e660943 \
    --hash=sha256:af2661f6ac6bbd1d081996f54fdd9385715c625ace8cec0869055c0cfbf38981 \
    --hash=sha256:bbf1062991d826ed27e2144f3afa4461afbb8ff56e8f703043e191a8163b1ee9 \
    --hash=sha256:bc067c462a86f0e57bf52fc6e058d90171a5420007dac00c46e90153050c69a7 \
    --hash=sha256:bf3fe71fbfb8ec0e310e0bc8537c3405a01f38f25f9394ed2135e6202fed542b \
    --hash=sha256:c0b83f044ce10a98027b105b3931548719a6e8c7ef986b4362651e0b5367c8dc \
    --hash=sha256:c27e577ece613ea12a0790e00b4eb80d901c59a3e16cad474f31d8b1529690b9 \
    --hash=sha256:c5c1c68ee401fc98271263410f0e5ce88285abacf7627132914e8adf3d70ff43 \
    --hash=sha256:c9721f81275499da1feeb36a2cf8192ea086283b3bd16b7dc4c9d7aedb7396d6 \
    --hash=sha256:cc82dde2a0d3e3ad472edce04897ad7146b8d8bfd1df8a32992eebb81af18fdc \
    --hash=sha256:cec596316640f2b394b8f0daa0ea61a8eae82d017b620b9f202befb972a59ea4 \
    --hash=sha256:cf41ecd1b0c0b6f7177ed965a54c2afbe888715c7cf6054dc12d53bc1494002c \
    --hash=sha256:d3304eb5a59442a8867f6920d484591c0fa09ffc29e9260be2feec3351e25869 \
    --hash=sha256:d480038c83691532ed52ff3147db51fa902fc78cb2d8349993a1cdb684435bff \
    --hash=sha256:df4f7784aca81a94f254c0a2767d592ee25f407e488f5fa7203e51093fb6ca27 \
    --hash=sha256:e43b188f0a5b75447bcc197728258166aa64365770ed1caa36595a5e1ca4bbba \
    --hash=sha256:e60cf3047a51edecdc4535a9196bbd6a732936b8e9f8184aabf4d16165884aaf \
    --hash=sha256:eac4b07d4e3743b172451e122ea964f72f152879d4f9adcf3f3d33e518f12ead \
    --hash=sha256:eb3712dc9b464793de0a4e42a7313d50293c751f94bddaf7332a1bc71bccdda9 \
    --hash=sha256:ecea603dd2fbf8242fd31a305a8b12a4ece2de28096870c65fdd0d1e35b8d9a6 \
    --hash=sha256:ef31985c4dedb5f1424e1aec6849a47dd37689cb7fa3c20b1b82187f26806261 \
    --hash=sha256:ef752769cd962f39ea0b6ffc82d1ea43a0012c5a6157c7a075212fa509cfcff2 \
    --hash=sha256:f25446b2981717dca9786bac841cb3fd7efb568e3e3c755dd980481c5cb9228d \
    --hash=sha256:f2ac30cf5eb5dff1b584627ae0b0e1186551a4f69ae3c75073911da497a29170 \
    --hash=sha256:fea03cf56568cc1cba08b470be6a0559e71c3a5b688d54b7179bb35ba23d0821
    # via
    #   ase
    #   mace-torch
matscipy==1.3.0 \
    --hash=sha256:24dfb15c199f9ed5e602ed4b453906eb32eef7b9df1fbaa74cba9bc0ca8396fe \
    --hash=sha256:25c7b964cd56942f1017b97ad52305eb7ea6323e0a27d97d67f014e53f7745cf \
    --hash=sha256:2d0a7646d421dc2ad6005a7ada110057133d460a35404233e026ea721e09500a \
    --hash=sha256:2db35794d5b0f3d7499fcc3ce803616e6056e9e22a0d92087d20b57eab9dd0a5 \
    --hash=sha256:3b8a8488c884554ee7a60e17a09939d8f6707840db914503589c1b35d62f0e86 \
    --hash=sha256:3f49a6af5a1c5cdc5a8530e44e861b8dbaea94dd85a32ca54eecf07b492612ba \
    --hash=sha256:58845e35f3c7cce9955627885d0b48905fc8ed8d7baf9d0b6be4ae64710af003 \
    --hash=sha256:5a88736e58cce92e90d2a070b583f4583b990b1e4d33b39c05a2cb13d6a0a457 \
    --hash=sha256:5deefde098397ef8f44556c7d4cbbc480e4472b7207b8f7873b2332696614312 \
    --hash=sha256:5fe01eb59c394ed851d2c61c369cbb9312a8380007dff59d4b0f888ef5a180db \
    --hash=sha256:5febc6d3cbf464079879de7122289f3ac400f422d0dd1a3a0c66a3ebfbf18d79 \
    --hash=sha256:60df5a90e5fc0db57e2518dfa08cf0ccb75dbae28b1c929513991f1d7dba9620 \
    --hash=sha256:60fa922416ce73114d20e23a4a658f3948a665bcced2f4c98d225d297c0684e0 \
    --hash=sha256:6b36667299384ee4d543e7f67beb9886ea02b0d9c8c3acded5df85112e0112e9 \
    --hash=sha256:726e43502c5ad5b65b9ed738897466a711c6ba03388099af79e756490504a2de \
    --hash=sha256:72a3c29c1c097708a93762a466e58003a43cac7a0c0076d7984157f0f52d6406 \
    --hash=sha256:7556819afa3e74bbda17208a31a965c22be783ff9b4dfb9724af0aa03aa9bb71 \
    --hash=sha256:76563d8383d332bcc1b93d995753f7e8a9a4c4c00c3b0b207aa0ac31717d02cb \
    --hash=sha256:7fcbe1d19f0299b8784daf92fddc9f7d419757fb31744ca59291c3bfe8f4b6a3 \
    --hash=sha256:7fcda4495bf587996665a2f0add6998f456761b5c4d4b874061ec251491a6b78 \
    --hash=sha256:841e3110a5f291405fd636c6ea3bee0aaab041d15782b330b72c7cfee933e581 \
    --hash=sha256:9397e1226c52ab854ffafdbb6e84a79955881709402e0966460d1c8af0b5ae06 \
    --hash=sha256:a3ef786bd52cf2eb5bd683212725a68bb3062f4482668db960740847483a73a2 \
    --hash=sha256:bcd9bc3216224835b527ee9ddb1e0edda20267380a6f2d50ff32243a9ba68a44 \
    --hash=sha256:bd65a9fa4bba70470e28f547d1b87ab1ca06460b872a13595b1f85725d6c2e72 \
    --hash=sha256:c3abf1fdd593f1dd1eaecacd1f73f26e1af1ee6d9f84e8ce299d094f459996bc \
    --hash=sha256:cdaf9d26c2af8fcbb2545960b739b893629cf2514d3493cab3c1b3124da411c2 \
    --hash=sha256:cff6b3baa2cd6fe74a20ffd6889bd1770cdaed4afa95ef934a9b5c31b082c0be \
    --hash=sha256:d7dfcac1faa72ad2d36bd072417c9ade3eab12e8b46d751c045723cd3373a556 \
    --hash=sha256:dbe25f2031c609a29c385f89db0995f324aec79846a28ee82a30e9351a374f00 \
    --hash=sha256:ed857844b0fe4a3b38507b15f8ab2f9748a04969614d0af7d188cc7d9f683b22
    # via mace-torch
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   sympy
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   torch
numpy==2.5.3 \
    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \
    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \
    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \
    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \
    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \
    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \
    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \
    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \
    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \
    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \
    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \
    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \
    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \
    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \
    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \
    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \
    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \
    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \
    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \
    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \
    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \
    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \
    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \
    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \
    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \
    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \
    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \
    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \
    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \
    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \
    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \
    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \
    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \
    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \
    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \
    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \
    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \
    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \
    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \
    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \
    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \
    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \
    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \
    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \
    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \
    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \
    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \
    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \
    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \
    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \
    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \
    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \
    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \
    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \
    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \
    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \
    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \
    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \
    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \
    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \
    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \
    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \
    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \
    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \
    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \
    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   mace-materials-pipeline (pyproject.toml)
    #   ase
    #   contourpy
    #   h5py
    #   mace-torch
    #   matplotlib
    #   matscipy
    #   pandas
    #   scipy
    #   torchmetrics
    #   torchvision
nvidia-cublas==13.1.1.3 \
    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \
    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \
    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   cuda-toolkit
    #   nvidia-cudnn-cu13
    #   nvidia-cusolver
nvidia-cuda-cupti==13.0.85 \
    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \
    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \
    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   cuda-toolkit
nvidia-cuda-nvrtc==13.0.88 \
    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \
    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \
    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   cuda-toolkit
    #   nvidia-cublas
nvidia-cuda-runtime==13.0.96 \
    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \
    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \
    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   cuda-toolkit
nvidia-cudnn-cu13==9.24.0.43 \
    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \
    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \
    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   torch
nvidia-cufft==12.0.0.61 \
    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \
    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \
    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   cuda-toolkit
nvidia-cufile==1.15.1.6 \
    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \
    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   cuda-toolkit
nvidia-curand==10.4.0.35 \
    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \
    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \
    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   cuda-toolkit
nvidia-cusolver==12.0.4.66 \
    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \
    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \
    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   cuda-toolkit
nvidia-cusparse==12.6.3.3 \
    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \
    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \
    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   cuda-toolkit
    #   nvidia-cusolver
nvidia-cusparselt-cu13==0.8.1 \
    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \
    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \
    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   torch
nvidia-nccl-cu13==2.30.7 \
    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \
    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   torch
nvidia-nvjitlink==13.4.92 \
    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \
    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \
    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \
    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   cuda-toolkit
    #   nvidia-cufft
    #   nvidia-cusolver
    #   nvidia-cusparse
nvidia-nvshmem-cu13==3.4.5 \
    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \
    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   torch
nvidia-nvtx==13.0.85 \
    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \
    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \
    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   cuda-toolkit
opt-einsum==3.4.0 \
    --hash=sha256:69bb92469f86a1565195ece4ac0323943e83477171b91d24c35afe028a90d7cd \
    --hash=sha256:96ca72f1b886d148241348783498194c577fa30a8faac108586b14f1ba4473ac
    # via
    #   mace-torch
    #   opt-einsum-fx
opt-einsum-fx==0.1.4 \
    --hash=sha256:7eeb7f91ecb70be65e6179c106ea7f64fc1db6319e3d1289a4518b384f81e74f \
    --hash=sha256:85f489f4c7c31fd88d5faf9669c09e61ec37a30098809fdcfe2a08a9e42f23c9
    # via e3nn
orjson==3.12.0 \
    --hash=sha256:010811c1b69773450a01cef97727a67b223242f350b77d4ca000e59a9ef2155a \
    --hash=sha256:01efac2074fffb4cb1ea3fab7861e9d0f2a26913854a972f5ac760525dbdaf6e \
    --hash=sha256:03091c8a64db4be38746597ceea68f33c238e27acd9bfe99fb59420224ae7a55 \
    --hash=sha256:08231552159be266a7269555bd9f7c016aee7d9ad6dab06eb58796c5ccb7101c \
    --hash=sha256:0b1ac5bf6609b2716c7954011c5fef6254922df029f45d032ee4ebf5d363cbed \
    --hash=sha256:103b5db66aa53c1f9e88c2524be4f383e831ba7dfd5f9f5af6336a177c622f11 \
    --hash=sha256:1192a7021b6d071aaf909864f6e924d6a2675ca360485b972b8401749311750b \
    --hash=sha256:11edb4660a6680abee9788a3a9072208a2c96538cc1322bd79542065229d8e54 \
    --hash=sha256:18a87929f31d94a77f7dc93cf527e91f39ce7fe7813d588a4de2507efd32a387 \
    --hash=sha256:1c680706fc8396d95e7c4c1f9482563f552137aef91b57237a3ad5aaf64629df \
    --hash=sha256:2b7bcefb9f40fa242fa6b06377232c048e655747790829609168c01162f60578 \
    --hash=sha256:2bb3ce43203936072dd8b4917b01d3aecfc02329bfb42510cb7cfb24708adc9c \
    --hash=sha256:2d3a9da945a4d96ae758fdaaca56742e6b73b6fd554c5d8876f252a6dad70b83 \
    --hash=sha256:2eb5c56e534127b2b8fa38d2363c8b1b8190367ee0d1d16c041517d880843b94 \
    --hash=sha256:31ed278a36304390adc3eec5d7f6fd593a7c3e99e5a06cd07866396c4b1b4710 \
    --hash=sha256:33efefcf5d88eaf400b47e2eba02f91f319bb9951be61ca500b7d536d3f2079d \
    --hash=sha256:3bb17a06f9bd15237b3216c044209fe92597379124018cfc196fbb846cde64df \
    --hash=sha256:3dbce9b6b3074b31a5d5dd322a9c4e5b16f206091ece4194c2e36952847a105e \
    --hash=sha256:40f92192227505acca4e2533ce565f8e6b9535f7d0d09b0968452f18b7376b38 \
    --hash=sha256:477ecaf6b9f88f873341b91fcc736119ca81b5e002a9f7f308ff5b4f2ce2a70e \
    --hash=sha256:50fae885cb073eac7556353ff3df93312b0d5137b0a5056b2bb63f97ed9a93c7 \
    --hash=sha256:532ff8cd4bd59a327a953a7dcde922c7fc25b85e29721bb8633265430d3a3873 \
    --hash=sha256:53c0c474a9d9aff9aebfc0c88de1f28f843d940e6e3a80729abdf6a20274356f \
    --hash=sha256:58c58e1de0006ffb580368d6793c36c7b0b021db066479cf281bf5061e732328 \
    --hash=sha256:5a0fdbc216388f653d3752ff310e710f59253bd4ed6a2bfb3f4f06b84714bbd8 \
    --hash=sha256:61318b6de893c7a9d9f3e5ecbadccbfc26a7eb417ccc7bbf0771de3b4d72f868 \
    --hash=sha256:644d005bc82f917337a95ce270c9f6f92f9834c2bed7b1477572f8db00784222 \
    --hash=sha256:6a2a79c89984dc719817d388c8709e0efc2a2795a934eaa746b4882eb6045adc \
    --hash=sha256:6a31348d7dfa64cd9c78bd1f510ff44c48fe64d71094e6b90e364dba3b55949e \
    --hash=sha256:747843254519dd43b93eee3153a19e5a509334320c4d2f823ec879232db5c796 \
    --hash=sha256:784106539f4b9d4b930e0b4eb8d45168507dae001945e71b4675a367f1e5e806 \
    --hash=sha256:7c2ad193c8004254f34b499f3bd2c80f043d10754aff2b38f93da574f4883f98 \
    --hash=sha256:83445adc40cba26d6d621185a45128ce455b766af368cad2ab64b970603a7978 \
    --hash=sha256:859fc4196855890150bb08e649b30d2c93b249b3e3edd0d3bb2231abf8aa8adc \
    --hash=sha256:8c3bb86dd10f39b3fbf434b7d5dc7cac77d6fc8ac572ae30a10731ede2c4b647 \
    --hash=sha256:8e29957429c35bbb5a185a119c523aa2428b7bbf1a293724c7b9375ed8f892a3 \
    --hash=sha256:8e386b0bc0ddd7cd2056f884b5a0af33592bd01ac66a7ca4b42a65a7e7774a13 \
    --hash=sha256:92ffc09e07233a6ab6d4e067f7841edcbcc134cb4812155cf171ea5255a421d7 \
    --hash=sha256:9a36ec60f1796f9a3f13e3b98390295e17a1c7c10155b448d264098bf9ee5900 \
    --hash=sha256:9caf3d09f47c3c70c4451ada20ef9bc4a4cdffa26f49862cf0a253b329aae2d5 \
    --hash=sha256:9e6fee342a48760e854d743e7a81534d8e2925a6f46e09f750cf56b50fd1de5d \
    --hash=sha256:a15f9a891bce5f5cc5d210e3ad8614d4d1b489a56448c099d6d2a7168b2d954a \
    --hash=sha256:a696529ec96a90d9a5f9570207efe403c8b08f8e4aa2783ee3403511e2fdfa10 \
    --hash=sha256:a6cf4b18e7de173f209f2084ffbd736dd72389a396326ee80a7022168be232e5 \
    --hash=sha256:a791f793b287bbc135b8e87c34e35c8bfc693e2a8a620fab1ae682b925f9a32e \
    --hash=sha256:a94f0f0c6fcbb2b5bd9734c57a489c7584a732bbdf04a39e8c83b861e9d03e92 \
    --hash=sha256:aa3e43a6846e91d7bde3d5a9c66090fcd8744f569a9b6cffc5e1ca38f6a461c0 \
    --hash=sha256:ad0422b92d5195443a39f80c3bcf731cc2e00f153bd32063a47b73b057bd0f03 \
    --hash=sha256:ad29eece0c601737f2a60edc2752a84e7a0785df3efb62e3012834700a5afe0d \
    --hash=sha256:b85931be5b6763c31283805c9bdaae1ca03ad9f6f12a15f1cbf6745b907932c2 \
    --hash=sha256:b9dca132b1fda5565088e65a6b6e742285e0aeceb6fae549fa8863e16c7d3998 \
    --hash=sha256:bc7a872f03522d90e0429e6c0c5cd23084f767bedcb4c58048eec19294613344 \
    --hash=sha256:bd57d79aefa3f84eec851d6de7a366795b9345cfaf17f82b4820430a7a5fa241 \
    --hash=sha256:bf44e374aadde77b1f6109f1030be51433eb61984379852766b6f4e187db7b1e \
    --hash=sha256:c6b11be792c3d2c6a4be2af4ebf97a68d0bf5f580aca6e86a418a354f6cc846a \
    --hash=sha256:d14203fb1aae2ad9b3d52f8a0e82aeb10197ef1c9bc61da7f358bd70b00123d5 \
    --hash=sha256:d39f3f5c3927e2dc0913fe5bbc1a2f6b1b9d1bba1de6358340d0ad0d0c00ca92 \
    --hash=sha256:d8e78d3d93705e3d27cc17cdb209e44d7a8ea203010cac6ce9c7ffc1ae1996f1 \
    --hash=sha256:dce0166feb0a737ab84f598c9a338cbc0b764a036617aa686194f53c7eba0c3e \
    --hash=sha256:e4ac5059baab4b3acbd99485de019ff8cda0fdf34b61fa74f7197a53db78bfe8 \
    --hash=sha256:e9683ee9ea0659da64f36574ef675b8a86330c34c19ea75db1fb93c3ff99e0ef \
    --hash=sha256:ed4ca42bd55955aa34deedcfdfd0e0c31abf51143aae158ae2bc3520b626e517 \
    --hash=sha256:f06dd838d1e07d9b1de0932ec0485ec92c4d5f5d1ad4817a656268c3e88be1e1 \
    --hash=sha256:f3c0683136acdc29afdf88a5bc2f7d3d0e34087788d1d63c0144b805a87a196f \
    --hash=sha256:fb2539159dfe8d371914f354360fa50e4a577cc89222a3828b9650a5e5040252
    # via mace-torch
packaging==26.3 \
    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \
    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   huggingface-hub
    #   lightning-utilities
    #   matplotlib
    #   matscipy
    #   opt-einsum-fx
    #   torchmetrics
pandas==3.0.6 \
    --hash=sha256:0704044b676496b8350e023b09f174a26772456c974a2b11c36bebb558c9490d \
    --hash=sha256:085e3786ae6b2e82b406266bce36690f72b9dc1421903ba9296b2981a9fcf586 \
    --hash=sha256:097090508a1dd335013d39106fc10b20f4fd4a171638e47b77d55798ed9dab6c \
    --hash=sha256:1bcb3e9ed29e74a7439cedff9e2aefd3ea65de84d7de9ccb6c194192541bd60e \
    --hash=sha256:1e7c0afdcaf6661d795fcefc2f647ddd1136f62cdc153fba177c685d97a87808 \
    --hash=sha256:1e92d9fa834c7d877130027cddc0cad8dcff97c1f6cca26bd6310f847228b658 \
    --hash=sha256:22172a92e7ee678ec0140c7af4fc9366b55413834a1cd86af78b3caa0b0574de \
    --hash=sha256:253e12cb9081b0afbac607920f6142975966bc315135e09de275fdbaa415d2de \
    --hash=sha256:265f562fdd1079f69f3de96dd425c3405224038c0af4f920c54bd240ee2c4640 \
    --hash=sha256:2a8fc94be2ee5f1d86f97aacd8cc566f81680b6498e76f3007421bb5d98151bf \
    --hash=sha256:2e5fa32ff162dfdbc280157d664f44d23049ae414725af9676df339c501d82cd \
    --hash=sha256:3ef908d28590b3f42d7070e7ad8f9b34b442b260b7f3c1afb57e0040c58cdb1b \
    --hash=sha256:429d9df32731ab01383ed98f2baa7a60368090d1a94fc06019a12062510e8630 \
    --hash=sha256:47121f9571503f724c9b93e297ab6254ac99c77adf5e9ed085ea419fd585c258 \
    --hash=sha256:4e25e2e1adee99ddfada6f7206a79ae8e9c8a8861b0e3eaaba165006d3eef18e \
    --hash=sha256:4ff44b2cb51cbd691c91f92c4ea6c71e34003f239ebd67c2e857dc898466b49c \
    --hash=sha256:50c44cbf5820b6b91a5f74aae04972472aefadd3cd9fbd1010409d85528bd570 \
    --hash=sha256:569e114072b24fc4970c12e2b4bab252671668a40b324318903380cab0254c0c \
    --hash=sha256:583be68728a31d0d750d5b8d9e00f02b153df0d4655f858bde93cb84cfc4227c \
    --hash=sha256:5e75072773c1b2f7cb63faa3a6f562aede11f3976f68ed34cb538bc091a28171 \
    --hash=sha256:5edd0a7abb0986ecce1ac81f56d99b6763f86aa6946dceb6c661224f90af5a19 \
    --hash=sha256:60d81f9e1799b36f3739e7fff44d1fbb2e8fd5a271b3863e03de9715fccda0fa \
    --hash=sha256:62f51d7f651c8054c5e82a69265c98082e795d1442df7ca6edc3a545d61214b1 \
    --hash=sha256:654aae059295dbba6ecd2328ca12712a2cf1676214c8699f1c29213f7ccf9c34 \
    --hash=sha256:66b07ef7315a31bfe1089cd3d71a7de781c9dca986762d0b4fe7c0ef17465d10 \
    --hash=sha256:6ff482fa91fa2bafd92e8fe66ce3645c851824310f295c1f0a2f96e928fc4541 \
    --hash=sha256:77ccbe5057aece6fc172b9b77f19c04335af6882bc2e10c8f3ee4e6bfb3da553 \
    --hash=sha256:7dac2d65e9087e8e7b5a45fe15c4920911a221df061ab629943ce016489145c7 \
    --hash=sha256:83e91d15738d7783c050197cef2f2cf82fc6353dae9865aa87ed1fa16aa4d55a \
    --hash=sha256:86fa853a12e0b70927e2b1ee00d56d2224ec9cbb4b9d58348b5ad52d2f21150e \
    --hash=sha256:8fe77b408d82e2615674dfed62533b95e18a03610573877422aada4f625d4947 \
    --hash=sha256:963ca21199097a84c7827c4678b04e30833084fbf8ef44fde3fa7180a29f8fa0 \
    --hash=sha256:97274c9adf6255bb48c620cd6959805efa7f09ea2167f0e0ae006a448cd2fca7 \
    --hash=sha256:994a79608263fe1c14cc48ffa7300e2b834b7d1cb406ffe96a08828cb0cdd79b \
    --hash=sha256:9ae8073aed8e21d1a7fe263dcdc6840743549722a6738198a0a46000fa9476f2 \
    --hash=sha256:9dab635a549e58a053c7b0fa054dc0bd7be22f0ed9a720f4a85d5fb993276172 \
    --hash=sha256:9e492cd4bdba6778de4fe0df7f4590c012161ebcf9902dce01b01dc683105514 \
    --hash=sha256:a3a22e07fe75347eaacc75b0e85297947af4fba6b4aae23916bd8b6828d0bba3 \
    --hash=sha256:a4dbd4dc65cbe645b92b8785d0f96dd7311010dc6606cf620e51b07b8788a12a \
    --hash=sha256:a77a1a44e4d88f1c6a2a64d3eb12efec8420875722e14279800b173a7c7c2804 \
    --hash=sha256:b27c8d890e4aa2171437ae2a39de1d215e674158e4865c4023a8b31c932513b2 \
    --hash=sha256:bd75ed0c840f709fc2ae26ddd9534ac77ca1a48ac0cce521a74acaa85f3340a7 \
    --hash=sha256:c6e4aae3e9bea26c6c9a20d88d96c86ec4a99b4db5fd516bcb4e829ab2c0ee36 \
    --hash=sha256:c826e9babb7790142c399f58599d8de679bea059d7b39c5b6efa2096fac37266 \
    --hash=sha256:cc39303913e2ea129915670de5d1c9fbd647f543bb72e5543bac8baa94e9e42f \
    --hash=sha256:d7564d86a94c2eb8ab290b07f63ddaae5c032fa53897c29a2ff2197d43aee8af \
    --hash=sha256:d7dcd21238cbb4828ff148481ba01cac8946dc5121457b5aeba28636f8f99a60 \
    --hash=sha256:db7ec631f26223beee8e5c9e0b8f23c24d8197bbd1d982421d4e3188bea51965 \
    --hash=sha256:e3dccb584123b399c07562ac4d62543e90ede49ddf8ce3c13ffc64cbe828c281 \
    --hash=sha256:e7c1905ef02c3d6d43d9dbd5b6ccb4da4870a0b0c821bbc103fbdb6f3ad2707b \
    --hash=sha256:eb6900de08ac85f93ac4948aa6b80842eba555875337b8359035ac9c43e92d34 \
    --hash=sha256:ee913a91669056c1de1a6b733fbfeab711de9e54e3bee2dfa5fe79d9457247d1 \
    --hash=sha256:ef738d71d1059245b6bb03e312be06d8b3821326a83486c1ad03b9aba3710e44 \
    --hash=sha256:f3ce8a6968045481e91a3990e797e348ce13db45ee164a7095bbc824e26c09dd \
    --hash=sha256:f4e7c52eb108d752e7592268108fd3e98efd76d83a3125cdd06c621c2e44359b \
    --hash=sha256:f8029ec0f1f89e4f985929ce1f6626dabf3140d61a4e9c1215afdab34eaf9a5d \
    --hash=sha256:fb625f426b375bcc96e3a04c5d5d266cd7be6ae5d6866e0e703382ab5164068c \
    --hash=sha256:ff51a4459ed036e93d1eb1bb5e6e7b28685d3cb6b7c12b91c05b31024e234729
    # via mace-torch
pillow==12.3.0 \
    --hash=sha256:00808c5e14ef63ac5161091d242999076604ff74b883423a11e5d7bbb38bf756 \
    --hash=sha256:04f01d28a6aaff387bf842a13be313df23ba0597a44f1a976c9feb3c6ff4711a \
    --hash=sha256:06ff022112bc9cbf83b60f8e028d94ad87b60621706487e65f673de61610ab59 \
    --hash=sha256:0740a512dc522224c77d9aa5a8d70d8b7d73fb91f2c21125d8d025d3b8990e45 \
    --hash=sha256:0847a763afefb695bc912d7c131e7e0632d4edc1d8698f58ddabec8e46b8b6d3 \
    --hash=sha256:0dd2064cbc55aaec028ef5fbb60fa47bb6c3e7918e07ff17935284b227a9d2df \
    --hash=sha256:0feb2e9d6ad6c9e3c06effe9d00f3f1e618a6643273576b016f591e9315a7139 \
    --hash=sha256:10e41f0fbf1eec8cfd234b8fe17a4caac7c9d0db4c204d3c173a8f9f6ef3232b \
    --hash=sha256:1182d52bc2d5e5d7d0949503aa7e36d12f42205dc287e4883f407b1988820d39 \
    --hash=sha256:164b31cd1a0490ab6efae01aa5df49da7061be0af1b30e035b6e9a1bfe34ee6e \
    --hash=sha256:1657923d2d45afb66526e5b933e5b3052e6bdea196c90d3abb2424e18c77dae8 \
    --hash=sha256:186941b6aef820ad110fb01fb06eb925374dc3a21b17e37ec9a53b250c6fe2d1 \
    --hash=sha256:1cca606cd25738df4ed873d5ad46bbdb3d83b5cbca291f6b4ff13a4df6b0bbe8 \
    --hash=sha256:21900ce7ba264168cd50defae43cd75d25c833ad4ad6e73ffc5596d12e25ac89 \
    --hash=sha256:236ff70b9312fb68943c703aa842ca6a758abfa45ac187a5e7c1452e96ef72b5 \
    --hash=sha256:23aceaa007d6172b02c277f0cd359c79492bbb14f7072b4ede9fbcaf20648130 \
    --hash=sha256:23d27a3e0307ec2244cc51e7287b919aa68d097504ebe19df4e76a98a3eea5bd \
    --hash=sha256:24870b09b224f7ae3c39ed07d10e819d06f8720bc551847b1d623832b5b0e28d \
    --hash=sha256:251bf95b67017e27b13d82f5b326234ca62d70f9cf4c2b9032de2358a3b12c7b \
    --hash=sha256:25b9b82bb22e6e2b3cd07b39c68b7b862001226cb3dff7130d1cb914121b39ed \
    --hash=sha256:28ce87c5ab450a9dd970b52e5aca5fe63ed432d18a2eaddd1979a00a1ba24ace \
    --hash=sha256:300557495eb45ebb8aec96c2da9c4be642fbf7cd937278b4013ba894ea8eb0eb \
    --hash=sha256:30f2aa603c41533cc25c05acd0da21636e84a315768feb631c937177db558931 \
    --hash=sha256:331b624368d4f1d069149002f25f44bc61c8919ce8ddb3c45bdad8f6e2d89510 \
    --hash=sha256:37d6d0a00072fd2948eb22bce7e1475f34569d90c87c59f7a2ec59541b77f7a6 \
    --hash=sha256:37dc8f7bbb66efe481bb60defacef820c950c24713fb44962ed6aa2a50966de1 \
    --hash=sha256:3b8182a766685eaa002637e28b4ec8d6b18819a0c71f579bf0dbaa5830297cce \
    --hash=sha256:3edce1d53195db527e0191f84b71d02022de0540bf43a16ed734ed7537b07385 \
    --hash=sha256:446c34dcc4324b084a53b705127dc15717b22c5e140ae0a3c38349d4efec071e \
    --hash=sha256:4998562bf62a445225f22e07c896bb04b35b1b1f2eb6d760584c9c51d7a5f78c \
    --hash=sha256:4b0a7fe987b14c31ebda6083f74f22b561fd3739bc0ac51e019622e3d72668c7 \
    --hash=sha256:4e8c2a84d977f50b9daed6eeaf3baef67d00d5d74d932288f02cb94518ee3ace \
    --hash=sha256:4f883547d4b7f0495ebe7056b0cc2aea76094e7a4abc8e933540f3271df27d9c \
    --hash=sha256:514435a37670e3e5e08f3945b68718b6ed329bb84367777e16f9f4dfe1e61a0f \
    --hash=sha256:53aa02d20d10c3d814d536aa4e5ac9b84ca0ff5a88377963b085ad6822f93e64 \
    --hash=sha256:5594fc43d548a7ed94949d139aa1341b270f1863f11cfd37f5a6c8b778a6b67f \
    --hash=sha256:571b9fcb07b97ef3a492028fb3d2dc0993ca23a06138b0315286566d29ef718a \
    --hash=sha256:57b3d78c95ba9059768b10e28b813002261d3f3dfc55cc48b0c988f625175827 \
    --hash=sha256:5afb51d599ea772b8365ae807ae557f18bccfe46ab261fd1c2a9ed700fc6eb17 \
    --hash=sha256:6b02afb9b97f65fbca5f31db6a2a3ba21aa93030225f150fa3f249717e938fb4 \
    --hash=sha256:6c0016e7b354317c4e9e525b937ac8596c38d2d232b419529b9cd7a1cd46e39a \
    --hash=sha256:71d6097b330eea8fd15097780c8e89cb1a8ce7838669f48c5bacd6f663dd4701 \
    --hash=sha256:756c768d0c9c2955feb7a56c37ea24aea2e369f8d36a88da270b6a9f19e62b5e \
    --hash=sha256:78cb2c6865a35ab8ff8b75fd122f6033b92a62c82801110e48ddd6c936a45d91 \
    --hash=sha256:7a743ff716f746fc19a9557f60dab1600d4613255f8a7aeb3cdde4db7eb15a66 \
    --hash=sha256:85f998ea1848bc6757289e739cfbdda3a04adfd58b02fc018ce54d754a5ce468 \
    --hash=sha256:8728f216dcdb6e6d555cf971cb34076139ad74b31fc2c14da4fafc741c5f6217 \
    --hash=sha256:877c3f311ff35410f690861c4409e7ccbf0cd2f878e50628a28e5a0bb689e658 \
    --hash=sha256:8cd2f7bdda092d99c9fc2fb7391354f306d01443d22785d0cbfafa2e2c8bb418 \
    --hash=sha256:8e95e1385e4998ae9694eeaa4730ba5457ff61185b3a55e2e7bea0880aef452a \
    --hash=sha256:962864dc93511324d51ddbb5b9f8731bf71675b93ca612a07441896f4688fb8c \
    --hash=sha256:9cf95fe4d0f84c82d282745d9bb08ad9f926efa00be4697e767b814ce40d4330 \
    --hash=sha256:9e881fca225083806662a5c43d627d215f258ff43c890f831966c7d7ba9c7402 \
    --hash=sha256:a2b55dd6b2a4c4b7d87ffa56bdb33fdc5fdb9a462173861a7bc097f17d91cb09 \
    --hash=sha256:a45650e8ce7fafffd731db8550230db6b0d306d181a90b67d3e6bca2f1990930 \
    --hash=sha256:a876864214e136f0eb367788dbd7df045f4806801518e2cfe9e13229cfe06d8f \
    --hash=sha256:ae26d61dfa7a47befdc7572b521024e8745f3d809bd95ca9505a7bba9ef849ec \
    --hash=sha256:af8d94b0db561cf68b88a267c5c44b49e134f525d0dc2cb7ed413a66bc23559a \
    --hash=sha256:b343699e8308bdc51978310e1c959c584e7869cc8c40780058c87da7781a1e94 \
    --hash=sha256:b3c777e849237620b022f7f297dd67705f9f5cf1685f09f02e46f93e92725468 \
    --hash=sha256:b629de27fda84b42cde7edef0d85f13b958b47f6e9bbcbba9b673c562a89bd8b \
    --hash=sha256:ba09209fbe443b4acccebe845d8a138b89a8f4fbaeedd44953490b5315d5e965 \
    --hash=sha256:ba54cfebe86920a559a7c4d6b9050791c20513650a1952ebe3368c7dc70306f8 \
    --hash=sha256:bcb46e2f9feff8d06323983bd83ed00c201fdcab3d74973e7072a889b3979fcd \
    --hash=sha256:bcc33feacfaefce60c12fd500a277533bdc02b10a19f7f6d348763d8140bbba7 \
    --hash=sha256:bf16ba1b4d0b6b7c8e534936632270cf70eb00dbe09005bc345b2677b726855c \
    --hash=sha256:cf1845d02ad822a369a49f2bb9345b1614744267682e7a03527dc3bf6eea1777 \
    --hash=sha256:d69141514cc30b774ceea5e3ed3a6635c8d8a96edf664689b890f4089111fb35 \
    --hash=sha256:d9c7f76c0673154f044e9d78c8655fb4213f6ca31a836df48b40fe5d187717b9 \
    --hash=sha256:dbce0b29841537a2fa4a214c2bbf14de3587c9680caa9b4e217568472490b28f \
    --hash=sha256:dc624f6bc473dacdf7ef7eb8678d0d08edf15cd94fad6ae5c7d6cc67a4e4902f \
    --hash=sha256:e158cb00350dc278f3b91551101aa7d12415a66ebf2c91d8d5ac14e56ddd3ad0 \
    --hash=sha256:e491916b378fba47242221bb9ead245211b70d504f495d105d17b14a24b4907c \
    --hash=sha256:e795b7eb908249c4e43c7c99fac7c2c75dab0c43566e37db472a355f63693d71 \
    --hash=sha256:e7e480451b9fa137494bccd3a7d69adbe8ac65a87d97be61e11f1b1050a5bac3 \
    --hash=sha256:e91206ee562682b51b98ef4b26a6ef48fd84e15fd4c4bc5ec768eb641d206838 \
    --hash=sha256:e9871b1ffbfa9656b60aeee92ed5136a5742696006fa322b29ea3d8da0ecc9cf \
    --hash=sha256:e9aeb04d6aef139de265b29683e119b638208f88cf73cdd1658aa07221165321 \
    --hash=sha256:ebaea975e03d3141d9d3a507df75c9b3ec90fa9d2ffd07567b3a978d9d790b26 \
    --hash=sha256:f0606c8bf2cdefea14a43530f7657cbbb7ecf1c4222512492ef4a4434a9501ec \
    --hash=sha256:f13c32a3abd6079a66d9526e18dad9b6d280384d49d7c54040cd57b6424041d9 \
    --hash=sha256:f7401aebd7f581d7f83a439d87d474999317ee099218e5ad25d125290990ba65 \
    --hash=sha256:fa4ecea169a355be7a3ade2c783e2ed12f0e40d2c5621cda8b3297faf7fbb9f5 \
    --hash=sha256:fbd139c8447d25dd750ab79ee274cc5e1fe80fc56340ab10b18a195e1b6eca3e \
    --hash=sha256:fdafc9cce40277e0f7a0feabce0ee50dd2fa1800f3b38015e51296b5e814048d \
    --hash=sha256:fe3cca2e4e8a592be0f269a1ca4835c25199d9f3ce815c8491048f785b0a0198 \
    --hash=sha256:ffd0c5368496f41b0944be820fcb7a838aa6e623d250b01acf2643939c3f99d7
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   matplotlib
    #   torchvision
prettytable==3.18.0 \
    --hash=sha256:439217116152244369caf3d9f1caf2f9fe29b03bd79e88d2928c8e718c95d680 \
    --hash=sha256:b3346e0e6f79180833aebaac088ae926340586cf6d7d991b9eb125b65f72313a
    # via mace-torch
pyparsing==3.3.3 \
    --hash=sha256:928ae7e20211f3b6f3915a72f06a0cfd29ab9d24279dd6346b6b1a7146397d36 \
    --hash=sha256:ece8c00a69cf01b45d0b1dedabb469c90d8caf996d4fda40f147627a122849a4
    # via matplotlib
python-dateutil==2.9.0.post0 \
    --hash=sha256:37dd54208da7e1cd875388217d5e00ebd4179249f90fb72437e91a35459a0ad3 \
    --hash=sha256:a8b2bc7bffae282281c8140a97d3aa9c14da0b136dfe83f850eea9a5f7470427
    # via
    #   matplotlib
    #   pandas
pyyaml==6.0.3 \
    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \
    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \
    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \
    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \
    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \
    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \
    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \
    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \
    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \
    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \
    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \
    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \
    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \
    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \
    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \
    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \
    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \
    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \
    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \
    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \
    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \
    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \
    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \
    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \
    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \
    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \
    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \
    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \
    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \
    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \
    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \
    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \
    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \
    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \
    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \
    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \
    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \
    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \
    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \
    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \
    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \
    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \
    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \
    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \
    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \
    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \
    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \
    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \
    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \
    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \
    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \
    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \
    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \
    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \
    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \
    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \
    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \
    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \
    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \
    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \
    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \
    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \
    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \
    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \
    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \
    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \
    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \
    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \
    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \
    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \
    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \
    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \
    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   huggingface-hub
    #   mace-torch
safetensors==0.8.0 \
    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \
    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \
    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \
    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \
    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \
    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \
    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \
    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \
    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \
    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \
    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \
    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \
    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \
    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \
    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \
    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \
    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   mace-materials-pipeline (pyproject.toml)
scipy==1.18.1 \
    --hash=sha256:011413b7426b75012840e35649e00fe0a2c3bae89fed433876e3a99251572efc \
    --hash=sha256:0ac49ea97594532dd44b7136094d35f5440fa06e6d9c6384a74c01764df388c5 \
    --hash=sha256:0e82073ecc7acc6436fac4b31674109c7e1d3e596789767eda01258a8c9e8123 \
    --hash=sha256:0fcb3c93519f27bb4f0c4b0f7802cdcaca7fcf93267b75edda2e9f4e8a55cbd7 \
    --hash=sha256:10ac20c69d880f77f375db44c22e3e6a644f9fefa291d4cd2fb9790a89fc99fd \
    --hash=sha256:11c423f1049c5755ad4409af52a9ada1cff96fe9b50795d4af3619f292901239 \
    --hash=sha256:179ce34a8d0fe273d8883ba59e17e052247d08973dfcb743ca52bb1cce2d60b0 \
    --hash=sha256:1bca3b943fc2567ea49cd02c99abde49da4d5178ec46f624bd8255cda8755beb \
    --hash=sha256:1d73131e358976663dd969e1fb4ed1404b815cd977eaaedc3b3a133ba2d81c35 \
    --hash=sha256:2a0b02f9fc46f8520330c23d45e6560db7e3a0d927232139427637f98943e11d \
    --hash=sha256:2d3ab0e8c69a17dd3559eab8cbb88f258e285c94d572c2719033f90f83290c89 \
    --hash=sha256:30f464bee641fa8e282577c7dce027308403213c6ca8270bba73285c91024bc5 \
    --hash=sha256:33a834464fdabc0f26a45508df31b3cc5d028e04dbf6c5ed398541418e0a12fe \
    --hash=sha256:3ab3523da44749156e1f68b464dc56af11ae4cbc5c739a49d05f32b982eca9f3 \
    --hash=sha256:3c085faa2cfa879c5141df483f836f4d691045a078224a670fa570fa01612d89 \
    --hash=sha256:457fd7a2a8edeb044ab6ffbc0aa03ff6cd18491356e5e0c834d76ce621b916d1 \
    --hash=sha256:49023963c193dacee096301452f223ee24d86ec5807f8df93c0f7221d119e305 \
    --hash=sha256:52c4b7422442aba924d03ad4019852b08a92e64ea187b933135687bfe2747307 \
    --hash=sha256:559ed65f60c1af5a03f3912605a1b5114f522c7c32fb23c3376ae8f03219fe28 \
    --hash=sha256:5632e3ae3d09197c446310cd5187de63e28448ce22f0f67b2b93d97503c0c230 \
    --hash=sha256:5e4d44984abc0020154ea81b247adeddcc3ac5527b975ff798bd1ba0adc513c2 \
    --hash=sha256:75b00eb8fb802090aa903f4ea1c7f5a584779f967361e68b7e98e531cc2d7174 \
    --hash=sha256:78a0d7c918e74a232394117160e7e3db503377572a45bcef8826e4ab8a35feba \
    --hash=sha256:78c0665edead396b1abb4897c41a5c1d9bf090c8a637a4c20a61678e0a264e66 \
    --hash=sha256:7bbf207c4453ce1ad2e00b17313852b33310b83090c2311bdaf97f93c0380d12 \
    --hash=sha256:7f4b8bc363b6d65ee2152bec57568e3c52639bb34c46057b09857a307ed5e21d \
    --hash=sha256:82f201b4c878551d48558337aab270d3c6cca5507b8737c8d8a608d234cccde0 \
    --hash=sha256:83de5453a7799afc9048b4616bd085cef126e36412f0ea2f6370c36a2a3a51e7 \
    --hash=sha256:88f0e784020649f88ea48c9f5ddfa403bf9205820667c0914740b392035afb82 \
    --hash=sha256:8bcf3c1ba5d6456e2effd30fcbd3459b044d683fcdac79a2e6830f0bdf7de487 \
    --hash=sha256:911de823097db8b63f034299d12662db93344e6ffa0b881cbb57748974b70168 \
    --hash=sha256:92c14f5bdbfb6216315ce33e78080474082de8b3830122ba97809bfbe65f75c0 \
    --hash=sha256:95298364e251be3e60249facbeeca03631d3bb7584f85879516ec55ac717b81f \
    --hash=sha256:9554bcc6d715ee87a633a3cc8e7703c6628b100dd29cb8a2efc4c0533c7ff729 \
    --hash=sha256:9f2897bf7737392ad0d5213ea7b6add72a4edf5679b3153106aeb88b6507b3b9 \
    --hash=sha256:a1d33a7836f7ddc1993427966a0823468ec41bcbdb1a9f9942d1d7e57f803ba3 \
    --hash=sha256:ac0333bdf38309aa3dcbe7e3fa7ea29e7a2c37c6ea306a757b700ded8e4596ad \
    --hash=sha256:bff0b729edd992766136b34e39cc76bc2fad905aa58897ee72a9cd000a6d8443 \
    --hash=sha256:c24acac1e18912761c4700239bbc1fd32f615af690f1584d49b35859be51324d \
    --hash=sha256:c35d74ce0e193ff740c2f2be2ac913ddc232fe6c1ff40b26cfecb9c670c63314 \
    --hash=sha256:c825cef2f49e46753726a7181a8e199804a912b29519ada542c6ebc654951899 \
    --hash=sha256:c9d18a33309122074ea483dd92dd444189166b8b2ec429fe9ed5ac73c7a0aa23 \
    --hash=sha256:cbf38d043c1aa4ab306e1ada6ab6eddacc3322a20b7af1b30bc93254b366fe09 \
    --hash=sha256:cd479fc04dd9401e3b4f49e76518768ef99c4f517a98c284eb091fd725719adf \
    --hash=sha256:ceb30a00ce7c92d459819443d29ca486d882b83fb6738bdcbb2a1cce94ac5daa \
    --hash=sha256:cfbf154f2ba187f2ed6cce2639efff7d105f1140573642c0161615b6d91d6a87 \
    --hash=sha256:d2924a03db38dc2e848bca2fe9f077dafb891480b91a00a0963a8cf86dfc31c1 \
    --hash=sha256:d416b16cccfd70fbf62400e84d0bb2f4e6af519a45557f1692c749b37f14b315 \
    --hash=sha256:d65d448389b8436493abcf629cc94ad0cf32aecaf06e1acca1de53cc795f2f12 \
    --hash=sha256:d84a09d0dad90ba6525d8ac1c2334b33e64bf3ccfe9e841f02feb867a22681e4 \
    --hash=sha256:ddef79fb382df40104a19bb7151b3b23e57c1778fcf857c71ceecd9bd264513f \
    --hash=sha256:e3b417bf8c2c7c16e8f58ad91db17783ec911ac16e7b50eb6eab6e809b4f5b07 \
    --hash=sha256:e402cf31eb68f453dbb2d36fc6d722b33f24a55d68b2ae1d92fa6305ca71c298 \
    --hash=sha256:e6fb6a55cc0ba97b59a1f288fb86dc6fce8bdfc0fffcbfd015e3a954bf2a2d93 \
    --hash=sha256:e708533e8b2ae2497d65346538a7dcc92814410b25b81432eac66de0f2af8265 \
    --hash=sha256:ea324d9dd34c38bfb9bec8ca4d1b407db97dbb74029f566b8e322b1b6fe56fe6 \
    --hash=sha256:eb0dfcf4e28a99c12c999744a2ff67c9b06200e20401c7c88186e33552a46331 \
    --hash=sha256:eda632a7981f69730d6281f451db9c1c370993a2c0d7ddb43e2a809a2862b83a \
    --hash=sha256:f29633129f9fa7e88a3f0fca835de2d030bfc9643f7799e1a0c46cee24d38fc7 \
    --hash=sha256:f55fa87b6c612ecd6b058f167c53231b1d14e412efe361d3d6e38b3631c73218 \
    --hash=sha256:fdaf5ea890a6183d0565f51a61799d67081bd5b1cf03c5f4b3fd3732108625c9
    # via
    #   ase
    #   e3nn
    #   matscipy
setuptools==84.0.0 \
    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \
    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   torch
six==1.17.0 \
    --hash=sha256:4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274 \
    --hash=sha256:ff70335d468e7eb6ec65b95b99d3a2836546063f63acc5171de367e834932a81
    # via python-dateutil
smmap==5.0.3 \
    --hash=sha256:4d9debb8b99007ae47165abc08670bd74cb74b5227dda7f643eccc4e9eb5642c \
    --hash=sha256:c106e05d5a61449cf6ba9a1e650227ecfb141590d2a98412103ff35d89fc7b2f
    # via gitdb
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   e3nn
    #   torch
torch==2.14.0 \
    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \
    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \
    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \
    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \
    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \
    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \
    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \
    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \
    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \
    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \
    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \
    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \
    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \
    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \
    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \
    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \
    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \
    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \
    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \
    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \
    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \
    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \
    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \
    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   mace-materials-pipeline (pyproject.toml)
    #   e3nn
    #   mace-torch
    #   opt-einsum-fx
    #   torch-ema
    #   torchmetrics
    #   torchvision
torch-ema==0.3 \
    --hash=sha256:5a3595405fa311995f01291a1d4a9242d6be08a0fc9db29152ec6cfd586ea414 \
    --hash=sha256:823ad8da5c10bc1eebcec739cc3f521aa9573229fe04e5673c304d29f1433279
    # via mace-torch
torchmetrics==1.9.0 \
    --hash=sha256:a488609948600df52d3db4fcdab02e62aab2a85ef34da67037dc3e65b8512faa \
    --hash=sha256:bfdcbff3dd1d96b3374bb2496eb39f23c4b28b8a845b6a18c313688e0d2d9ca1
    # via mace-torch
torchvision==0.29.0 \
    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \
    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \
    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \
    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \
    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \
    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \
    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \
    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \
    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \
    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \
    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \
    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \
    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \
    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \
    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \
    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \
    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \
    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \
    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \
    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \
    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \
    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \
    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \
    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   mace-materials-pipeline (pyproject.toml)
tqdm==4.70.1 \
    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \
    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   huggingface-hub
    #   mace-torch
triton==3.8.0 \
    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \
    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \
    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \
    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \
    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \
    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \
    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \
    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \
    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \
    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \
    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \
    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   torch
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
    # via
    #   -c ast-audio-classification-pipeline lock @ 16eee39
    #   anyio
    #   ase
    #   huggingface-hub
    #   lightning-utilities
    #   torch
wcwidth==0.9.1 \
    --hash=sha256:03cfca3dcbffa86564290fe3c9978a6191ba003e8ced7f7dbda315fcb3fbe725 \
    --hash=sha256:0665ee822ea04e25801e6e82e5407528be0863e88a17b0e7ca038843a4a3ac0c \
    --hash=sha256:0d68a30d504c68cfdff2a5f804675c1e7ab4c0bbe878024c8b680ec5579cde67 \
    --hash=sha256:10b00ba23482e352f874d2e8135e7ace9da838646c7dd800566246bbd46125ff \
    --hash=sha256:356376852357b8fca71fe5415808ec421679e04b4a98eb7c9cb6a7984b911a05 \
    --hash=sha256:40d936d72c9bdc10df43f93a8be502bc5024b487259139f66a328722c07f34a9 \
    --hash=sha256:5823209b0d43af322ce698c689380d7c15ca31fa8e6e3be8459f27031bef0af5 \
    --hash=sha256:61bd7aef9cafb6cb77a37a169998d7928ce82a51522146d11f60db9e7d1cb43a \
    --hash=sha256:69bb970cf5652b88cfdb9d3fdd1764fc15e5f8ad531643e7bb3e894cd969740e \
    --hash=sha256:6e1272b7986cefe79783737e38bdb9eaae0682b333c7bd132024441193dd5ce7 \
    --hash=sha256:708158c082364af442f9983de7b6ec9ac0d2e1b825ada25f0911b1f138d55405 \
    --hash=sha256:747fb724223f417a17541a95a17c1dac3a8ef9a0cf41684950f0eab191a35f65 \
    --hash=sha256:991d1c8834f548e9c1f16432075ee84638e122312556bbf1ed595ea8fffc4673 \
    --hash=sha256:9f1636c5075ffd5c2e835b4561874f6e5dd2bbeba3c6c2c99f067d0d16883af8 \
    --hash=sha256:b5da43d6967668982e44a52fb551967d293f86d26cd86036ac95bbdd34394ed9 \
    --hash=sha256:bcb9ed4a367cc025bf1092ac679a156759605182d60b7ed3c28f7221a42ddf55 \
    --hash=sha256:c4cead196551112cb8f43cdd1f80c235ef2456e34b1a9955c424537ec99961b2 \
    --hash=sha256:dc10e262c3ac0abbfd0a2a51e45a848b1b7f500b21ff512630277973ba25674d \
    --hash=sha256:e0c3a1c45c5b9550c6919a4449e95f5b177f6e165786376981db8f1addae9b21 \
    --hash=sha256:eab587e18e7cadf1a750b0098fc8bebfb62c125eb9306f2268f0443a282a3d78 \
    --hash=sha256:fe021c4d8de9d36c31a0cb41d0d2546dadd3b0708a301f1a0c66ce200851831f
    # via prettytable
'''

SKIP_INSTALL = os.environ.get("DIMER_NOTEBOOK_CI_PREINSTALLED") == "1"
ISOLATED_ENV = Path(os.environ.get("DIMER_ISOLATED_ENV", "dimer_isolated_env")).resolve()
ISOLATED_PYTHON = ISOLATED_ENV / "bin" / "python"
ISOLATED_TOOLS = ISOLATED_ENV.with_name(ISOLATED_ENV.name + "_tools")

if SKIP_INSTALL:
    print("DIMER_NOTEBOOK_CI_PREINSTALLED=1: the pins are already installed; the notebook runs in this kernel.")
else:
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle or Linux Jupyter): its locked environment is built for manylinux x86_64.")
    setup_started = time.perf_counter()
    if hashlib.sha256(LOCK_TEXT.encode("utf-8")).hexdigest() != LOCK_SHA256:
        raise RuntimeError("The carried lock does not match its digest: regenerate the notebook from the repository instead of editing this cell.")
    ISOLATED_TOOLS.mkdir(parents=True, exist_ok=True)
    lock_path = ISOLATED_TOOLS / LOCK_NAME
    lock_path.write_text(LOCK_TEXT, encoding="utf-8", newline="\n")
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2**attempt)
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError("The pinned uv wheel failed its size/SHA-256 check: refusing to run it. Run this cell again; if it repeats, the download is being altered.")
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(name for name in archive.namelist() if name.endswith(".data/scripts/uv"))
        uv = ISOLATED_TOOLS / "uv"
        uv.write_bytes(archive.read(member))
    uv.chmod(0o700)
    # uv gets no kernel Python path; the managed interpreter is downloaded once and reused on a re-run.
    uv_env = dict(os.environ)
    for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP"):
        uv_env.pop(name, None)
    if not ISOLATED_PYTHON.is_file():
        subprocess.run([str(uv), "venv", "--quiet", "--managed-python", "--python", MANAGED_PYTHON, str(ISOLATED_ENV)], env=uv_env, check=True)
    isolated_version = subprocess.run([str(ISOLATED_PYTHON), "-c", "import platform; print(platform.python_version())"], env=uv_env, check=True, capture_output=True, text=True).stdout.strip()
    if isolated_version != MANAGED_PYTHON:
        raise RuntimeError(f"{ISOLATED_ENV} holds Python {isolated_version}, not {MANAGED_PYTHON}: delete that folder (or start a fresh runtime) and run this cell again.")
    subprocess.run([str(uv), "pip", "install", "--quiet", "--python", str(ISOLATED_PYTHON), "--require-hashes", "--only-binary", ":all:", "--index-url", "https://pypi.org/simple", "--no-deps", "-r", str(lock_path)], env=uv_env, check=True)
    print({"isolated_environment": str(ISOLATED_ENV), "isolated_python": isolated_version, "kernel_python": platform.python_version(), "locked_packages": LOCKED_PACKAGES, "setup_seconds": round(time.perf_counter() - setup_started)})

{'isolated_environment': '/content/dimer_isolated_env', 'isolated_python': '3.12.12', 'kernel_python': '3.13.15', 'locked_packages': 73, 'setup_seconds': 61}


The next cell starts one Python process in that environment and routes **every later code cell** to it. Printed output comes back to the notebook as usual, variables persist from cell to cell, and an error stops **Run all** as it would in the kernel. These two cells are marked `# dimer: kernel cell` and are the only cells that run in the kernel. If you re-run a single cell later, it still runs in the isolated environment with the variables created so far; to start over, restart the session and choose **Run all**. `DIMER_NOTEBOOK_CI_PREINSTALLED=1` lets an executor that has already installed exactly these pins run every cell in its own kernel instead.

In [ ]:
# @title Infrastructure: route the remaining cells to the isolated environment
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import signal
from multiprocessing.connection import Connection

# The worker runs in the isolated environment. It executes each routed cell in one persistent namespace and sends
# back printed text, displayed objects and matplotlib figures, so every cell behaves as it would in the kernel.
_WORKER_SOURCE = r"""
import ast, base64, builtins, io, linecache, os, signal, sys, traceback, types
from multiprocessing.connection import Connection

_send = Connection(int(sys.argv[1]), readable=False)
_recv = Connection(int(sys.argv[2]), writable=False)


class _Stream(io.TextIOBase):
    def __init__(self, name):
        self._name = name

    @property
    def encoding(self):
        return "utf-8"

    def writable(self):
        return True

    def isatty(self):
        return False

    def write(self, text):
        if text:
            _send.send(("stream", self._name, str(text)))
        return len(text)


sys.stdout, sys.stderr = _Stream("stdout"), _Stream("stderr")


def _figure_bundle(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    return {"image/png": base64.b64encode(buffer.getvalue()).decode("ascii"), "text/plain": repr(fig)}


def _flush_figures():
    plt = sys.modules.get("matplotlib.pyplot")
    if plt is None:
        return
    for number in plt.get_fignums():
        _send.send(("display", _figure_bundle(plt.figure(number))))
    plt.close("all")


def _mimebundle(obj):
    if hasattr(obj, "savefig"):
        return _figure_bundle(obj)
    data = {"text/plain": repr(obj)}
    for method, mime in (("_repr_html_", "text/html"), ("_repr_markdown_", "text/markdown"), ("_repr_png_", "image/png")):
        render = getattr(obj, method, None)
        if callable(render):
            try:
                value = render()
            except Exception:
                value = None
            if isinstance(value, bytes):
                value = base64.b64encode(value).decode("ascii")
            if value is not None:
                data[mime] = value
    return data


def display(*objects, **kwargs):
    for obj in objects:
        _send.send(("display", _mimebundle(obj)))


try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot

    matplotlib.pyplot.show = lambda *args, **kwargs: _flush_figures()
except ImportError:
    pass

if os.environ.get("DIMER_KERNEL_IS_COLAB") == "1":
    # google.colab only exists in the kernel; forward the BYOD upload dialog to it.
    def _upload():
        _send.send(("upload",))
        reply = _recv.recv()
        if reply[1] is None:
            raise RuntimeError("The notebook kernel could not open the upload dialog.")
        return reply[1]

    import importlib.machinery

    def _stub(name, package):
        # A spec on every stub: importlib.util.find_spec("google.colab") (accelerate does this) raises on a None __spec__.
        module = types.ModuleType(name)
        module.__spec__ = importlib.machinery.ModuleSpec(name, None, is_package=package)
        if package:
            module.__path__ = []
        return module

    try:
        import google
    except ImportError:
        google = _stub("google", True)
        sys.modules["google"] = google
    _colab = _stub("google.colab", True)
    _files = _stub("google.colab.files", False)
    _files.upload = _upload
    _colab.files = _files
    google.colab = _colab
    sys.modules["google.colab"] = _colab
    sys.modules["google.colab.files"] = _files

_main = types.ModuleType("__main__")
_main.__dict__.update(__builtins__=builtins, display=display)
sys.modules["__main__"] = _main
_count = 0
while True:
    # An interrupt only lands inside a running cell; between cells it is ignored.
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    try:
        message = _recv.recv()
    except EOFError:
        break
    if message[0] != "run":
        continue
    _count += 1
    filename = f"<isolated cell {_count}>"
    source = message[1]
    linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
    try:
        signal.signal(signal.SIGINT, signal.default_int_handler)
        tree = ast.parse(source, filename)
        tail = ast.Expression(tree.body.pop().value) if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        exec(compile(tree, filename, "exec"), _main.__dict__)
        if tail is not None:
            value = eval(compile(tail, filename, "eval"), _main.__dict__)
            if value is not None:
                display(value)
        _flush_figures()
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        _send.send(("done",))
    except BaseException as exc:
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        frames = exc.__traceback__.tb_next if exc.__traceback__ is not None else None
        _send.send(("error", "".join(traceback.format_exception(type(exc), exc, frames)), f"{type(exc).__name__}: {exc}"))
"""


class IsolatedCellError(RuntimeError):
    """A routed cell raised inside the isolated environment; its traceback is printed above."""


class IsolatedRuntime:
    """One persistent worker process in the isolated environment, fed one cell at a time."""

    def __init__(self, python, display=None):
        to_kernel_r, to_kernel_w = os.pipe()
        to_worker_r, to_worker_w = os.pipe()
        env = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1", DIMER_NOTEBOOK_CI_PREINSTALLED="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            env.pop(name, None)
        env["DIMER_KERNEL_IS_COLAB"] = "1" if "google.colab" in sys.modules else "0"
        self.proc = subprocess.Popen(
            [str(python), "-c", _WORKER_SOURCE, str(to_kernel_w), str(to_worker_r)],
            pass_fds=(to_kernel_w, to_worker_r),
            env=env,
            start_new_session=True,  # interrupts reach the worker only through run(), exactly once
        )
        os.close(to_kernel_w)
        os.close(to_worker_r)
        self._recv = Connection(to_kernel_r, writable=False)
        self._send = Connection(to_worker_w, readable=False)
        if display is None:
            from IPython.display import display
        self._display = display

    def _exited(self):
        return RuntimeError(
            f"The isolated environment's Python process exited (code {self.proc.wait()}); a crash of this kind is "
            "usually running out of memory. Restart the session and choose Run all again."
        )

    def run(self, source):
        try:
            self._send.send(("run", source))
        except OSError:
            raise self._exited() from None
        while True:
            try:
                message = self._recv.recv()
            except EOFError:
                raise self._exited() from None
            except KeyboardInterrupt:
                self.proc.send_signal(signal.SIGINT)
                continue
            kind = message[0]
            if kind == "stream":
                (sys.stdout if message[1] == "stdout" else sys.stderr).write(message[2])
            elif kind == "display":
                self._display(message[1], raw=True)
            elif kind == "upload":
                self._send.send(("upload", self._colab_upload()))
            elif kind == "error":
                sys.stderr.write(message[1])
                raise IsolatedCellError(message[2]) from None
            elif kind == "done":
                return

    @staticmethod
    def _colab_upload():
        try:
            from google.colab import files
        except ImportError:
            return None
        return files.upload()

    def close(self):
        self._send.close()
        self.proc.wait(timeout=30)


def _is_user_cell():
    # ipykernel transforms a cell before executing it; its caller knows whether this is a silent frontend request.
    frame = sys._getframe(2)
    while frame is not None:
        local = frame.f_locals
        if "silent" in local and "store_history" in local:
            return bool(local["store_history"]) and not bool(local["silent"])
        frame = frame.f_back
    return True


def _route_to_isolated_runtime(lines):
    source = "".join(lines)
    if not source.strip() or "# dimer: kernel cell" in source or not _is_user_cell():
        return lines
    return [f"_DIMER_ISOLATED_RUNTIME.run({source!r})\n"]


if SKIP_INSTALL:
    print("Routing disabled: the notebook runs in this kernel.")
else:
    from IPython import get_ipython as _kernel_shell

    _ip = _kernel_shell()
    _ip.input_transformers_cleanup[:] = [
        t for t in _ip.input_transformers_cleanup if getattr(t, "__name__", "") != "_route_to_isolated_runtime"
    ]
    if isinstance(globals().get("_DIMER_ISOLATED_RUNTIME"), IsolatedRuntime):
        _DIMER_ISOLATED_RUNTIME.close()
    _DIMER_ISOLATED_RUNTIME = IsolatedRuntime(ISOLATED_PYTHON)
    _ip.input_transformers_cleanup.append(_route_to_isolated_runtime)
    print(f"Every later code cell now runs in {ISOLATED_PYTHON} (pid {_DIMER_ISOLATED_RUNTIME.proc.pid}).")

Every later code cell now runs in /content/dimer_isolated_env/bin/python (pid 1881).


### Record the runtime

This is the first cell that runs in the isolated environment. Nothing is installed here: the locked environment already holds the pins listed in the cell. It records the notebook's source revision and the versions actually imported. Look for a dictionary reporting the notebook's source revision, the isolated Python (3.12.12), `torch`, `mace`, `e3nn`, `ase` versions, and whether CUDA is available. (The pip install and stale-import guard in the cell are skipped: the isolated worker, like an executor that has pre-installed exactly these pins, sets `DIMER_NOTEBOOK_CI_PREINSTALLED=1`.)

In [ ]:
# @title Infrastructure: record the runtime
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'mace-torch==0.3.16',
    'e3nn==0.4.4',
    'ase==3.29.0',
    'numpy==2.5.3',
    'safetensors==0.8.0',
    'huggingface-hub==1.32.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'mace-materials-pipeline',
    'repository_revision': '4101558ae1adc44d0ab30fd8ffa9223bc46f2899',
    'embedded_module': 'src/mace_materials_pipeline/pipeline.py',
    'embedded_modules': ['src/mace_materials_pipeline/metrics.py', 'src/mace_materials_pipeline/pipeline.py', 'src/mace_materials_pipeline/samples.py'],
    'module_sha256': '83a374bb186e656af799aeda6586655e49773c36ec6ac4b43f0a160cc571bce0',
    'generator': 'build_notebook.py/2.1',
    'notebook_spec': '2.2',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, mace, e3nn, ase
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'mace': mace.__version__, 'e3nn': e3nn.__version__, 'ase': ase.__version__, 'cuda': torch.cuda.is_available()})

{'notebook_source': {'repository': 'mace-materials-pipeline', 'repository_revision': '4101558ae1adc44d0ab30fd8ffa9223bc46f2899', 'embedded_module': 'src/mace_materials_pipeline/pipeline.py', 'embedded_modules': ['src/mace_materials_pipeline/metrics.py', 'src/mace_materials_pipeline/pipeline.py', 'src/mace_materials_pipeline/samples.py'], 'module_sha256': '83a374bb186e656af799aeda6586655e49773c36ec6ac4b43f0a160cc571bce0', 'generator': 'build_notebook.py/2.1', 'notebook_spec': '2.2'}, 'python': '3.12.12', 'torch': '2.14.0+cu130', 'mace': '0.3.16', 'e3nn': '0.4.4', 'ase': '3.29.0', 'cuda': True}


## 2. Pipeline code (carried verbatim from `src/mace_materials_pipeline/` @ `4101558ae1ad`)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/mace_materials_pipeline/metrics.py`

In [ ]:
# @title Infrastructure: carried module `src/mace_materials_pipeline/metrics.py` (you may run it without reading it)
"""Energy / force regression metrics and the trivial baselines every adapted number is read against.

Energies are compared per atom (eV/atom) because total energies scale with system size; forces are
compared per Cartesian component (eV/Å). Two baselines need no model: the **composition baseline**
fits one reference energy per element to the training set (the classical "E0" regression, which
cannot see atomic displacements at all) and the **zero-force baseline** predicts every force as zero,
so its MAE is the mean absolute reference force. The third comparison point — the frozen foundation
model after E0 calibration — comes from the pipeline itself.
"""

from __future__ import annotations

import math
from collections.abc import Mapping, Sequence
from typing import Any


def _flatten(rows: Sequence[Sequence[float]]) -> list[float]:
    return [float(x) for row in rows for x in row]


def _mae(errors: Sequence[float]) -> float:
    return sum(abs(e) for e in errors) / len(errors) if errors else math.nan


def _rmse(errors: Sequence[float]) -> float:
    return math.sqrt(sum(e * e for e in errors) / len(errors)) if errors else math.nan


def regression_metrics(references: Sequence[Mapping[str, Any]], predictions: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Per-atom energy and per-component force errors between labelled structures and predictions."""
    if len(references) != len(predictions):
        raise ValueError("references and predictions differ in length")
    if not references:
        raise ValueError("at least one structure is required")
    energy_errors: list[float] = []
    force_errors: list[float] = []
    n_atoms = 0
    for ref, pred in zip(references, predictions, strict=True):
        if ref.get("energy") is None or ref.get("forces") is None:
            raise ValueError(f"{ref.get('name', 'structure')}: energy and forces are required")
        count = len(ref["symbols"])
        n_atoms += count
        energy_errors.append((float(pred["energy"]) - float(ref["energy"])) / count)
        pf, rf = _flatten(pred["forces"]), _flatten(ref["forces"])
        if len(pf) != len(rf):
            raise ValueError("force arrays differ in shape")
        force_errors.extend(p - r for p, r in zip(pf, rf, strict=True))
    return {
        "n_structures": len(references),
        "n_atoms": n_atoms,
        "energy_mae_per_atom": _mae(energy_errors),
        "energy_rmse_per_atom": _rmse(energy_errors),
        "force_mae": _mae(force_errors),
        "force_rmse": _rmse(force_errors),
        "units": {"energy": "eV/atom", "forces": "eV/Å"},
    }


def _lstsq(rows: Sequence[Sequence[float]], targets: Sequence[float]) -> list[float]:
    """Least squares via numpy (a light dependency already required for structures)."""
    import numpy as np

    solution, *_ = np.linalg.lstsq(np.array(rows, dtype=float), np.array(targets, dtype=float), rcond=None)
    return [float(x) for x in solution]


def composition_baseline(train: Sequence[Mapping[str, Any]], evaluation: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Fit one energy per element on `train` (E = Σ n_element × E0_element) and score `evaluation`.

    It is blind to geometry by construction: every structure of the same composition gets the same
    energy, and its force prediction is identically zero."""
    if not train or not evaluation:
        raise ValueError("train and evaluation sets must be non-empty")
    elements = sorted({s for record in train for s in record["symbols"]})
    rows = [[record["symbols"].count(el) for el in elements] for record in train]
    e0 = _lstsq(rows, [float(record["energy"]) for record in train])
    predictions = []
    for record in evaluation:
        unknown = sorted(set(record["symbols"]) - set(elements))
        if unknown:
            raise ValueError(f"elements {unknown} absent from the training set; the composition baseline cannot score them")
        energy = sum(record["symbols"].count(el) * e for el, e in zip(elements, e0, strict=True))
        predictions.append({"energy": energy, "forces": [[0.0, 0.0, 0.0] for _ in record["symbols"]]})
    metrics = regression_metrics(evaluation, predictions)
    metrics["e0_ev"] = dict(zip(elements, e0, strict=True))
    metrics["baseline"] = "composition (per-element E0 regression, zero forces)"
    return metrics


def zero_force_baseline(evaluation: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Force MAE / RMSE of predicting zero force everywhere: the magnitude a model has to beat."""
    if not evaluation:
        raise ValueError("evaluation set must be non-empty")
    components = [x for record in evaluation for x in _flatten(record["forces"])]
    return {
        "n_structures": len(evaluation),
        "force_mae": _mae(components),
        "force_rmse": _rmse(components),
        "units": {"forces": "eV/Å"},
        "baseline": "zero forces",
    }

**Module 2/3:** `src/mace_materials_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
# @title Infrastructure: carried module `src/mace_materials_pipeline/pipeline.py` (you may run it without reading it)
"""MACE-MP-0b2 small (`mace-foundations/mace-mp-0`) DIMER pipeline: verified snapshot, energy / force /
stress prediction for atomic structures, and bounded fine-tuning of the foundation interatomic potential
to a user's reference data with a portable adapter.

MACE-MP-0 is a universal machine-learned interatomic potential (Batatia et al., 2023): an equivariant
message-passing model trained on the Materials Project PBE/PBE+U trajectories that predicts a total
energy, per-atom energies, forces and stress for any structure of the 89 supported elements. The "0b2
small" checkpoint is the 128x0e, two-interaction, 8.2 M-parameter variant with the Agnesi radial
transform and ZBL pair repulsion, published November 2024.

The upstream distribution format is a **pickled torch module** (`.model`). Under the fleet asset
specification (§11) that is executable serialization, not data, so this package never loads it as a
runtime artifact. Instead:

* `audit_model_file` statically disassembles the pickle (outer archive and the TorchScript archives
  e3nn embeds in it) without executing anything, lists every global it would import and every
  generated source string it would `exec`, and refuses anything outside the torch / e3nn / mace
  allow-lists. The audit result is digest-pinned in `PICKLE_AUDIT_SHA256`.
* `convert_model` unpickles the digest-verified, audit-clean source **once**, extracts the
  constructor arguments and the state dict, and writes a code-free pair: a JSON config plus a
  safetensors file. Their digests are pinned in `CONVERTED_SHA256`; the conversion is deterministic.
* `MaceMaterialsPipeline.from_pretrained` rebuilds `ScaleShiftMACE(**config)` from the **installed**
  `mace-torch` package and loads the safetensors with `strict=True`. The DIMER-hosted artifact is the
  converted pair; the `.model` file is the immutable provenance, not the served asset.

Behaviour is preserved: on three probe structures (rattled Cu and Al supercells, an H2O molecule) the
rebuilt model and the pickled model differ by at most 2.8e-14 eV in energy and 3.6e-15 eV/Å in
forces, and the upstream `MACECalculator` agrees to the same precision (docs/WEIGHTS.md).

Everything model-related is imported lazily so that snapshot verification, the pickle audit and input
validation run (and can refuse) before `torch`, `e3nn` or `mace` are imported (fleet RTM-001). `ase`
and `numpy` are used for structure handling and are imported freely.
"""

from __future__ import annotations

import contextlib
import copy
import hashlib
import io
import json
import math
import pickletools
import re
import time
import warnings
import zipfile
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any


@contextlib.contextmanager
def _float64_default() -> Any:
    """Run a block with torch's default dtype set to float64 (the model's dtype) and hand the caller's default
    back afterwards, even on failure. Loading this pipeline must not change tensor creation elsewhere in the
    hosting process, while every tensor mace-torch builds for us (model, graph data) must be float64."""
    import torch

    previous = torch.get_default_dtype()
    torch.set_default_dtype(torch.float64)
    try:
        yield
    finally:
        torch.set_default_dtype(previous)

MODEL_ID = "mace-foundations/mace-mp-0"
MODEL_REVISION = "e291ace2bfae073c3ebc7ae2f9479a525989baa7"
MODEL_LICENSE = "mit"
MODEL_KEY = "mace-mp-0b2-small"
ARTIFACT_FORMAT = "org.valcorza.mace-materials.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Immutable upstream source asset (pickled torch module). Byte-identical to the upstream release asset
# ACEsuit/mace-mp `mace_mp_0b2/mace-small-density-agnesi-stress.model` (2024-11-12); see docs/WEIGHTS.md.
SOURCE_MODEL_NAME = "mace-mp-0b2-small.model"
SOURCE_MODEL_BYTES = 67_622_684
SOURCE_MODEL_SHA256 = "d5773bf9440e96d6eb8c598f84bd0e6369fcfa432f626a87f890e07da3c651c9"
# Code-free serving pair produced deterministically by `convert_model` (asset spec §11.2).
CONVERTED_CONFIG_NAME = "mace-mp-0b2-small.config.json"
CONVERTED_WEIGHTS_NAME = "mace-mp-0b2-small.safetensors"
CONVERTED_SHA256 = {
    CONVERTED_CONFIG_NAME: "130b641179c5da6dbf6ec439b9c20661e3d60512007f6f2d42669d4f87fc9893",
    CONVERTED_WEIGHTS_NAME: "2ed99065c4decf21613b7038dde6bda2b694b2f414012c0ca743f7ff7f86fe93",
}
CONVERTED_BYTES = {CONVERTED_CONFIG_NAME: 3_279, CONVERTED_WEIGHTS_NAME: 67_400_566}
# Digest of the static pickle audit (sorted globals + every generated source string, see audit_model_file).
PICKLE_AUDIT_SHA256 = "9bb150f1ecc9212e594313ba0ef9aa4884b7634dfc8a11da8cbe0c9ebf70d1e5"

# Architecture facts asserted against the converted config before the model library is imported.
MODEL_CLASS = "ScaleShiftMACE"
R_MAX = 5.0
NUM_INTERACTIONS = 2
HIDDEN_IRREPS = "128x0e"
NUM_ELEMENTS = 89
CORRELATION = 3
MAX_ELL = 3
SUPPORTED_ATOMIC_NUMBERS: tuple[int, ...] = tuple(range(1, 84)) + tuple(range(89, 95))  # H..Bi, Ac..Pu
PARAMETER_COUNT = 8_221_984
STATE_TENSORS = 86  # 80 weight / buffer tensors from the pickle + 6 path-pruning flags of mace-torch 0.3.16

# Operational ceilings (structural validation, before any model import).
MAX_STRUCTURES_PER_CALL = 32
MAX_ATOMS_PER_STRUCTURE = 512
MAX_ATOMS_PER_CALL = 4_096
MIN_INTERATOMIC_DISTANCE = 0.5  # Å; closer than any chemistry, the ZBL core would dominate
MAX_CELL_LENGTH = 200.0  # Å
MIN_CELL_HEIGHT = 1.0  # Å for each periodic direction
MAX_ABS_COORDINATE = 10_000.0  # Å
MAX_ABS_ENERGY_PER_ATOM = 1_000.0  # eV
MAX_ABS_FORCE = 1_000.0  # eV/Å

_SYMBOL_RE = re.compile(r"^[A-Z][a-z]?$")

# Static pickle-audit allow-lists (asset spec §11): what a MACE `.model` file may legitimately reference.
PICKLE_ALLOWED_MODULE_ROOTS = ("torch", "e3nn", "mace", "collections", "_codecs", "__torch__")
PICKLE_ALLOWED_EXACT_GLOBALS = frozenset({"__builtin__.set"})
# Every line torch.fx's `reduce_graph_module` would exec from the pickled import block, verbatim.
GENERATED_CODE_ALLOWED_IMPORT_LINES = frozenset(
    {
        "import torch",
        "from math import inf",
        "from math import nan",
        "from torch import device",
        "import torch.fx._pytree as fx_pytree",
        "import torch.utils._pytree as pytree",
        "NoneType = type(None)",
    }
)
GENERATED_CODE_FORBIDDEN = re.compile(
    r"\b(os|subprocess|exec|eval|compile|open|__import__|socket|sys|builtins|importlib|pickle|shutil|"
    r"requests|urllib|globals|locals|breakpoint|input|ctypes|signal|threading|multiprocessing)\b"
)
GENERATED_CODE_CALL_PREFIXES = ("torch.", "fx_pytree.", "pytree.", "math.", "device(")


# --------------------------------------------------------------------------------------------------
# snapshot manifest, staging, static pickle audit and conversion
# --------------------------------------------------------------------------------------------------


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"no snapshot manifest at {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    listed = {entry["path"] for entry in manifest["files"]}
    if SOURCE_MODEL_NAME not in listed:
        raise ValueError(f"manifest does not list {SOURCE_MODEL_NAME}; refusing to proceed")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256_file(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
        if entry["path"] == SOURCE_MODEL_NAME and (digest != SOURCE_MODEL_SHA256 or size != SOURCE_MODEL_BYTES):
            raise ValueError(f"{SOURCE_MODEL_NAME}: manifest digest disagrees with the package constant")
    return manifest


def verify_converted(path: str | Path | None = None) -> dict[str, Any]:
    """Check the code-free serving pair (config JSON + safetensors) against the pinned digests."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    report: dict[str, Any] = {"files": []}
    for name, expected in CONVERTED_SHA256.items():
        file_path = root / name
        if not file_path.is_file():
            raise FileNotFoundError(f"converted file missing: {file_path}")
        size = file_path.stat().st_size
        if size != CONVERTED_BYTES[name]:
            raise ValueError(f"{name}: size {size} != pinned {CONVERTED_BYTES[name]}")
        digest = _sha256_file(file_path)
        if digest != expected:
            raise ValueError(f"{name}: sha256 {digest} != pinned {expected}")
        report["files"].append({"path": name, "bytes": size, "sha256": digest})
    return report


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the snapshot against its DIMER manifest (size + SHA-256 of every listed Hub file) and,
    when the converted serving pair is present, that pair against the pinned digests."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _verify_manifest(root, MODEL_ID, MODEL_REVISION)
    converted = all((root / name).is_file() for name in CONVERTED_SHA256)
    if converted:
        verify_converted(root)
    return {**manifest, "converted": converted}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest entries that are absent locally (a fresh clone commits the manifest and the
    3 KB converted config but git-ignores the 67.6 MB `.model` source and the safetensors it converts
    to). `verify_snapshot` still runs after; conversion happens in `from_pretrained`."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _audit_pickle_bytes(data: bytes, report: dict[str, Any]) -> None:
    """Walk one pickle stream with `pickletools.genops` (no execution) and record what it references."""
    stack: list[Any] = []
    for op, arg, _pos in pickletools.genops(io.BytesIO(data)):
        if op.name == "GLOBAL":  # pickletools renders the (module, name) pair space-separated
            key = arg.replace("\n", " ").replace(" ", ".", 1)
            report["globals"][key] = report["globals"].get(key, 0) + 1
        elif op.name == "STACK_GLOBAL":
            key = f"{stack[-2]}.{stack[-1]}"
            report["globals"][key] = report["globals"].get(key, 0) + 1
        if op.name in ("SHORT_BINUNICODE", "BINUNICODE", "UNICODE", "SHORT_BINSTRING", "BINSTRING"):
            stack.append(arg)
            if isinstance(arg, str):
                if arg.startswith("PK\x03\x04"):
                    report["nested_archives"] += 1
                    _audit_archive_bytes(arg.encode("latin-1"), report)
                elif "def forward" in arg:
                    report["code_strings"].append(arg)
                elif re.search(r"^(?:import|from)\s", arg, re.M):
                    report["import_blocks"].append(arg)
        elif op.name in ("MEMOIZE", "BINPUT", "LONG_BINPUT", "PUT"):
            pass
        else:
            stack.append(None)


def _audit_archive_bytes(blob: bytes, report: dict[str, Any]) -> None:
    archive = zipfile.ZipFile(io.BytesIO(blob))
    for name in archive.namelist():
        if name.endswith(".pkl"):
            _audit_pickle_bytes(archive.read(name), report)
        elif name.endswith(".py"):
            report["torchscript_sources"].append(archive.read(name).decode("utf-8"))


def audit_model_file(path: str | Path | None = None) -> dict[str, Any]:
    """Statically audit the pickled `.model` file without executing it.

    Disassembles the outer torch archive and every nested TorchScript archive with `pickletools`,
    collects every GLOBAL the unpickler would import, every fx-generated source string it would
    `exec`, and every TorchScript source it would compile, then checks them against the allow-lists.
    Raises `ValueError` on any violation. Returns the audit report with its digest."""
    file_path = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR / SOURCE_MODEL_NAME
    if not file_path.is_file():
        raise FileNotFoundError(f"source model not found: {file_path}")
    report: dict[str, Any] = {
        "file": file_path.name,
        "globals": {},
        "nested_archives": 0,
        "code_strings": [],
        "import_blocks": [],
        "torchscript_sources": [],
    }
    _audit_archive_bytes(file_path.read_bytes(), report)
    violations: list[str] = []
    for name in sorted(report["globals"]):
        root = name.split(".")[0]
        if name in PICKLE_ALLOWED_EXACT_GLOBALS or root in PICKLE_ALLOWED_MODULE_ROOTS:
            continue
        violations.append(f"global outside allow-list: {name}")
    import_lines: set[str] = set()
    for block in report["import_blocks"]:
        import_lines.update(line.strip() for line in block.splitlines() if line.strip())
    for code in report["code_strings"]:
        for match in re.finditer(r"^\s*((?:from|import)\s[^\n]+)", code, re.M):
            import_lines.add(match.group(1).strip())
    for line in sorted(import_lines - GENERATED_CODE_ALLOWED_IMPORT_LINES):
        violations.append(f"generated code import line outside allow-list: {line}")
    for kind, sources in (
        ("fx", report["code_strings"]),
        ("import-block", report["import_blocks"]),
        ("torchscript", report["torchscript_sources"]),
    ):
        for src in sources:
            hit = GENERATED_CODE_FORBIDDEN.search(src)
            if hit:
                violations.append(f"forbidden token {hit.group(0)!r} in {kind} source")
                break
    for code in report["code_strings"]:
        for match in re.finditer(r"(?<![\w.])([A-Za-z_][\w.]*)\(", code):
            callee = match.group(1)
            if callee in ("forward", "slice", "int", "float", "tuple", "list", "range", "len", "getattr_1"):
                continue
            if not callee.startswith(GENERATED_CODE_CALL_PREFIXES) and "." not in callee:
                violations.append(f"generated code calls a bare name: {callee}")
                break
    summary = {
        "globals": sorted(report["globals"]),
        "nested_archives": report["nested_archives"],
        "fx_code_strings": len(report["code_strings"]),
        "import_blocks": len(report["import_blocks"]),
        "torchscript_sources": len(report["torchscript_sources"]),
        "generated_code_import_lines": sorted(import_lines),
        "violations": violations,
    }
    digest = hashlib.sha256()
    digest.update("\n".join(summary["globals"]).encode("utf-8"))
    digest.update(b"\n--fx--\n" + "\n---\n".join(sorted(report["code_strings"])).encode("utf-8"))
    digest.update(b"\n--imports--\n" + "\n---\n".join(sorted(report["import_blocks"])).encode("utf-8"))
    digest.update(b"\n--ts--\n" + "\n---\n".join(sorted(report["torchscript_sources"])).encode("utf-8"))
    summary["audit_sha256"] = digest.hexdigest()
    if violations:
        raise ValueError(f"{file_path.name}: pickle audit failed: {violations}")
    return summary


def _config_from_json(config: Mapping[str, Any]) -> dict[str, Any]:
    """Assert the converted config names the pinned architecture (before any model import)."""
    expected = {
        "model_class": MODEL_CLASS,
        "r_max": R_MAX,
        "num_interactions": NUM_INTERACTIONS,
        "hidden_irreps": HIDDEN_IRREPS,
        "num_elements": NUM_ELEMENTS,
        "correlation": CORRELATION,
        "max_ell": MAX_ELL,
        "dtype": "float64",
        "heads": ["default"],
    }
    for key, value in expected.items():
        if config.get(key) != value:
            raise ValueError(f"converted config {key}={config.get(key)!r} disagrees with the package constant {value!r}")
    if tuple(config.get("atomic_numbers", ())) != SUPPORTED_ATOMIC_NUMBERS:
        raise ValueError("converted config atomic_numbers disagree with SUPPORTED_ATOMIC_NUMBERS")
    if len(config.get("atomic_energies", ())) != NUM_ELEMENTS:
        raise ValueError("converted config atomic_energies length != NUM_ELEMENTS")
    return dict(config)


def _json_ready(value: Any) -> Any:
    """Serialise `extract_config_mace_model` output: classes and callables by name, irreps as strings."""
    import numpy as np
    import torch
    from e3nn import o3

    if isinstance(value, type):
        return value.__name__
    if isinstance(value, o3.Irreps):
        return str(value)
    if callable(value):
        return value.__name__
    if isinstance(value, (np.ndarray, torch.Tensor)):
        return value.tolist()
    if isinstance(value, (list, tuple)):
        return [int(x) if isinstance(x, np.integer) else x for x in value]
    return value


def build_model(config: Mapping[str, Any]) -> Any:
    """Instantiate `ScaleShiftMACE` from the converted JSON config using the installed mace-torch."""
    import mace  # noqa: F401  (import order: mace sets TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD before e3nn loads its constants)
    import numpy as np
    import torch
    from e3nn import o3
    from mace.modules import ScaleShiftMACE, blocks

    # The model is float64: construct it under that default and hand the caller's default back afterwards.
    with _float64_default():
        kwargs = {k: v for k, v in config.items() if k not in ("model_class", "dtype")}
        kwargs["interaction_cls"] = getattr(blocks, kwargs["interaction_cls"])
        kwargs["interaction_cls_first"] = getattr(blocks, kwargs["interaction_cls_first"])
        kwargs["readout_cls"] = getattr(blocks, kwargs["readout_cls"])
        kwargs["hidden_irreps"] = o3.Irreps(kwargs["hidden_irreps"])
        kwargs["MLP_irreps"] = o3.Irreps(kwargs["MLP_irreps"])
        kwargs["gate"] = {"silu": torch.nn.functional.silu}[kwargs["gate"]]
        kwargs["atomic_energies"] = np.array(kwargs["atomic_energies"], dtype=float)
        kwargs["atomic_inter_scale"] = float(kwargs["atomic_inter_scale"])
        kwargs["atomic_inter_shift"] = float(kwargs["atomic_inter_shift"])
        if kwargs.get("edge_irreps") is not None:
            kwargs["edge_irreps"] = o3.Irreps(kwargs["edge_irreps"])
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return ScaleShiftMACE(**kwargs)


# State-dict keys of the pickled checkpoint that mace-torch 0.3.16 no longer registers (the ZBL block now
# derives its cutoff from covalent radii); their values (p=5, r_max=5.0) are not used by the current forward.
STALE_SOURCE_KEYS = ("pair_repulsion_fn.cutoff.p", "pair_repulsion_fn.cutoff.r_max", "pair_repulsion_fn.r_max")


def convert_model(path: str | Path | None = None, *, audit: bool = True) -> dict[str, Any]:
    """Convert the digest-verified `.model` pickle into the code-free serving pair, deterministically.

    Order of operations: digest check → static audit (refuses on any allow-list violation or audit
    digest drift) → single unpickle (`torch.load(weights_only=False)`, the only place this package
    executes anything from the file) → `extract_config_mace_model` → JSON config + safetensors state
    dict of a freshly constructed model that loaded the pickled tensors. Returns the digests."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    source = root / SOURCE_MODEL_NAME
    if not source.is_file():
        raise FileNotFoundError(f"source model not found: {source}")
    size = source.stat().st_size
    if size != SOURCE_MODEL_BYTES:
        raise ValueError(f"{SOURCE_MODEL_NAME}: size {size} != pinned {SOURCE_MODEL_BYTES}")
    digest = _sha256_file(source)
    if digest != SOURCE_MODEL_SHA256:
        raise ValueError(f"{SOURCE_MODEL_NAME}: sha256 {digest} != pinned {SOURCE_MODEL_SHA256}")
    if audit:
        summary = audit_model_file(source)
        if summary["audit_sha256"] != PICKLE_AUDIT_SHA256:
            raise ValueError(
                f"{SOURCE_MODEL_NAME}: pickle audit digest {summary['audit_sha256']} != pinned {PICKLE_AUDIT_SHA256}"
            )
    import mace  # noqa: F401
    import torch
    from mace.tools.scripts_utils import extract_config_mace_model
    from safetensors.torch import save_file

    # Conversion runs under a float64 default and restores the caller's default even on failure.
    with _float64_default():
        started = time.perf_counter()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            pickled = torch.load(source, map_location="cpu", weights_only=False)
        if type(pickled).__name__ != MODEL_CLASS:
            raise ValueError(f"{SOURCE_MODEL_NAME} unpickled to {type(pickled).__name__}, expected {MODEL_CLASS}")
        config = {k: _json_ready(v) for k, v in extract_config_mace_model(pickled).items()}
        config["model_class"] = type(pickled).__name__
        config["dtype"] = str(next(pickled.parameters()).dtype).replace("torch.", "")
        _config_from_json(config)
        text = json.dumps(config, indent=2, sort_keys=True) + "\n"
        (root / CONVERTED_CONFIG_NAME).write_bytes(text.encode("utf-8"))

        source_state = {k: v.contiguous() for k, v in pickled.state_dict().items()}
        model = build_model(config)
        result = model.load_state_dict({k: v for k, v in source_state.items() if k not in STALE_SOURCE_KEYS}, strict=False)
        if result.unexpected_keys or any(not k.endswith("_zeroed") for k in result.missing_keys):
            raise ValueError(
                f"state-dict layout drift between the pickle and mace-torch: "
                f"missing={result.missing_keys} unexpected={result.unexpected_keys}"
            )
        canonical = {k: v.contiguous() for k, v in model.state_dict().items()}
        if len(canonical) != STATE_TENSORS:
            raise ValueError(f"converted state dict has {len(canonical)} tensors, expected {STATE_TENSORS}")
        for key, tensor in source_state.items():
            if key not in STALE_SOURCE_KEYS and not torch.equal(tensor, canonical[key]):
                raise ValueError(f"converted tensor {key} is not bit-identical to the pickled one")
        save_file(canonical, str(root / CONVERTED_WEIGHTS_NAME), metadata={"format": "pt"})
        report = verify_converted(root)
        return {
            "source": {"path": SOURCE_MODEL_NAME, "bytes": size, "sha256": digest},
            "converted": report["files"],
            "dropped_keys": list(STALE_SOURCE_KEYS),
            "added_flag_keys": sorted(result.missing_keys),
            "seconds": round(time.perf_counter() - started, 2),
        }


# --------------------------------------------------------------------------------------------------
# structures and validation (no model import)
# --------------------------------------------------------------------------------------------------

INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "1..MAX_STRUCTURES_PER_CALL structures, each {symbols, positions, cell, pbc} (Å), "
        "optionally energy (eV) and forces (eV/Å)"
    ),
    "structures": [1, MAX_STRUCTURES_PER_CALL],
    "atoms_per_structure": [1, MAX_ATOMS_PER_STRUCTURE],
    "atoms_per_call": [1, MAX_ATOMS_PER_CALL],
    "elements": "the 89 elements of MACE-MP-0 (Z = 1..83 and 89..94)",
    "min_interatomic_distance_angstrom": MIN_INTERATOMIC_DISTANCE,
    "cell": f"3x3 lattice vectors in Å, each shorter than {MAX_CELL_LENGTH} Å; a cell is required whenever any pbc flag is true",
    "validation": (
        "element support, finite coordinates, cell/pbc consistency and interatomic-distance and size ceilings only. "
        "Nothing checks that a structure is chemically sensible, charge-neutral or near equilibrium; a random cloud "
        "of supported atoms is scored without complaint"
    ),
    "units": "energies in eV, forces in eV/Å, stress in eV/Å³ (ASE conventions)",
}


def _atomic_symbols() -> dict[str, int]:
    from ase.data import atomic_numbers

    return dict(atomic_numbers)


def _check_structure(structure: Mapping[str, Any], index: int) -> dict[str, Any]:
    """Return a normalised copy of one structure or raise ValueError describing the first defect."""
    import numpy as np

    label = f"structure[{index}]"
    if not isinstance(structure, Mapping):
        raise ValueError(f"{label} must be a mapping with symbols/positions/cell/pbc, got {type(structure).__name__}")
    symbols = structure.get("symbols")
    positions = structure.get("positions")
    if not isinstance(symbols, (list, tuple)) or not symbols:
        raise ValueError(f"{label}: symbols must be a non-empty list of element symbols")
    n_atoms = len(symbols)
    if n_atoms > MAX_ATOMS_PER_STRUCTURE:
        raise ValueError(f"{label}: {n_atoms} atoms exceed the ceiling of {MAX_ATOMS_PER_STRUCTURE}")
    table = _atomic_symbols()
    numbers = []
    for symbol in symbols:
        if not isinstance(symbol, str) or not _SYMBOL_RE.match(symbol) or symbol not in table:
            raise ValueError(f"{label}: {symbol!r} is not an element symbol")
        z = table[symbol]
        if z not in SUPPORTED_ATOMIC_NUMBERS:
            raise ValueError(f"{label}: element {symbol} (Z={z}) is outside the 89 elements MACE-MP-0 supports")
        numbers.append(z)
    try:
        pos = np.asarray(positions, dtype=float)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{label}: positions must be an (n_atoms, 3) array of numbers") from exc
    if pos.shape != (n_atoms, 3):
        raise ValueError(f"{label}: positions shape {pos.shape} != ({n_atoms}, 3)")
    if not np.all(np.isfinite(pos)) or np.abs(pos).max() > MAX_ABS_COORDINATE:
        raise ValueError(f"{label}: positions must be finite and within ±{MAX_ABS_COORDINATE} Å")
    pbc_raw = structure.get("pbc", False)
    if isinstance(pbc_raw, bool):
        pbc = [pbc_raw] * 3
    else:
        pbc = list(pbc_raw) if isinstance(pbc_raw, (list, tuple)) else None
        if pbc is None or len(pbc) != 3 or not all(isinstance(flag, (bool, int)) for flag in pbc):
            raise ValueError(f"{label}: pbc must be a bool or a list of three bools")
        pbc = [bool(flag) for flag in pbc]
    cell_raw = structure.get("cell")
    cell = None
    if cell_raw is not None:
        try:
            cell = np.asarray(cell_raw, dtype=float)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{label}: cell must be a 3x3 array of numbers") from exc
        if cell.shape != (3, 3) or not np.all(np.isfinite(cell)):
            raise ValueError(f"{label}: cell must be a finite 3x3 array")
        lengths = np.linalg.norm(cell, axis=1)
        if lengths.max() > MAX_CELL_LENGTH:
            raise ValueError(f"{label}: cell vector of {lengths.max():.1f} Å exceeds {MAX_CELL_LENGTH} Å")
        volume = abs(float(np.linalg.det(cell)))
        if any(pbc):
            if volume <= 0.0:
                raise ValueError(f"{label}: periodic cell is singular")
            for axis in range(3):
                if pbc[axis]:
                    cross = np.cross(cell[(axis + 1) % 3], cell[(axis + 2) % 3])
                    height = volume / np.linalg.norm(cross)
                    if height < MIN_CELL_HEIGHT:
                        raise ValueError(
                            f"{label}: periodic cell height {height:.3f} Å along axis {axis} is below {MIN_CELL_HEIGHT} Å"
                        )
    elif any(pbc):
        raise ValueError(f"{label}: pbc requests periodicity but no cell is given")
    from ase.geometry import get_distances

    if n_atoms > 1:
        _, dists = get_distances(pos, cell=cell if any(pbc) else None, pbc=pbc if any(pbc) else None)
        np.fill_diagonal(dists, np.inf)
        d_min = float(dists.min())
        if d_min < MIN_INTERATOMIC_DISTANCE:
            raise ValueError(f"{label}: two atoms are {d_min:.3f} Å apart, below the {MIN_INTERATOMIC_DISTANCE} Å floor")
    else:
        d_min = math.inf
    energy = structure.get("energy")
    forces_raw = structure.get("forces")
    if energy is not None:
        if isinstance(energy, bool) or not isinstance(energy, (int, float)) or not math.isfinite(float(energy)):
            raise ValueError(f"{label}: energy must be a finite number in eV")
        if abs(float(energy)) / n_atoms > MAX_ABS_ENERGY_PER_ATOM:
            raise ValueError(f"{label}: |energy| per atom exceeds {MAX_ABS_ENERGY_PER_ATOM} eV")
    forces = None
    if forces_raw is not None:
        try:
            forces = np.asarray(forces_raw, dtype=float)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{label}: forces must be an (n_atoms, 3) array of numbers") from exc
        if forces.shape != (n_atoms, 3) or not np.all(np.isfinite(forces)) or np.abs(forces).max() > MAX_ABS_FORCE:
            raise ValueError(f"{label}: forces must be a finite ({n_atoms}, 3) array within ±{MAX_ABS_FORCE} eV/Å")
    name = structure.get("name")
    if name is not None and (not isinstance(name, str) or len(name) > 200):
        raise ValueError(f"{label}: name must be a string of at most 200 characters")
    return {
        "name": name if name is not None else f"structure-{index}",
        "symbols": [str(s) for s in symbols],
        "numbers": numbers,
        "positions": pos.tolist(),
        "cell": cell.tolist() if cell is not None else None,
        "pbc": pbc,
        "energy": float(energy) if energy is not None else None,
        "forces": forces.tolist() if forces is not None else None,
        "n_atoms": n_atoms,
        "min_distance": d_min,
        "periodic": any(pbc),
    }


def _check_structures(structures: Any) -> list[dict[str, Any]]:
    if isinstance(structures, Mapping) or not isinstance(structures, Sequence) or isinstance(structures, (str, bytes)):
        raise ValueError("structures must be a list of structure mappings")
    if not structures:
        raise ValueError("at least one structure is required")
    if len(structures) > MAX_STRUCTURES_PER_CALL:
        raise ValueError(f"{len(structures)} structures exceed the ceiling of {MAX_STRUCTURES_PER_CALL} per call")
    checked = [_check_structure(s, i) for i, s in enumerate(structures)]
    total = sum(s["n_atoms"] for s in checked)
    if total > MAX_ATOMS_PER_CALL:
        raise ValueError(f"{total} atoms in one call exceed the ceiling of {MAX_ATOMS_PER_CALL}")
    return checked


def structure_digest(structure: Mapping[str, Any]) -> str:
    """SHA-256 of the geometry (symbols, positions rounded to 1e-6 Å, cell, pbc); labels excluded."""
    checked = _check_structure(structure, 0)
    payload = {
        "symbols": checked["symbols"],
        "positions": [[round(x, 6) for x in row] for row in checked["positions"]],
        "cell": [[round(x, 6) for x in row] for row in checked["cell"]] if checked["cell"] else None,
        "pbc": checked["pbc"],
    }
    return hashlib.sha256(json.dumps(payload, separators=(",", ":")).encode("utf-8")).hexdigest()


def validate_inputs(structures: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Structural validation only; raises ValueError before any model library is imported."""
    checked = _check_structures(structures)
    return {
        "n_structures": len(checked),
        "n_atoms": sum(s["n_atoms"] for s in checked),
        "elements": sorted({s for c in checked for s in c["symbols"]}),
        "periodic": [c["periodic"] for c in checked],
        "min_distance": min(c["min_distance"] for c in checked),
        "labelled": sum(1 for c in checked if c["energy"] is not None and c["forces"] is not None),
    }


def to_atoms(structure: Mapping[str, Any]) -> Any:
    """ase.Atoms for one validated structure (labels attached as `info['energy']` / `arrays['forces']`)."""
    import numpy as np
    from ase import Atoms

    checked = _check_structure(structure, 0)
    atoms = Atoms(symbols=checked["symbols"], positions=np.array(checked["positions"]), pbc=checked["pbc"])
    if checked["cell"] is not None:
        atoms.set_cell(np.array(checked["cell"]))
    if checked["energy"] is not None:
        atoms.info["energy"] = checked["energy"]
    if checked["forces"] is not None:
        atoms.arrays["forces"] = np.array(checked["forces"])
    atoms.info["name"] = checked["name"]
    return atoms


def from_atoms(atoms: Any, *, name: str | None = None) -> dict[str, Any]:
    """Structure mapping from ase.Atoms; picks up energy/forces from `info`/`arrays` or a SinglePointCalculator."""
    import numpy as np

    energy = atoms.info.get("energy")
    forces = atoms.arrays.get("forces")
    calc = getattr(atoms, "calc", None)
    if calc is not None and getattr(calc, "results", None):
        energy = calc.results.get("energy", energy)
        forces = calc.results.get("forces", forces)
    cell = np.array(atoms.get_cell())
    has_cell = bool(np.any(cell != 0.0))
    return {
        "name": name or atoms.info.get("name"),
        "symbols": list(atoms.get_chemical_symbols()),
        "positions": atoms.get_positions().tolist(),
        "cell": cell.tolist() if has_cell else None,
        "pbc": [bool(x) for x in atoms.get_pbc()],
        "energy": float(energy) if energy is not None else None,
        "forces": np.asarray(forces, dtype=float).tolist() if forces is not None else None,
    }


# --------------------------------------------------------------------------------------------------
# pipeline
# --------------------------------------------------------------------------------------------------

TRAINABLE_ALWAYS = ("readouts.", "atomic_energies_fn.", "scale_shift.")
ENERGY_WEIGHT = 100.0  # on the per-atom energy MSE (eV/atom)²
FORCES_WEIGHT = 10.0  # on the force-component MSE (eV/Å)²


def _trainable_prefixes(trainable_blocks: int) -> tuple[str, ...]:
    if not isinstance(trainable_blocks, int) or not 0 <= trainable_blocks <= NUM_INTERACTIONS:
        raise ValueError(f"trainable_blocks must be an int in 0..{NUM_INTERACTIONS}")
    prefixes = list(TRAINABLE_ALWAYS)
    for k in range(NUM_INTERACTIONS - trainable_blocks, NUM_INTERACTIONS):
        prefixes += [f"interactions.{k}.", f"products.{k}."]
    return tuple(prefixes)


@dataclass
class MaceMaterialsPipeline:
    """Energy / force / stress prediction and bounded fine-tuning on top of the verified MACE-MP-0b2 small model."""

    model: Any
    config: dict[str, Any]
    device: str
    weights_dir: Path
    source: str
    adapter: dict[str, Any] | None = None
    _z_table: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str = "cpu",
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        require_source: bool = True,
        report: Callable[[dict[str, Any]], None] | None = None,
    ) -> MaceMaterialsPipeline:
        """Verify, convert if needed, rebuild and strictly load. With `require_source=False` the pickled
        source may be absent (the DIMER-hosted case) as long as the converted pair verifies. `report`
        receives the static-audit summary and the conversion record when a conversion happens."""
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if require_source:
            stage_missing_files(root, allow_download=allow_download)
            snapshot = verify_snapshot(root)
            if not snapshot["converted"]:
                if report is not None:
                    audit = audit_model_file(root / SOURCE_MODEL_NAME)
                    report(
                        {
                            "pickle_audit": {k: v for k, v in audit.items() if k != "globals"},
                            "global_count": len(audit["globals"]),
                        }
                    )
                conversion = convert_model(root)
                if report is not None:
                    report({"conversion": conversion})
                snapshot = verify_snapshot(root)
            elif report is not None:
                report({"conversion": "converted pair already present and digest-verified"})
            source = "converted from the manifest-verified source pickle"
        else:
            # Converted-only deployment: only the code-free pair is verified, whether or not the committed
            # source manifest happens to sit beside it; the pickle is never required or fetched here.
            verify_converted(root)
            source = "converted pair, pinned digests (source pickle not required)"
        config = _config_from_json(json.loads((root / CONVERTED_CONFIG_NAME).read_text(encoding="utf-8")))
        import torch
        from safetensors.torch import load_file

        model = build_model(config)
        state = load_file(str(root / CONVERTED_WEIGHTS_NAME))
        model.load_state_dict(state, strict=True)
        n_params = sum(p.numel() for p in model.parameters())
        if n_params != PARAMETER_COUNT:
            raise ValueError(f"rebuilt model has {n_params} parameters, expected {PARAMETER_COUNT}")
        model.to(torch.device(device)).eval()
        for p in model.parameters():
            p.requires_grad_(False)
        from mace.tools import AtomicNumberTable

        return cls(
            model=model,
            config=config,
            device=device,
            weights_dir=root,
            source=source,
            _z_table=AtomicNumberTable(list(SUPPORTED_ATOMIC_NUMBERS)),
        )

    def reset_to_pretrained(self) -> dict[str, Any]:
        """Return to the pinned base weights: re-verify the converted pair, reload it strictly into the same
        model object and drop any adapter. Calibration, fine-tuning and a loaded artifact all change the
        weights in place, so every experiment that should start from the foundation model calls this first."""
        from safetensors.torch import load_file

        verify_converted(self.weights_dir)
        state = load_file(str(self.weights_dir / CONVERTED_WEIGHTS_NAME))
        self.model.load_state_dict(state, strict=True)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        was_adapted = self.adapter is not None
        self.adapter = None
        return {"reset_to": "pinned base weights (converted pair, digest-verified)", "was_adapted": was_adapted}

    # ---- batching -------------------------------------------------------------------------------------

    def _dataset(self, checked: Sequence[Mapping[str, Any]], *, labels: bool) -> list[Any]:
        from mace.data import AtomicData, config_from_atoms
        from mace.data.utils import KeySpecification

        spec = KeySpecification(info_keys={"energy": "energy"}, arrays_keys={"forces": "forces"})
        data = []
        with _float64_default():  # graph tensors follow the default dtype; the model is float64
            for structure in checked:
                atoms = to_atoms(structure)
                if labels and (structure["energy"] is None or structure["forces"] is None):
                    raise ValueError(f"{structure['name']}: energy and forces are required for this operation")
                conf = config_from_atoms(atoms, key_specification=spec)
                data.append(AtomicData.from_config(conf, z_table=self._z_table, cutoff=R_MAX))
        return data

    def _loader(self, data: Sequence[Any], batch_size: int, shuffle: bool, seed: int = 0) -> Any:
        import torch
        from mace.tools import torch_geometric

        generator = torch.Generator().manual_seed(seed) if shuffle else None
        return torch_geometric.dataloader.DataLoader(
            list(data), batch_size=batch_size, shuffle=shuffle, drop_last=False, generator=generator
        )

    def _forward(self, model: Any, batch: Any, *, training: bool, stress: bool) -> dict[str, Any]:
        import torch

        batch = batch.to(self.device)  # graph tensors are built on the CPU; the model may live on CUDA
        data = batch.to_dict()
        # Forces are -dE/dx, so the forward needs autograd even at inference; no_grad would zero them.
        out = model(data, training=training, compute_force=True, compute_stress=stress)
        if not training:
            out = {k: (v.detach() if isinstance(v, torch.Tensor) else v) for k, v in out.items()}
        return out

    # ---- inference ------------------------------------------------------------------------------------

    def predict(self, structures: Sequence[Mapping[str, Any]], *, batch_size: int = 8) -> dict[str, Any]:
        """Total energy (eV), per-atom energies, forces (eV/Å) and, for fully periodic structures, stress (eV/Å³)."""
        checked = _check_structures(structures)
        started = time.perf_counter()
        data = self._dataset(checked, labels=False)
        model = self.model
        # Stress is a batch-level switch in the MACE forward, so fully periodic structures (stress-eligible)
        # and everything else are batched separately; results are put back in the caller's order.
        eligible = [i for i, s in enumerate(checked) if s["periodic"] and all(s["pbc"])]
        others = [i for i in range(len(checked)) if i not in set(eligible)]
        results: list[dict[str, Any] | None] = [None] * len(checked)
        for indices, stress in ((eligible, True), (others, False)):
            if not indices:
                continue
            position = 0
            for batch in self._loader([data[i] for i in indices], batch_size, shuffle=False):
                out = self._forward(model, batch, training=False, stress=stress)
                ptr = batch.ptr.tolist()
                energies = out["energy"].cpu().tolist()
                node_energy = out["node_energy"].cpu()
                forces = out["forces"].cpu()
                stresses = out.get("stress")
                for i in range(len(ptr) - 1):
                    lo, hi = ptr[i], ptr[i + 1]
                    results[indices[position]] = {
                        "energy": float(energies[i]),
                        "energy_per_atom": float(energies[i]) / (hi - lo),
                        "node_energies": node_energy[lo:hi].tolist(),
                        "forces": forces[lo:hi].tolist(),
                        "stress": stresses[i].cpu().tolist() if (stress and stresses is not None) else None,
                    }
                    position += 1
        if any(r is None for r in results):
            raise RuntimeError("prediction did not cover every structure")
        for structure, result in zip(checked, results, strict=True):
            result["name"] = structure["name"]
            result["n_atoms"] = structure["n_atoms"]
            result["periodic"] = structure["periodic"]
            if not structure["periodic"] or not all(structure["pbc"]):
                result["stress"] = None
        return {
            "model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": MODEL_KEY, "adapted": self.adapter is not None},
            "units": {"energy": "eV", "forces": "eV/Å", "stress": "eV/Å³"},
            "results": results,
            "seconds": round(time.perf_counter() - started, 3),
        }

    def _predict_in_chunks(
        self, structures: Sequence[Mapping[str, Any]], *, batch_size: int = 8, check_only: bool = False
    ) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
        """`predict` over a list of any length: consecutive chunks that each respect the per-call ceilings
        (MAX_STRUCTURES_PER_CALL structures, MAX_ATOMS_PER_CALL atoms). Returns (checked, results); with
        `check_only` it validates every structure and returns (checked, [])."""
        if isinstance(structures, Mapping) or not isinstance(structures, Sequence) or isinstance(structures, (str, bytes)):
            raise ValueError("structures must be a list of structure mappings")
        if not structures:
            raise ValueError("at least one structure is required")
        checked_all = [_check_structure(s, i) for i, s in enumerate(structures)]
        if check_only:
            return checked_all, []
        chunks: list[list[dict[str, Any]]] = [[]]
        atoms = 0
        for structure in checked_all:
            full = len(chunks[-1]) >= MAX_STRUCTURES_PER_CALL or atoms + structure["n_atoms"] > MAX_ATOMS_PER_CALL
            if chunks[-1] and full:
                chunks.append([])
                atoms = 0
            chunks[-1].append(structure)
            atoms += structure["n_atoms"]
        results: list[dict[str, Any]] = []
        for chunk in chunks:
            results.extend(self.predict(chunk, batch_size=batch_size)["results"])
        return checked_all, results

    def evaluate(self, structures: Sequence[Mapping[str, Any]], *, batch_size: int = 8) -> dict[str, Any]:
        """Energy and force errors against the reference labels carried by the structures. Any number of
        structures: they are scored in chunks that respect the per-call ceilings of `predict`."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import regression_metrics` removed — names are kernel globals defined by the carried modules

        checked, _ = self._predict_in_chunks(structures, batch_size=batch_size, check_only=True)
        if any(s["energy"] is None or s["forces"] is None for s in checked):
            raise ValueError("every structure needs energy and forces for evaluation")
        checked, results = self._predict_in_chunks(checked, batch_size=batch_size)
        return regression_metrics(checked, results)

    # ---- adaptation -----------------------------------------------------------------------------------

    def calibrate_e0(self, structures: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Least-squares per-element energy offsets between the reference labels and the current model,
        added to the model's atomic reference energies. Two-parameter change on a Cu/Al dataset; the
        cheapest possible adaptation to a different level of theory. Returns the shifts in eV."""
        import numpy as np
        import torch

        checked, results = self._predict_in_chunks(structures)
        if any(s["energy"] is None for s in checked):
            raise ValueError("every structure needs an energy for E0 calibration")
        elements = sorted({s for c in checked for s in c["symbols"]})
        counts = np.array([[c["symbols"].count(el) for el in elements] for c in checked], dtype=float)
        residual = np.array([c["energy"] - r["energy"] for c, r in zip(checked, results, strict=True)])
        shifts, *_ = np.linalg.lstsq(counts, residual, rcond=None)
        table = _atomic_symbols()
        with torch.no_grad():
            energies = self.model.atomic_energies_fn.atomic_energies
            for element, shift in zip(elements, shifts, strict=True):
                energies[self._z_table.z_to_index(table[element])] += float(shift)
        return {"elements": elements, "shifts_ev": [float(s) for s in shifts], "n_structures": len(checked)}

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 6,
        lr: float = 1e-3,
        batch_size: int = 4,
        trainable_blocks: int = 1,
        energy_weight: float = ENERGY_WEIGHT,
        forces_weight: float = FORCES_WEIGHT,
        calibrate: bool = True,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning to the reference energies and forces of `train`.

        Always trains the readouts, the per-element reference energies and the scale/shift block;
        `trainable_blocks` (0..2) additionally unfreezes that many trailing interaction + product
        blocks (1 = the last block, 1.92 M of 8.22 M parameters). Loss = energy_weight × MSE(per-atom
        energy) + forces_weight × MSE(force components), Adam, fixed learning rate, no scheduler. The
        epoch with the lowest validation loss is kept; epoch 0 records the (E0-calibrated) frozen model
        so every number is comparable to the starting point.

        An adapted pipeline (after `adapt` or `load_artifact`) is refused: training would continue from the
        adapted weights while epoch 0 still claimed to be the frozen model, and the exported adapter would not
        describe every change from the base. Call `reset_to_pretrained()` first."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if self.adapter is not None:
            raise ValueError(
                "this pipeline is already adapted; call pipe.reset_to_pretrained() to return to the pinned base "
                "weights before adapting again (otherwise training would stack on the previous adaptation)"
            )
        if not isinstance(epochs, int) or not 1 <= epochs <= 50:
            raise ValueError("epochs must be an int in 1..50")
        if not (0.0 < lr <= 0.1):
            raise ValueError("lr must be in (0, 0.1]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= MAX_STRUCTURES_PER_CALL:
            raise ValueError(f"batch_size must be an int in 1..{MAX_STRUCTURES_PER_CALL}")
        prefixes = _trainable_prefixes(trainable_blocks)
        train_checked = validate_dataset(train, min_records=4)["records"]
        val_checked = validate_dataset(val, min_records=1)["records"] if val else []
        import torch

        torch.manual_seed(seed)
        started = time.perf_counter()
        model = self.model
        pre_state = copy.deepcopy(model.state_dict())  # restored if anything below raises
        calibration = None
        for name, param in model.named_parameters():
            param.requires_grad_(name.startswith(prefixes))
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        n_total = sum(p.numel() for p in model.parameters())
        optimiser = torch.optim.Adam(params, lr=lr)
        train_data = self._dataset(train_checked, labels=True)
        val_data = self._dataset(val_checked, labels=True) if val_checked else []

        def weighted_loss(out: Mapping[str, Any], batch: Any) -> Any:
            device = out["energy"].device
            n_atoms = batch.ptr.to(device).diff().to(out["energy"].dtype)
            e_loss = torch.mean(((out["energy"] - batch.energy.to(device)) / n_atoms) ** 2)
            f_loss = torch.mean((out["forces"] - batch.forces.to(device)) ** 2)
            return energy_weight * e_loss + forces_weight * f_loss

        def val_loss() -> float | None:
            if not val_data:
                return None
            model.eval()
            total, count = 0.0, 0
            for batch in self._loader(val_data, batch_size, shuffle=False):
                out = self._forward(model, batch, training=False, stress=False)
                total += float(weighted_loss(out, batch)) * batch.num_graphs
                count += batch.num_graphs
            return total / count

        try:
            calibration = self.calibrate_e0(train_checked) if calibrate else None
            history: list[dict[str, Any]] = []
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = 0
            entry: dict[str, Any] = {
                "epoch": 0,
                "train_loss": None,
                "val_loss": val_loss(),
                "note": "frozen model after E0 calibration" if calibrate else "frozen model",
            }
            if val_checked:
                entry["val"] = self.evaluate(val_checked)
            history.append(entry)
            best_val = entry["val_loss"] if entry["val_loss"] is not None else math.inf
            if progress:
                progress(entry)
            for epoch in range(1, epochs + 1):
                model.train()
                total, count = 0.0, 0
                for batch in self._loader(train_data, batch_size, shuffle=True, seed=seed + epoch):
                    out = self._forward(model, batch, training=True, stress=False)
                    loss = weighted_loss(out, batch)
                    optimiser.zero_grad(set_to_none=True)
                    loss.backward()
                    optimiser.step()
                    total += float(loss.detach()) * batch.num_graphs
                    count += batch.num_graphs
                model.eval()
                entry = {"epoch": epoch, "train_loss": total / count, "val_loss": val_loss()}
                if val_checked:
                    entry["val"] = self.evaluate(val_checked)
                history.append(entry)
                if progress:
                    progress(entry)
                if entry["val_loss"] is None or entry["val_loss"] < best_val:
                    best_val = entry["val_loss"] if entry["val_loss"] is not None else best_val
                    best_state = copy.deepcopy(model.state_dict())
                    best_epoch = epoch
        except BaseException:
            # Transactional: a failure in calibration, training, validation or the progress callback
            # leaves the model exactly as it was before adapt(), frozen, with no adapter attached.
            model.load_state_dict(pre_state, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        model.load_state_dict(best_state, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable_prefixes": list(prefixes),
            "trainable_blocks": trainable_blocks,
            "n_trainable": n_trainable,
            "n_total": n_total,
            "epochs": epochs,
            "best_epoch": best_epoch,
            "lr": lr,
            "batch_size": batch_size,
            "energy_weight": energy_weight,
            "forces_weight": forces_weight,
            "calibration": calibration,
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted tensors (only those the adaptation could change) as safetensors plus a manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        prefixes = tuple(self.adapter["trainable_prefixes"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in self.model.state_dict().items() if k.startswith(prefixes)}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "converted_sha256": dict(CONVERTED_SHA256),
            },
            "adapter": {k: v for k, v in self.adapter.items() if k != "history"},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": _sha256_file(weights_path)}
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return out

    def _check_artifact_manifest(self, root: Path, manifest: Mapping[str, Any]) -> tuple[Path, list[str]]:
        """Validate an adapter manifest before anything is deserialised: format and version, the pinned base,
        exactly one weights entry named `adapter.safetensors` inside the artifact directory, an in-range
        `trainable_blocks`, and a tensor list equal to the exact set that scope implies for this base."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not supported "
                f"(expected {ARTIFACT_FORMAT_VERSION!r})"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError("artifact was adapted from a different base model or revision")
        if base.get("converted_sha256") != CONVERTED_SHA256:
            raise ValueError("artifact records different converted-base digests")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one weights file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact weights file must be named {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weights file must sit inside the artifact directory")
        adapter = manifest.get("adapter")
        if not isinstance(adapter, Mapping):
            raise ValueError("artifact manifest has no adapter record")
        trainable_blocks = adapter.get("trainable_blocks")
        if isinstance(trainable_blocks, bool) or not isinstance(trainable_blocks, int):
            raise ValueError("artifact adapter.trainable_blocks must be an int")
        prefixes = _trainable_prefixes(trainable_blocks)  # range-checked there
        expected = sorted(k for k in self.model.state_dict() if k.startswith(prefixes))
        if not isinstance(manifest.get("tensors"), list) or sorted(manifest["tensors"]) != expected:
            raise ValueError(
                f"artifact tensor list does not match the {len(expected)} tensors that "
                f"trainable_blocks={trainable_blocks} may change on this base"
            )
        return weights_path, expected

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, scope and digest, then overwrite exactly the tensors the scope allows."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path, expected = self._check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        digest = _sha256_file(weights_path)
        if digest != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from the validated manifest")
        state = self.model.state_dict()
        for key, value in tensors.items():
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(f"artifact tensor {key} has shape {tuple(value.shape)}, base has {tuple(state[key].shape)}")
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        self.model.load_state_dict(merged, strict=True)
        self.model.eval()
        self.adapter = {**manifest["adapter"], "history": manifest.get("history", [])}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str = "cpu",
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        require_source: bool = True,
    ) -> MaceMaterialsPipeline:
        pipeline = cls.from_pretrained(
            device=device, weights_dir=weights_dir, allow_download=allow_download, require_source=require_source
        )
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 3/3:** `src/mace_materials_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
# @title Infrastructure: carried module `src/mace_materials_pipeline/samples.py` (you may run it without reading it)
"""Tutorial dataset, dataset validation, stratified split and extended-XYZ I/O for the MACE pipeline.

The default dataset is generated in code (no download): rattled and strained face-centred-cubic
2×2×2 supercells of copper, aluminium and a random Cu₁₆Al₁₆ substitutional alloy, **labelled with
ASE's effective-medium-theory potential (EMT)**. EMT is a real, deterministic, analytic interatomic
potential for these metals — not a DFT code and not a surrogate of MACE — so the labels are a
different level of theory from the PBE data MACE-MP-0 was trained on. That is exactly the situation a
foundation potential is fine-tuned in: the frozen model already predicts sensible forces, but its
energies sit on a different reference and its curvature differs, and the adaptation has to close the
gap on held-out structures.

Design choices made so the trivial baselines sit where they should:

* Every structure of one composition has the **same atom count and the same element multiset**, so a
  per-element energy regression (the composition baseline) predicts one energy per composition and
  cannot see the rattle or the strain at all.
* Displacement amplitudes and strains are drawn independently of composition, so nothing about a
  structure's label is recoverable from which elements it contains.
* Structures are labelled with forces as well as energies, so the force error — where the foundation
  model already starts well and must improve further — is measured alongside the energy error.
"""

from __future__ import annotations

import hashlib
import json
import random
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_ATOMS_PER_STRUCTURE, _check_structure, from_atoms, structure_digest, to_atoms` removed — names are kernel globals defined by the carried modules

SAMPLE_COMPOSITIONS = ("Cu32", "Al32", "Cu16Al16")
SAMPLE_SIZE = 48
LATTICE_CONSTANTS = {"Cu": 3.60, "Al": 4.05}  # Å, fcc conventional cell
SUPERCELL = (2, 2, 2)  # 32 atoms
STRAIN_RANGE = 0.03  # isotropic, ±3 %
RATTLE_SIGMAS = (0.05, 0.10, 0.15)  # Å, Gaussian displacement per Cartesian component
SAMPLE_LABEL_SOURCE = "ase.calculators.emt.EMT"
MIN_RECORDS = 8
MAX_DATASET_RECORDS = 1_000  # BYOD ceiling: about an hour of CPU fine-tuning at the default epochs
# Smallest split sizes the notebook runs end to end with: adapt() needs 4 training structures, Section 8 scores a
# non-empty validation split and every section reads a non-empty test split.
MIN_SPLIT_SIZES = {"train": 4, "val": 1, "test": 1}
# How split_dataset keeps related structures together (SPL3/SPL5). "stratified" treats every structure as an
# independent draw and is only right for data like the generated sample; frames of one trajectory or relaxation are
# near-duplicates and must not straddle splits, so BYOD defaults to "auto".
SPLIT_MODES = ("auto", "group", "composition", "stratified")
GROUP_INFO_KEYS = ("group", "trajectory")  # extended-XYZ per-frame info keys read by load_byod_dataset


def _emt_energy_forces(atoms: Any) -> tuple[float, list[list[float]]]:
    from ase.calculators.emt import EMT

    atoms = atoms.copy()
    atoms.calc = EMT()
    energy = float(atoms.get_potential_energy())
    forces = atoms.get_forces().tolist()
    atoms.calc = None
    return energy, forces


def build_sample_structure(composition: str, *, strain: float, sigma: float, rng: random.Random) -> dict[str, Any]:
    """One rattled, strained fcc 2×2×2 supercell of the named composition (geometry only, no labels)."""
    import numpy as np
    from ase.build import bulk

    if composition not in SAMPLE_COMPOSITIONS:
        raise ValueError(f"composition must be one of {SAMPLE_COMPOSITIONS}")
    if composition == "Cu16Al16":
        a0 = 0.5 * (LATTICE_CONSTANTS["Cu"] + LATTICE_CONSTANTS["Al"])  # Vegard's law
        atoms = bulk("Cu", "fcc", a=a0, cubic=True) * SUPERCELL
        symbols = ["Cu"] * len(atoms)
        for index in rng.sample(range(len(atoms)), 16):
            symbols[index] = "Al"
        atoms.set_chemical_symbols(symbols)
    else:
        element = composition[:2]
        atoms = bulk(element, "fcc", a=LATTICE_CONSTANTS[element], cubic=True) * SUPERCELL
    atoms.set_cell(atoms.cell * (1.0 + strain), scale_atoms=True)
    displacement = np.array([[rng.gauss(0.0, sigma) for _ in range(3)] for _ in range(len(atoms))])
    atoms.positions = atoms.positions + displacement
    structure = from_atoms(atoms)
    structure["name"] = f"{composition}-s{strain:+.4f}-r{sigma:.2f}"
    structure["composition"] = composition
    structure["strain"] = strain
    structure["rattle_sigma"] = sigma
    return structure


def generate_sample_dataset(*, seed: int = 42, size: int = SAMPLE_SIZE) -> list[dict[str, Any]]:
    """Deterministic EMT-labelled dataset: `size` structures cycling through the three compositions."""
    if not isinstance(size, int) or not 3 <= size <= 96 or size % 3:
        raise ValueError("size must be a multiple of 3 in 3..96")
    rng = random.Random(seed)
    records = []
    for index in range(size):
        composition = SAMPLE_COMPOSITIONS[index % len(SAMPLE_COMPOSITIONS)]
        strain = rng.uniform(-STRAIN_RANGE, STRAIN_RANGE)
        sigma = RATTLE_SIGMAS[rng.randrange(len(RATTLE_SIGMAS))]
        structure = build_sample_structure(composition, strain=strain, sigma=sigma, rng=rng)
        structure["name"] = f"{structure['name']}-{index:02d}"
        energy, forces = _emt_energy_forces(to_atoms(structure))
        structure["energy"] = energy
        structure["forces"] = forces
        structure["label_source"] = SAMPLE_LABEL_SOURCE
        records.append(structure)
    return records


def composition_key(structure: Mapping[str, Any]) -> str:
    """Hill-order formula string, e.g. Al16Cu16, used to stratify splits."""
    counts: dict[str, int] = {}
    for symbol in structure["symbols"]:
        counts[symbol] = counts.get(symbol, 0) + 1
    return "".join(f"{el}{counts[el]}" for el in sorted(counts))


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    require_labels: bool = True,
) -> dict[str, Any]:
    """Structural validation of a labelled dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of structure mappings")
    if len(records) < min_records:
        raise ValueError(f"{len(records)} records; at least {min_records} are required")
    if len(records) > MAX_DATASET_RECORDS:
        raise ValueError(f"{len(records)} records exceed the dataset ceiling of {MAX_DATASET_RECORDS}; use a subset")
    checked = []
    names: set[str] = set()
    for index, record in enumerate(records):
        structure = _check_structure(record, index)
        if require_labels and (structure["energy"] is None or structure["forces"] is None):
            raise ValueError(f"{structure['name']}: energy and forces are required")
        if structure["name"] in names:
            raise ValueError(f"duplicate structure name {structure['name']!r}")
        names.add(structure["name"])
        structure["composition"] = composition_key(structure)
        for key in ("strain", "rattle_sigma", "label_source"):
            if key in record:
                structure[key] = record[key]
        group = record.get("group")
        if group is not None:
            if not isinstance(group, (str, int)) or isinstance(group, bool) or not str(group).strip():
                raise ValueError(f"{structure['name']}: group must be a non-empty string or integer")
            structure["group"] = str(group)
        checked.append(structure)
    compositions: dict[str, int] = {}
    for structure in checked:
        compositions[structure["composition"]] = compositions.get(structure["composition"], 0) + 1
    return {
        "records": checked,
        "n_records": len(checked),
        "n_atoms": sum(s["n_atoms"] for s in checked),
        "compositions": compositions,
        "elements": sorted({s for c in checked for s in c["symbols"]}),
        "max_atoms": max(s["n_atoms"] for s in checked),
        "labelled": all(s["energy"] is not None and s["forces"] is not None for s in checked),
        "digest": dataset_digest(checked),
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """SHA-256 over per-structure geometry digests and labels rounded to 1e-6."""
    parts = []
    for record in records:
        energy = None if record.get("energy") is None else round(float(record["energy"]), 6)
        forces = None if record.get("forces") is None else [[round(float(x), 6) for x in row] for row in record["forces"]]
        parts.append([structure_digest(record), energy, forces])
    return hashlib.sha256(json.dumps(parts, separators=(",", ":")).encode("utf-8")).hexdigest()


def _effective_split_mode(checked: Sequence[Mapping[str, Any]], group_by: str) -> str:
    if group_by not in SPLIT_MODES:
        raise ValueError(f"group_by must be one of {SPLIT_MODES}")
    tagged = sum(1 for s in checked if s.get("group") is not None)
    if group_by == "group" and tagged != len(checked):
        raise ValueError(
            f"group_by='group' needs a group tag on every structure; {len(checked) - tagged} of {len(checked)} have none. "
            "Add group=<id> (or trajectory=<id>) to each frame's comment line in the extended XYZ, or use 'composition'"
        )
    if group_by == "auto":
        return "group" if tagged == len(checked) else "composition"
    return group_by


def split_group_key(structure: Mapping[str, Any], mode: str) -> str:
    """The unit a split keeps together: the structure's group tag in 'group' mode, else its composition."""
    return str(structure["group"]) if mode == "group" else composition_key(structure)


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.2,
    test_fraction: float = 0.25,
    seed: int = 0,
    group_by: str = "stratified",
) -> dict[str, list[dict[str, Any]]]:
    """Split a labelled dataset into train / val / test.

    `group_by="stratified"` shuffles within each composition and cuts the fractions there, so every split sees
    every composition: right only when every structure is an independent draw (the generated sample).
    `"group"` (every record carries a `group` tag, e.g. a trajectory id), `"composition"` (one group per formula)
    and `"auto"` (group tags when every record has one, else composition) assign whole groups to one split, so
    near-duplicate frames never straddle train and test; the fractions then count groups, not structures.
    Every split must reach MIN_SPLIT_SIZES, and every element of the validation and test splits must occur in
    training (the composition baseline and the E0 calibration are fitted there); otherwise a ValueError names
    the rule that failed and what to change."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records, min_records=4)["records"]
    mode = _effective_split_mode(checked, group_by)
    rng = random.Random(seed)
    splits: dict[str, list[dict[str, Any]]] = {"train": [], "val": [], "test": []}
    if mode == "stratified":
        by_composition: dict[str, list[dict[str, Any]]] = {}
        for structure in checked:
            by_composition.setdefault(structure["composition"], []).append(structure)
        for composition in sorted(by_composition):
            group = list(by_composition[composition])
            rng.shuffle(group)
            n_test = max(1, round(len(group) * test_fraction))
            n_val = round(len(group) * val_fraction)
            splits["test"].extend(group[:n_test])
            splits["val"].extend(group[n_test : n_test + n_val])
            splits["train"].extend(group[n_test + n_val :])
    else:
        groups: dict[str, list[dict[str, Any]]] = {}
        for structure in checked:
            groups.setdefault(split_group_key(structure, mode), []).append(structure)
        keys = sorted(groups)
        if len(keys) < 3:
            raise ValueError(
                f"a {mode} split needs at least 3 groups (one each for train, validation and test); found "
                f"{len(keys)}: {keys}. Supply more trajectories or compositions, tag frames with group=<id>, or, only "
                "if every structure is an independent draw, use 'stratified'"
            )
        rng.shuffle(keys)
        n_val = max(1, round(len(keys) * val_fraction)) if val_fraction > 0 else 0
        n_test = min(max(1, round(len(keys) * test_fraction)), len(keys) - n_val - 1)
        for key in keys[:n_test]:
            splits["test"].extend(groups[key])
        for key in keys[n_test : n_test + n_val]:
            splits["val"].extend(groups[key])
        for key in keys[n_test + n_val :]:
            splits["train"].extend(groups[key])
    for name, minimum in MIN_SPLIT_SIZES.items():
        if len(splits[name]) < minimum:
            what = {"train": "training", "val": "validation", "test": "test"}[name]
            raise ValueError(
                f"the {what} split has {len(splits[name])} structure(s); at least {minimum} are needed "
                f"(splits {len(splits['train'])}/{len(splits['val'])}/{len(splits['test'])} with {mode} grouping). "
                "Supply more structures or groups, or change VAL_FRACTION / TEST_FRACTION"
            )
    train_elements = {s for record in splits["train"] for s in record["symbols"]}
    for name in ("val", "test"):
        missing = sorted({s for record in splits[name] for s in record["symbols"]} - train_elements)
        if missing:
            raise ValueError(
                f"elements {missing} occur in the {name} split but not in training, so the composition baseline and "
                "the E0 calibration cannot cover them; change SEED, add structures containing them to more groups, "
                "or regroup the data"
            )
    return splits


def split_report(splits: Mapping[str, Sequence[Mapping[str, Any]]], *, group_by: str = "stratified") -> dict[str, Any]:
    """Which groups landed in which split, and any group found in more than one split (expected only for
    'stratified', where compositions are shared by design)."""
    everything = [s for name in ("train", "val", "test") for s in splits[name]]
    mode = _effective_split_mode(everything, group_by)
    key_mode = "group" if mode == "group" else "composition"
    members = {name: sorted({split_group_key(s, key_mode) for s in splits[name]}) for name in ("train", "val", "test")}
    train, val, test = (set(members[n]) for n in ("train", "val", "test"))
    return {
        "mode": mode,
        "unit": "group tag" if mode == "group" else "composition",
        "sizes": {name: len(splits[name]) for name in ("train", "val", "test")},
        "groups": members,
        "groups_in_more_than_one_split": sorted((train & (val | test)) | (val & test)),
    }


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read an extended-XYZ file (`ase.io.read`, all frames) into structure mappings with labels from
    `energy` / `forces` (info/arrays or the attached single-point results). A per-frame `group=<id>` (or
    `trajectory=<id>`) on the comment line becomes the record's `group`, which `split_dataset` keeps together."""
    import ase.io

    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    if file_path.suffix.lower() not in (".xyz", ".extxyz"):
        raise ValueError("BYOD datasets must be extended XYZ (.xyz / .extxyz)")
    frames = ase.io.read(str(file_path), index=":", format="extxyz")
    if not isinstance(frames, list):
        frames = [frames]
    records = []
    for index, atoms in enumerate(frames):
        if len(atoms) > MAX_ATOMS_PER_STRUCTURE:
            raise ValueError(f"frame {index}: {len(atoms)} atoms exceed {MAX_ATOMS_PER_STRUCTURE}")
        record = from_atoms(atoms, name=atoms.info.get("name") or f"{file_path.stem}-{index:04d}")
        for key in GROUP_INFO_KEYS:
            if atoms.info.get(key) is not None:
                record["group"] = str(atoms.info[key])
                break
        records.append(record)
    return records


def write_dataset_xyz(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write structures (with energy/forces when present) as extended XYZ, the shape BYOD expects."""
    import ase.io
    from ase.calculators.singlepoint import SinglePointCalculator

    frames = []
    for record in records:
        atoms = to_atoms(record)
        if record.get("group") is not None:
            atoms.info["group"] = str(record["group"])  # kept so a re-read file splits the same way
        atoms.info.pop("energy", None)
        forces = atoms.arrays.pop("forces", None)
        if record.get("energy") is not None and forces is not None:
            atoms.calc = SinglePointCalculator(atoms, energy=float(record["energy"]), forces=forces)
        frames.append(atoms)
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    ase.io.write(str(out), frames, format="extxyz")
    return out

## 3. Pin, stage and verify the model

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `2`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `e291ace2bfae…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `MaceMaterialsPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), report=print)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "mace-mp-0b2-small",
  "modelId": "mace-foundations/mace-mp-0",
  "revision": "e291ace2bfae073c3ebc7ae2f9479a525989baa7",
  "files": [
    {
      "path": "README.md",
      "bytes": 24,
      "sha256": "d8d7a46d41a1a37fe4f0a5f637bf55c649310185329127d8a2204632e480be17"
    },
    {
      "path": "mace-mp-0b2-small.model",
      "bytes": 67622684,
      "sha256": "d5773bf9440e96d6eb8c598f84bd0e6369fcfa432f626a87f890e07da3c651c9",
      "note": "Pickled torch module (executable serialization, asset spec 11.2). Byte-identical to the upstream GitHub release asset ACEsuit/mace-mp mace_mp_0b2/mace-small-density-agnesi-stress.model. Never served: statically audited and converted once into mace-mp-0b2-small.config.json + mace-mp-0b2-small.safetensors (digests pinned in pipeline.py)."
    }
  ],
  "totalBytes": 67622708
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = MaceMaterialsPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), report=print)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

{'model_id': 'mace-foundations/mace-mp-0', 'revision': 'e291ace2bfae073c3ebc7ae2f9479a525989baa7', 'license': 'mit', 'files': 2, 'total_bytes': 67622708}


{'weights_dir': '/content/weights/mace-mp-0b2-small', 'fetched': ['README.md', 'mace-mp-0b2-small.model']}
{'verified_files': 2, 'revision': 'e291ace2bfae073c3ebc7ae2f9479a525989baa7'}


{'pickle_audit': {'nested_archives': 18, 'fx_code_strings': 28, 'import_blocks': 10, 'torchscript_sources': 30, 'generated_code_import_lines': ['NoneType = type(None)', 'from math import inf', 'from math import nan', 'from torch import device', 'import torch', 'import torch.fx._pytree as fx_pytree', 'import torch.utils._pytree as pytree'], 'violations': [], 'audit_sha256': '9bb150f1ecc9212e594313ba0ef9aa4884b7634dfc8a11da8cbe0c9ebf70d1e5'}, 'global_count': 61}


/content/dimer_isolated_env/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
/content/dimer_isolated_env/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


/content/dimer_isolated_env/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


{'conversion': {'source': {'path': 'mace-mp-0b2-small.model', 'bytes': 67622684, 'sha256': 'd5773bf9440e96d6eb8c598f84bd0e6369fcfa432f626a87f890e07da3c651c9'}, 'converted': [{'path': 'mace-mp-0b2-small.config.json', 'bytes': 3279, 'sha256': '130b641179c5da6dbf6ec439b9c20661e3d60512007f6f2d42669d4f87fc9893'}, {'path': 'mace-mp-0b2-small.safetensors', 'bytes': 67400566, 'sha256': '2ed99065c4decf21613b7038dde6bda2b694b2f414012c0ca743f7ff7f86fe93'}], 'dropped_keys': ['pair_repulsion_fn.cutoff.p', 'pair_repulsion_fn.cutoff.r_max', 'pair_repulsion_fn.r_max'], 'added_flag_keys': ['products.0.symmetric_contractions.contractions.0.weights_0_zeroed', 'products.0.symmetric_contractions.contractions.0.weights_1_zeroed', 'products.0.symmetric_contractions.contractions.0.weights_max_zeroed', 'products.1.symmetric_contractions.contractions.0.weights_0_zeroed', 'products.1.symmetric_contractions.contractions.0.weights_1_zeroed', 'products.1.symmetric_contractions.contractions.0.weights_max_zeroed'], '

{'device': 'cuda', 'source': 'converted from the manifest-verified source pickle'}


## 4. Sample dataset, validation and split

The default dataset is generated in code with a fixed seed: 16 structures each of Cu₃₂, Al₃₂ and a random substitutional Cu₁₆Al₁₆ alloy — 2×2×2 fcc supercells with an isotropic strain drawn from ±3 % and Gaussian rattles of σ = 0.05, 0.10 or 0.15 Å — labelled with energies and forces from ASE's EMT calculator. `validate_dataset` checks every structure (element support, distances, cell/pbc, label shapes) and the dataset contract before any model runs.

**How the split keeps structures independent.** Frames of one molecular-dynamics trajectory or one relaxation are near-duplicates: if some land in training and their neighbours in the test split, the test error measures memory, not generalisation. `split_dataset` therefore has two kinds of mode. The generated sample uses `'stratified'`: every structure is its own independent draw (its own strain, rattle and alloy arrangement), so it shuffles within each composition and cuts 20 % validation / 25 % test, and every split sees every composition. BYOD data uses `BYOD_SPLIT` (default `'auto'`): whole groups go to one split — the `group=<id>` or `trajectory=<id>` tags on your frames when every frame has one, otherwise whole compositions — and the fractions then count groups, not structures. The cell prints which groups landed where and asserts that no group sits in two splits. Use `'stratified'` for your own data only if every structure is an independent draw.

**Predict before running:** with 48 structures, three compositions and the 20 % / 25 % fractions, how many structures will each split hold, and will any composition be missing from the test split?

Four refusal probes follow the split — an unsupported element, overlapping atoms, periodicity without a cell and a malformed force array — each rejected before `torch` does anything.

In [ ]:
import json
import os
from pathlib import Path

USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ''  # @param {type:"string"}
BYOD_SPLIT = 'auto'  # @param ["auto", "group", "composition", "stratified"]
VAL_FRACTION = 0.2  # @param {type:"number"}
TEST_FRACTION = 0.25  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
# BYOD outputs carry their own names, so a BYOD run never overwrites (or is mistaken for) the sample's.
OUTPUTS = {
    'dataset': 'outputs/mace_materials_byod_dataset.xyz' if USE_BYOD else 'outputs/mace_materials_sample_dataset.xyz',
    'report': 'outputs/mace_materials_byod_evaluation_report.json' if USE_BYOD else 'outputs/mace_materials_evaluation_report.json',
    'predictions': 'outputs/mace_materials_byod_predictions.json' if USE_BYOD else 'outputs/mace_materials_predictions.json',
    'adapter': 'outputs/mace_materials_byod_adapter' if USE_BYOD else 'outputs/mace_materials_adapter',
    'result': 'outputs/mace_materials_byod_result.json' if USE_BYOD else 'outputs/mace_materials_result.json',
    'activity_adapter': 'outputs/mace_materials_byod_activity_adapter' if USE_BYOD else 'outputs/mace_materials_activity_adapter',
}
if USE_BYOD:
    print({'expected_byod_input': 'one extended-XYZ file: per frame a Lattice and pbc when periodic, energy (eV), forces (eV/A), optional group=<id> or trajectory=<id>', 'records': f'{MIN_RECORDS}..{MAX_DATASET_RECORDS}', 'split': BYOD_SPLIT, 'source': 'BYOD_PATH' if BYOD_PATH else 'Colab upload dialog'})
    if BYOD_PATH:
        byod_path = Path(BYOD_PATH)
        if not byod_path.is_file():
            raise FileNotFoundError(f'BYOD_PATH {BYOD_PATH!r} is not a file in this runtime: give the path of one extended-XYZ file and run this cell again')
        data_source = f'BYOD file {byod_path.name} (BYOD_PATH)'
    else:
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError('USE_BYOD = True needs a file: this runtime has no Colab upload dialog, so set BYOD_PATH to the path of one extended-XYZ file and run this cell again') from None
        uploaded = files.upload()
        if not uploaded:
            raise ValueError('no file uploaded; run this cell again and choose one extended-XYZ file')
        if len(uploaded) != 1:
            raise ValueError(f'{len(uploaded)} files uploaded; run this cell again and choose exactly one extended-XYZ file')
        file_name, payload = next(iter(uploaded.items()))
        byod_path = Path('work') / Path(file_name).name
        byod_path.parent.mkdir(parents=True, exist_ok=True)
        byod_path.write_bytes(payload)
        data_source = f'BYOD upload {byod_path.name}'
    records = load_byod_dataset(byod_path)
    split_mode = BYOD_SPLIT
else:
    records = generate_sample_dataset(seed=SEED)
    data_source = f'generated fcc supercells labelled by {SAMPLE_LABEL_SOURCE} (seed {SEED}, {SAMPLE_SIZE} structures)'
    split_mode = 'stratified'  # every generated structure is an independent draw (own strain, rattle and alloy arrangement)

dataset_manifest = validate_dataset(records)
splits = split_dataset(records, val_fraction=VAL_FRACTION, test_fraction=TEST_FRACTION, seed=SEED, group_by=split_mode)
train_records, val_records, test_records = splits['train'], splits['val'], splits['test']
split_summary = split_report(splits, group_by=split_mode)
write_dataset_xyz(records, OUTPUTS['dataset'])

print({'data_source': data_source, 'n_records': dataset_manifest['n_records'], 'n_atoms': dataset_manifest['n_atoms'], 'compositions': dataset_manifest['compositions']})
print({'elements': dataset_manifest['elements'], 'max_atoms': dataset_manifest['max_atoms'], 'labelled': dataset_manifest['labelled'], 'digest': dataset_manifest['digest'][:16] + '...'})
print({'energy_per_atom_range_eV': (round(min(r['energy'] / len(r['symbols']) for r in records), 4), round(max(r['energy'] / len(r['symbols']) for r in records), 4))})
print({'split_mode': split_summary['mode'], 'kept_together': split_summary['unit'], 'sizes': split_summary['sizes']})
for name in ('train', 'val', 'test'):
    print({name: split_summary['groups'][name]})
if split_summary['mode'] != 'stratified':
    assert not split_summary['groups_in_more_than_one_split'], split_summary['groups_in_more_than_one_split']
print({'written': OUTPUTS['dataset'], 'example': {k: (v if not isinstance(v, list) else f'list[{len(v)}]') for k, v in records[0].items()}})

print({'validation': INPUT_SCHEMA['validation']})
probes = {
    'unsupported element': {'symbols': ['Po', 'Cu'], 'positions': [[0, 0, 0], [2.5, 0, 0]]},
    'overlapping atoms': {'symbols': ['Cu', 'Cu'], 'positions': [[0, 0, 0], [0.3, 0, 0]]},
    'pbc without a cell': {'symbols': ['Cu'], 'positions': [[0, 0, 0]], 'pbc': True},
    'malformed forces': {**records[0], 'forces': [[0.0, 0.0]] * len(records[0]['symbols'])},
}
for name, structure in probes.items():
    try:
        validate_inputs([structure])
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

{'data_source': 'generated fcc supercells labelled by ase.calculators.emt.EMT (seed 42, 48 structures)', 'n_records': 48, 'n_atoms': 1536, 'compositions': {'Cu32': 16, 'Al32': 16, 'Al16Cu16': 16}}
{'elements': ['Al', 'Cu'], 'max_atoms': 32, 'labelled': True, 'digest': 'd2409300e9cfa1a0...'}
{'energy_per_atom_range_eV': (0.0108, 0.5372)}
{'split_mode': 'stratified', 'kept_together': 'composition', 'sizes': {'train': 27, 'val': 9, 'test': 12}}
{'train': ['Al16Cu16', 'Al32', 'Cu32']}
{'val': ['Al16Cu16', 'Al32', 'Cu32']}
{'test': ['Al16Cu16', 'Al32', 'Cu32']}
{'written': 'outputs/mace_materials_sample_dataset.xyz', 'example': {'name': 'Cu32-s+0.0084-r0.05-00', 'symbols': 'list[32]', 'positions': 'list[32]', 'cell': 'list[3]', 'pbc': 'list[3]', 'energy': 0.43281304737949, 'forces': 'list[32]', 'composition': 'Cu32', 'strain': 0.008365607907473024, 'rattle_sigma': 0.05, 'label_source': 'ase.calculators.emt.EMT'}}
{'validation': 'element support, finite coordinates, cell/pbc consistency and 

**What to notice:** 48 structures, three compositions of 16, 1,536 atoms, `split_mode` `stratified` with sizes 27/9/12, every composition in every split, EMT energies of a few tenths of an eV per atom, the written `outputs/mace_materials_sample_dataset.xyz`, and four refusals, each naming the rule.

<details><summary>Check your reasoning</summary>Each composition has 16 structures: 25 % is 4 for the test split, 20 % rounds to 3 for validation, and 9 stay in training, so the splits hold 3 × 9 = 27, 3 × 3 = 9 and 3 × 4 = 12 structures and every composition is in every split. That is right for these structures because each is an independent draw. Had they been 48 frames of three trajectories, the same split would put neighbouring frames on both sides; with `BYOD_SPLIT = 'auto'` the same file without group tags would instead be split by composition, one whole composition per split — and then refused, because the validation and test elements must also appear in training (Cu₃₂ alone in training cannot cover Al).</details>

## 5. Zero-shot prediction and physics checks on the frozen model

`pipe.predict` returns, per structure, the total energy (eV), the per-atom energy, per-atom energy contributions, forces (eV/Å) and — for fully periodic cells — the stress tensor (eV/Å³). The model is the pinned foundation potential, untouched: if this cell runs after an adaptation (a BYOD re-run, for example), it first returns the pipeline to the pinned base weights with `pipe.reset_to_pretrained()` and says so.

Five checks say whether the rebuilt model behaves like an interatomic potential should. **Rotation:** rotating the cell and every atom leaves the energy unchanged and rotates the forces with it (equivariance). **Translation** and **permutation** of atoms change nothing. **Gradient consistency:** the force on one atom equals the central finite difference of the energy along that coordinate, to ~1e-8 eV/Å — the forces are the analytic gradient, not a separate head. **Extensivity:** a 2×1×1 supercell has twice the energy. A rebuilt model with a wrong config would fail these before any accuracy number.

**Predict before running:** which of the five differences do you expect to be at the level of floating-point round-off (around 1e-13), and which one larger? Why?

In [ ]:
import time

import numpy as np

if pipe.adapter is not None:
    print({'reset_to_pretrained': pipe.reset_to_pretrained()})
s0 = test_records[0]
t0 = time.perf_counter()
prediction = pipe.predict(test_records[:8])
print({'structures': len(prediction['results']), 'seconds': round(time.perf_counter() - t0, 2), 'units': prediction['units']})
base = prediction['results'][0]
stress_diag = [round(base['stress'][i][i], 5) for i in range(3)] if base['stress'] is not None else None
print({'name': base['name'], 'energy_eV': round(base['energy'], 4), 'energy_per_atom_eV': round(base['energy_per_atom'], 4), 'reference_eV': round(s0['energy'], 4), 'max_force_eV_A': round(float(np.abs(base['forces']).max()), 4), 'stress_diag_eV_A3': stress_diag})

rng = np.random.default_rng(0)
q, _ = np.linalg.qr(rng.normal(size=(3, 3)))
if np.linalg.det(q) < 0:
    q[:, 0] *= -1
rotated = {**s0, 'positions': (np.array(s0['positions']) @ q.T).tolist(), 'cell': None if s0['cell'] is None else (np.array(s0['cell']) @ q.T).tolist()}
rot = pipe.predict([rotated])['results'][0]
translated = {**s0, 'positions': (np.array(s0['positions']) + 1.234).tolist()}
tr = pipe.predict([translated])['results'][0]
perm = rng.permutation(len(s0['symbols']))
permuted = {**s0, 'symbols': [s0['symbols'][i] for i in perm], 'positions': [s0['positions'][i] for i in perm]}
pm = pipe.predict([permuted])['results'][0]
h = 1e-4
plus, minus = np.array(s0['positions']), np.array(s0['positions'])
atom = min(3, len(s0['symbols']) - 1)
plus[atom, 0] += h
minus[atom, 0] -= h
e_plus = pipe.predict([{**s0, 'positions': plus.tolist()}])['results'][0]['energy']
e_minus = pipe.predict([{**s0, 'positions': minus.tolist()}])['results'][0]['energy']
fd_force = -(e_plus - e_minus) / (2 * h)
periodic = s0['cell'] is not None and all(s0['pbc'])  # the supercell check needs a fully periodic cell
supercell = pipe.predict([from_atoms(to_atoms(s0) * (2, 1, 1))])['results'][0] if periodic else None
physics = {
    'rotation_energy_diff': abs(rot['energy'] - base['energy']),
    'rotation_force_diff': float(np.abs(np.array(rot['forces']) - np.array(base['forces']) @ q.T).max()),
    'translation_energy_diff': abs(tr['energy'] - base['energy']),
    'permutation_energy_diff': abs(pm['energy'] - base['energy']),
    'permutation_force_diff': float(np.abs(np.array(pm['forces']) - np.array(base['forces'])[perm]).max()),
    'finite_difference_force': fd_force,
    'analytic_force': base['forces'][atom][0],
    'gradient_consistency_diff': abs(fd_force - base['forces'][atom][0]),
    'extensivity_diff_per_atom': abs(supercell['energy'] - 2 * base['energy']) / supercell['n_atoms'] if periodic else float('nan'),
    'batch_vs_single_energy_diff': abs(pipe.predict([s0])['results'][0]['energy'] - base['energy']),
}
for key, value in physics.items():
    print({key: f'{value:.3e}'})
assert physics['rotation_energy_diff'] < 1e-8 and physics['rotation_force_diff'] < 1e-8
assert physics['permutation_energy_diff'] < 1e-8 and physics['translation_energy_diff'] < 1e-8
assert physics['gradient_consistency_diff'] < 1e-6
assert not periodic or physics['extensivity_diff_per_atom'] < 1e-8

{'structures': 8, 'seconds': 5.71, 'units': {'energy': 'eV', 'forces': 'eV/Å', 'stress': 'eV/Å³'}}
{'name': 'Cu16Al16-s-0.0200-r0.15-23', 'energy_eV': -121.9445, 'energy_per_atom_eV': -3.8108, 'reference_eV': 10.6741, 'max_force_eV_A': 5.4132, 'stress_diag_eV_A3': [-0.07295, -0.07229, -0.06873]}


{'rotation_energy_diff': '2.842e-14'}
{'rotation_force_diff': '1.843e-14'}
{'translation_energy_diff': '0.000e+00'}
{'permutation_energy_diff': '1.421e-14'}
{'permutation_force_diff': '5.107e-15'}
{'finite_difference_force': '-3.604e+00'}
{'analytic_force': '-3.604e+00'}
{'gradient_consistency_diff': '2.016e-08'}
{'extensivity_diff_per_atom': '0.000e+00'}
{'batch_vs_single_energy_diff': '0.000e+00'}


**What to notice:** the timing line, one structure's energy beside its EMT reference (several eV per atom apart — Section 6 explains why), and ten differences, all small enough for the four assertions to pass.

<details><summary>Check your reasoning</summary>The rotation, translation, permutation, extensivity and batch-versus-single differences come out at the round-off level of float64 arithmetic (0 to about 1e-13) because the model is built to respect those symmetries exactly. The finite-difference check is the larger one, about 1e-8 eV/Å in this repository's CPU check: a central difference with step h = 1e-4 Å has a truncation error of order h², and the energies it subtracts are total energies of hundreds of eV, so cancellation costs digits. Agreement to 1e-8 against forces of order 0.1–1 eV/Å is what "the forces are the gradient of the energy" looks like numerically.</details>

## 6. Baselines and the zero-shot error

Three numbers frame everything that follows. The **composition baseline** fits one energy per element on the training split (the classical E0 regression) and predicts zero forces: it is blind to geometry by construction, so its energy error is the within-composition spread of the labels and its force error is the mean absolute reference force. The **zero-force baseline** is that force number alone. The **frozen foundation model** is evaluated as is (this cell, too, returns to the pinned base weights first if the pipeline was adapted). Energies are per atom (eV/atom); forces are per Cartesian component (eV/Å).

**Predict before running:** the frozen model was trained on PBE (DFT) energies and is scored here against EMT labels. Which will it get closer to the reference — the forces or the energies? Will it beat the composition baseline on energy?

In [ ]:
if pipe.adapter is not None:
    print({'reset_to_pretrained': pipe.reset_to_pretrained()})
baseline_composition = composition_baseline(train_records, test_records)
baseline_zero_force = zero_force_baseline(test_records)
t0 = time.perf_counter()
zero_shot_test = pipe.evaluate(test_records)
zero_shot_seconds = round(time.perf_counter() - t0, 2)
print({'composition_baseline': {'energy_mae_per_atom': round(baseline_composition['energy_mae_per_atom'], 4), 'force_mae': round(baseline_composition['force_mae'], 4), 'e0_eV': {k: round(v, 4) for k, v in baseline_composition['e0_ev'].items()}}})
print({'zero_force_baseline': {'force_mae': round(baseline_zero_force['force_mae'], 4), 'force_rmse': round(baseline_zero_force['force_rmse'], 4)}})
print({'frozen_foundation_model': {k: round(v, 4) for k, v in zero_shot_test.items() if isinstance(v, float)}, 'seconds': zero_shot_seconds})
if not USE_BYOD:  # the sample's recorded result; on your data a frozen model that loses to zero forces is a finding
    assert zero_shot_test['force_mae'] < baseline_zero_force['force_mae']
elif zero_shot_test['force_mae'] >= baseline_zero_force['force_mae']:
    print('On this data the frozen model predicts forces no better than zero: expect adaptation to have a lot to do.')

{'composition_baseline': {'energy_mae_per_atom': 0.0929, 'force_mae': 0.7376, 'e0_eV': {'Al': 0.0866, 'Cu': 0.2295}}}
{'zero_force_baseline': {'force_mae': 0.7376, 'force_rmse': 1.2401}}
{'frozen_foundation_model': {'energy_mae_per_atom': 3.9807, 'energy_rmse_per_atom': 3.986, 'force_mae': 0.2079, 'force_rmse': 0.3498}, 'seconds': 2.06}


**What to notice:** the frozen model's force MAE well below the zero-force baseline, and its energy MAE far above the composition baseline's.

<details><summary>Check your reasoning</summary>Forces. In this repository's CPU check the frozen model's force MAE was 0.2079 eV/Å against 0.7376 eV/Å for predicting zero — it already knows how metal atoms push on each other. Its energy MAE was 3.98 eV/atom against 0.0929 eV/atom for the composition baseline: PBE total energies and EMT energies sit on different absolute references (EMT's zero is close to the isolated-atom reference, PBE's is not), so every structure is off by roughly the same few eV per atom. A per-element offset removes most of that — which is exactly the first step of Section 7. The forces are unaffected by such an offset, because a constant energy per atom has zero gradient.</details>

## 7. Calibrate and fine-tune

The cell first returns the pipeline to the pinned foundation weights (`pipe.reset_to_pretrained()`), so every run of this cell — the default one, a re-run after you change a field, or a BYOD run — starts from the same model; `pipe.adapt` refuses to train a pipeline that is already adapted. `pipe.adapt` then does two things in order. First it **calibrates the per-element reference energies**: a least-squares fit of one offset per element between the training labels and the frozen predictions, added to the model's atomic energies — two numbers on a Cu/Al dataset, and the cheapest possible move to a new level of theory. That calibrated frozen model is recorded as epoch 0 so every later number is comparable to it. Then it trains the readouts, the reference energies, the scale/shift block and — with `TRAINABLE_BLOCKS = 1` — the last interaction and product blocks: 1.92 M of 8.22 M parameters, Adam at a fixed learning rate, loss = 100 × MSE(per-atom energy) + 10 × MSE(force components), batches of 4, no scheduler, seeded shuffling. The epoch with the lowest validation loss is kept.

To try another setting here, change the field, select this cell and choose **Runtime → Run after**: Sections 7–10 run again from the foundation model and overwrite the outputs. Section 10 does the same comparison without touching the main run.

**Predict before running:** how far will the validation energy MAE fall from epoch 0, and will the force MAE fall by the same factor?

In [ ]:
EPOCHS = 8  # @param {type:"integer"}
LEARNING_RATE = 1e-3  # @param {type:"number"}
BATCH_SIZE = 4  # @param {type:"integer"}
TRAINABLE_BLOCKS = 1  # @param {type:"integer"}

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4), 'val_loss': round(entry['val_loss'], 4)}
    if 'val' in entry:
        row['val_energy_mae_per_atom'] = round(entry['val']['energy_mae_per_atom'], 5)
        row['val_force_mae'] = round(entry['val']['force_mae'], 5)
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

print({'reset_to_pretrained': pipe.reset_to_pretrained()})
t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_blocks=TRAINABLE_BLOCKS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'best_epoch': adapt_result['best_epoch'], 'seconds': adapt_seconds})
print({'e0_calibration_eV': dict(zip(adapt_result['calibration']['elements'], [round(s, 4) for s in adapt_result['calibration']['shifts_ev']]))})
print({'trainable_prefixes': adapt_result['trainable_prefixes']})

{'reset_to_pretrained': {'reset_to': 'pinned base weights (converted pair, digest-verified)', 'was_adapted': False}}


{'epoch': 0, 'train_loss': None, 'val_loss': 2.255, 'val_energy_mae_per_atom': 0.09854, 'val_force_mae': 0.19665, 'note': 'frozen model after E0 calibration'}


{'epoch': 1, 'train_loss': 1.5224, 'val_loss': 1.1228, 'val_energy_mae_per_atom': 0.05143, 'val_force_mae': 0.17508}


{'epoch': 2, 'train_loss': 0.6944, 'val_loss': 0.6089, 'val_energy_mae_per_atom': 0.01213, 'val_force_mae': 0.14812}


{'epoch': 3, 'train_loss': 0.4114, 'val_loss': 0.5026, 'val_energy_mae_per_atom': 0.01698, 'val_force_mae': 0.13065}


{'epoch': 4, 'train_loss': 0.3303, 'val_loss': 0.3518, 'val_energy_mae_per_atom': 0.0096, 'val_force_mae': 0.11157}


{'epoch': 5, 'train_loss': 0.2422, 'val_loss': 0.2645, 'val_energy_mae_per_atom': 0.00837, 'val_force_mae': 0.09726}


{'epoch': 6, 'train_loss': 0.1847, 'val_loss': 0.2303, 'val_energy_mae_per_atom': 0.01133, 'val_force_mae': 0.08712}


{'epoch': 7, 'train_loss': 0.1578, 'val_loss': 0.1997, 'val_energy_mae_per_atom': 0.01086, 'val_force_mae': 0.08195}


{'epoch': 8, 'train_loss': 0.1354, 'val_loss': 0.1694, 'val_energy_mae_per_atom': 0.00759, 'val_force_mae': 0.07617}
{'trainable_parameters': 1919128, 'total_parameters': 8221984, 'best_epoch': 8, 'seconds': 25.6}
{'e0_calibration_eV': {'Al': 3.768, 'Cu': 4.1763}}
{'trainable_prefixes': ['readouts.', 'atomic_energies_fn.', 'scale_shift.', 'interactions.1.', 'products.1.']}


**What to notice:** epoch 0 labelled `frozen model after E0 calibration`, the two calibration shifts (several eV each), the validation energy MAE falling by about an order of magnitude, and the force MAE falling by a smaller factor.

<details><summary>Check your reasoning</summary>In this repository's CPU check epoch 0 (the calibrated frozen model) scored 0.0985 eV/atom and 0.1967 eV/Å on validation, after shifts of Al +3.768 and Cu +4.176 eV; the kept epoch (epoch 8) scored 0.0076 eV/atom and 0.0762 eV/Å, with 1,919,128 of 8,221,984 parameters trained in about 66 s. The energy falls further because two offsets plus a retrained readout can absorb most of the systematic PBE-versus-EMT difference, while the forces depend on the curvature of the energy surface, which only the trained interaction block can change. The validation energy MAE does not have to fall every epoch: selection uses the weighted loss (energy and forces together), so a kept epoch can have a slightly worse energy MAE than its neighbour.</details>

## 8. Held-out evaluation

The test split was never used for calibration, training or epoch selection. The adapted numbers are read against the composition baseline (what a model that cannot see geometry achieves), the zero-force baseline and the frozen foundation model from Section 6. Twelve test structures from one seeded split give no dispersion estimate — the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark.

**Predict before running:** will the adapted energy MAE beat the composition baseline, and by how much?

In [ ]:
test_metrics = pipe.evaluate(test_records)
val_metrics = pipe.evaluate(val_records)
print({'test_adapted': {k: round(v, 5) for k, v in test_metrics.items() if isinstance(v, float)}, 'n_structures': test_metrics['n_structures']})
comparison = {
    'energy_mae_per_atom': {'composition_baseline': round(baseline_composition['energy_mae_per_atom'], 5), 'frozen_model': round(zero_shot_test['energy_mae_per_atom'], 5), 'adapted': round(test_metrics['energy_mae_per_atom'], 5)},
    'force_mae': {'zero_force_baseline': round(baseline_zero_force['force_mae'], 5), 'frozen_model': round(zero_shot_test['force_mae'], 5), 'adapted': round(test_metrics['force_mae'], 5)},
}
for metric, values in comparison.items():
    print({metric: values})
evaluation_report = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digest': dataset_manifest['digest'],
    'splits': {'train': len(train_records), 'validation': len(val_records), 'test': len(test_records)},
    'split': split_summary,
    'baselines': {'composition': baseline_composition, 'zero_force': baseline_zero_force, 'frozen_model': zero_shot_test},
    'physics_checks': physics,
    'validation_metrics': val_metrics,
    'test_metrics': test_metrics,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k != 'history'},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
    'uncertainty': 'none: point predictions with no per-prediction uncertainty and no out-of-domain flag',
}
with open(OUTPUTS['report'], 'w', encoding='utf-8') as f:
    json.dump(evaluation_report, f, indent=2)
beats = {'energy_beats_composition_baseline': test_metrics['energy_mae_per_atom'] < baseline_composition['energy_mae_per_atom'], 'force_beats_frozen_model': test_metrics['force_mae'] < zero_shot_test['force_mae']}
print({'reading': beats, 'report': OUTPUTS['report']})
if USE_BYOD:
    # Your data decides; a baseline that wins is a result to report, not an error (see Interpretation, 'Baselines first').
    if not all(beats.values()):
        print('On this data the adapted model does not beat every reference above: read the comparison before using the adapter.')
else:
    # The generated sample's recorded result; a failure here means the run differs from the recorded one.
    assert test_metrics['energy_mae_per_atom'] < baseline_composition['energy_mae_per_atom']
    assert test_metrics['force_mae'] < zero_shot_test['force_mae']

{'test_adapted': {'energy_mae_per_atom': 0.00725, 'energy_rmse_per_atom': 0.00953, 'force_mae': 0.07803, 'force_rmse': 0.12403}, 'n_structures': 12}
{'energy_mae_per_atom': {'composition_baseline': 0.09291, 'frozen_model': 3.98075, 'adapted': 0.00725}}
{'force_mae': {'zero_force_baseline': 0.7376, 'frozen_model': 0.20787, 'adapted': 0.07803}}
{'reading': {'energy_beats_composition_baseline': True, 'force_beats_frozen_model': True}, 'report': 'outputs/mace_materials_evaluation_report.json'}


**What to notice:** the comparison table — adapted energy MAE an order of magnitude below the composition baseline, adapted force MAE well below the frozen model's.

<details><summary>Check your reasoning</summary>In this repository's CPU check (and in the earlier recorded Kaggle T4 run of the same default path) the test energy MAE was 0.00725 eV/atom adapted against 0.09291 for the composition baseline and 3.98 for the frozen model, and the force MAE 0.07803 eV/Å adapted against 0.2079 frozen and 0.7376 for zero forces. Twelve structures from one seeded split of one toy level of theory: the numbers show that the calibration and fine-tuning contract works on held-out structures of the same kind, not how well the model would do on DFT data or on other compositions.</details>

## 9. Inference on new structures, artifact export and fresh reload

For the sample, three structures are generated with a different seed — one per composition — and labelled with EMT so the adapted model's errors on them can be printed alongside its predictions; they were never part of any split. For BYOD there is no labeller in the notebook, so the new structures are three test-split structures with a fresh 0.03 Å rattle: new, unscored geometries, predicted without reference labels. The stress tensor is returned for fully periodic cells. Each prediction is a point value: there is no per-prediction uncertainty and no flag for a structure unlike the training data.

`pipe.save_artifact` writes only the tensors the adaptation could change (readouts, reference energies, scale/shift and the unfrozen blocks) as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digests of the converted base pair, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). The export goes to a staging folder first. `MaceMaterialsPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digest **before** deserialising, and overlays the tensors onto a freshly rebuilt base — a new object from files, not the in-memory model (VER2). Only when the reloaded model reproduces the in-memory energies and forces (VER4) does the staged adapter replace the previous one, so a failed check never deletes a good adapter.

**Predict before running:** will the reloaded model's energies equal the in-memory model's exactly, or only to about 1e-6?

In [ ]:
import random
import shutil

if USE_BYOD:
    new_rng = np.random.default_rng(7)
    new_records = []
    for record in test_records[:3]:
        positions = np.array(record['positions']) + new_rng.normal(0.0, 0.03, size=(len(record['symbols']), 3))
        new_records.append({'name': record['name'] + '-new', 'symbols': record['symbols'], 'positions': positions.tolist(), 'cell': record['cell'], 'pbc': record['pbc']})
    new_source = 'three new geometries: BYOD test-split structures with a fresh 0.03 A rattle (seed 7), unlabelled'
else:
    new_rng = random.Random(7)
    new_records = []
    for composition in SAMPLE_COMPOSITIONS:
        structure = build_sample_structure(composition, strain=new_rng.uniform(-0.03, 0.03), sigma=0.10, rng=new_rng)
        structure['energy'], structure['forces'] = _emt_energy_forces(to_atoms(structure))
        new_records.append(structure)
    new_source = 'freshly generated fcc supercells (seed 7), one per composition'
validate_inputs(new_records)
new_prediction = pipe.predict(new_records)
labelled_new = all(r.get('energy') is not None and r.get('forces') is not None for r in new_records)
print({'new_source': new_source, 'adapted': new_prediction['model']['adapted'], 'uncertainty': 'none (point predictions, no out-of-domain flag)'})
for record, result in zip(new_records, new_prediction['results']):
    row = {'name': result['name'], 'n_atoms': result['n_atoms'], 'energy_per_atom_eV': round(result['energy_per_atom'], 4)}
    if result['stress'] is not None:
        row['pressure_eV_A3'] = round(-sum(result['stress'][i][i] for i in range(3)) / 3, 5)
    if labelled_new:
        row['reference_per_atom_eV'] = round(record['energy'] / result['n_atoms'], 4)
        row['force_mae_eV_A'] = round(float(np.abs(np.array(result['forces']) - np.array(record['forces'])).mean()), 4)
    print(row)
new_metrics = pipe.evaluate(new_records) if labelled_new else None
if new_metrics is not None:
    print({'new_structures': {k: round(v, 5) for k, v in new_metrics.items() if isinstance(v, float)}, 'note': 'sanity check on generated structures, not an evaluation'})

with open(OUTPUTS['predictions'], 'w', encoding='utf-8') as f:
    json.dump({'source': new_source, 'units': new_prediction['units'], 'uncertainty': None, 'results': [{k: v for k, v in r.items() if k != 'node_energies'} for r in new_prediction['results']]}, f, indent=2)

def export_and_reload(model_pipe, target, metadata):
    # Export to a staging folder, reload it into a fresh pipeline, compare, and only then replace `target`.
    target = Path(target)
    staging = target.with_name(target.name + '.staging')
    shutil.rmtree(staging, ignore_errors=True)
    model_pipe.save_artifact(staging, metadata=metadata)
    fresh = MaceMaterialsPipeline.from_artifact(staging, weights_dir=WEIGHTS_DIR, device=model_pipe.device)
    before = model_pipe.predict(test_records[:4])['results']
    after = fresh.predict(test_records[:4])['results']
    check = {
        'max_abs_energy_diff': max(abs(a['energy'] - b['energy']) for a, b in zip(before, after)),
        'max_abs_force_diff': max(float(np.abs(np.array(a['forces']) - np.array(b['forces'])).max()) for a, b in zip(before, after)),
    }
    if not (check['max_abs_energy_diff'] < 1e-9 and check['max_abs_force_diff'] < 1e-9):
        raise AssertionError(f'reload parity failed {check}; the previous {target} was left in place and the export is in {staging}')
    shutil.rmtree(target, ignore_errors=True)
    staging.rename(target)
    return fresh, check, json.loads((target / 'manifest.json').read_text(encoding='utf-8'))

artifact_dir = Path(OUTPUTS['adapter'])
reloaded, parity, artifact_manifest = export_and_reload(pipe, artifact_dir, {'tutorial': 'mace_materials', 'data_source': data_source})
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['max_abs_energy_diff'] < 1e-9 and parity['max_abs_force_diff'] < 1e-9

import platform

import safetensors

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model': {**evaluation_report['model'], 'model_license': MODEL_LICENSE, 'device': pipe.device, 'source': pipe.source},
    'provenance': {
        'source_asset': next(e for e in MANIFEST['files'] if e['path'] == SOURCE_MODEL_NAME),
        'pickle_audit_sha256': PICKLE_AUDIT_SHA256,
        'converted': verify_converted(WEIGHTS_DIR)['files'],
        'pickle_unpickled_once_for_conversion': True,
        'served_from_pickle': False,
        'remote_code_executed': False,
        'torch_force_no_weights_only_load': os.environ.get('TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'),
    },
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'mace': mace.__version__, 'e3nn': e3nn.__version__, 'ase': ase.__version__, 'safetensors': safetensors.__version__},
    'data_source': data_source,
    'split': split_summary,
    'test_metrics': test_metrics,
    'comparison': comparison,
    'new_structures': {'source': new_source, 'metrics': new_metrics},
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes']},
    'reload_parity': parity,
    'uncertainty': 'none: point predictions with no per-prediction uncertainty and no out-of-domain flag',
}
with open(OUTPUTS['result'], 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

print('outputs/:')
for path in sorted(Path('outputs').rglob('*')):
    if path.is_file():
        print(f'  - {path.as_posix()} ({path.stat().st_size / 1024:.1f} KB)')

{'new_source': 'freshly generated fcc supercells (seed 7), one per composition', 'adapted': True, 'uncertainty': 'none (point predictions, no out-of-domain flag)'}
{'name': 'Cu32-s-0.0106-r0.10', 'n_atoms': 32, 'energy_per_atom_eV': 0.1141, 'pressure_eV_A3': 0.05932, 'reference_per_atom_eV': 0.1115, 'force_mae_eV_A': 0.0278}
{'name': 'Al32-s+0.0231-r0.10', 'n_atoms': 32, 'energy_per_atom_eV': 0.0709, 'pressure_eV_A3': -0.02398, 'reference_per_atom_eV': 0.0596, 'force_mae_eV_A': 0.0265}
{'name': 'Cu16Al16-s+0.0074-r0.10', 'n_atoms': 32, 'energy_per_atom_eV': 0.1669, 'pressure_eV_A3': -0.01814, 'reference_per_atom_eV': 0.1548, 'force_mae_eV_A': 0.0786}
{'new_structures': {'energy_mae_per_atom': 0.00866, 'energy_rmse_per_atom': 0.00966, 'force_mae': 0.04432, 'force_rmse': 0.06732}, 'note': 'sanity check on generated structures, not an evaluation'}


{'artifact': 'outputs/mace_materials_adapter', 'format': 'org.valcorza.mace-materials.adapter.v1', 'tensors': 39, 'bytes': 16155811, 'sha256': 'fc8cda7ca0ef6f4c...'}
{'reload_parity': {'max_abs_energy_diff': 1.4210854715202004e-14, 'max_abs_force_diff': 9.298117831235686e-15}, 'reloaded_best_epoch': 8}
outputs/:
  - outputs/mace_materials_adapter/adapter.safetensors (15777.2 KB)
  - outputs/mace_materials_adapter/manifest.json (7.3 KB)
  - outputs/mace_materials_evaluation_report.json (8.2 KB)
  - outputs/mace_materials_predictions.json (12.5 KB)
  - outputs/mace_materials_result.json (4.2 KB)
  - outputs/mace_materials_sample_dataset.xyz (166.9 KB)


**What to notice:** three new structures with their predictions (and, for the sample, their EMT errors), the adapter's tensor count and size, and reload-parity differences that are exactly zero or at round-off.

<details><summary>Check your reasoning</summary>Exactly, on one device. The reloaded model is the same digest-verified base files plus the same adapter tensors (checked against the manifest's SHA-256 before loading), and float64 inference in evaluation mode is deterministic on one CPU, so this repository's CPU check printed 0.0 for both differences (the recorded Kaggle T4 run showed about 1e-14, GPU summation order). The adapter held 39 tensors in 16,155,811 bytes; on the three new sample structures the adapted model's errors were 0.0087 eV/atom and 0.0443 eV/Å. A parity failure would mean the adapter did not capture every change the adaptation made — which is why `adapt` refuses an already adapted pipeline.</details>

## 10. Your turn — change one thing: which blocks train

This activity fine-tunes a **second, separate pipeline** (`activity_pipe`, loaded from the same verified base files) with a different trainable scope, so the main model, its outputs and its adapter stay as they are. It uses the same splits and the Section 7 fields `EPOCHS`, `LEARNING_RATE` and `BATCH_SIZE`; only `ACTIVITY_TRAINABLE_BLOCKS` differs. On **Run all** it trains the readouts alone (`0`: 2,192 parameters), exports that adapter to `outputs/mace_materials_activity_adapter` and checks its reload parity too.

**Predict → Change one thing → Run → Observe → Explain**

1. **Predict.** Training only the readouts (2,192 parameters) instead of the last interaction block as well (1.92 M): will epoch 0 differ between the two runs? Which test error — energy or force — will suffer more?
2. **Change one thing.** Leave `ACTIVITY_TRAINABLE_BLOCKS = 0` for the first pass; afterwards set it to `2` (everything trains).
3. **Run.** Run this cell (on **Run all** it has already run once with `0`).
4. **Observe.** The table prints the Section 7 run and every activity run of this session: trainable parameters, epoch-0 validation errors, the kept epoch, the test errors and the time.
5. **Explain.** Why is epoch 0 the same in every row? Why does the force error barely move when only the readouts train? Is `2` worth its extra time on this small set?

**Optional experiments** (each re-run from the cell named): a different `SEED` in Section 4 (Run after from Section 4) changes which structures are held out; more `EPOCHS` or another `LEARNING_RATE` in Section 7 (Run after from Section 7); your own extended-XYZ data through BYOD (Section 4) — read the composition baseline before the adapted number.

In [ ]:
ACTIVITY_TRAINABLE_BLOCKS = 0  # @param [0, 1, 2]

activity_pipe = MaceMaterialsPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=pipe.device)
t0 = time.perf_counter()
activity_result = activity_pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_blocks=int(ACTIVITY_TRAINABLE_BLOCKS))
activity_seconds = round(time.perf_counter() - t0, 1)
activity_test = activity_pipe.evaluate(test_records)
_, activity_parity, _ = export_and_reload(activity_pipe, OUTPUTS['activity_adapter'], {'tutorial': 'mace_materials', 'activity': 'Section 10', 'data_source': data_source})

def activity_row(label, result, test, seconds):
    epoch0 = result['history'][0]['val']
    return {'run': label, 'trainable_blocks': result['trainable_blocks'], 'trainable_parameters': result['n_trainable'], 'epoch0_val_energy_mae': round(epoch0['energy_mae_per_atom'], 5), 'epoch0_val_force_mae': round(epoch0['force_mae'], 5), 'best_epoch': result['best_epoch'], 'test_energy_mae': round(test['energy_mae_per_atom'], 5), 'test_force_mae': round(test['force_mae'], 5), 'seconds': seconds}

activity_history = globals().get('activity_history', [])
activity_history.append(activity_row(f'activity {len(activity_history) + 1}', activity_result, activity_test, activity_seconds))
print(activity_row('Section 7', adapt_result, test_metrics, adapt_seconds))
for row in activity_history:
    print(row)
start_diff = abs(activity_result['history'][0]['val']['energy_mae_per_atom'] - adapt_result['history'][0]['val']['energy_mae_per_atom'])
print({'epoch0_difference_eV_per_atom': start_diff, 'activity_reload_parity': activity_parity})
assert start_diff < 1e-6, 'both runs must start from the same calibrated frozen model'
del activity_pipe  # the main pipeline `pipe` is unchanged

{'run': 'Section 7', 'trainable_blocks': 1, 'trainable_parameters': 1919128, 'epoch0_val_energy_mae': 0.09854, 'epoch0_val_force_mae': 0.19665, 'best_epoch': 8, 'test_energy_mae': 0.00725, 'test_force_mae': 0.07803, 'seconds': 25.6}
{'run': 'activity 1', 'trainable_blocks': 0, 'trainable_parameters': 2192, 'epoch0_val_energy_mae': 0.09854, 'epoch0_val_force_mae': 0.19665, 'best_epoch': 8, 'test_energy_mae': 0.0904, 'test_force_mae': 0.20745, 'seconds': 20.6}
{'epoch0_difference_eV_per_atom': 9.853229343548264e-16, 'activity_reload_parity': {'max_abs_energy_diff': 1.4210854715202004e-14, 'max_abs_force_diff': 5.9396931817445875e-15}}


**What to notice:** the same epoch-0 validation errors in every row, the activity's reload parity at zero, and the test errors of the readouts-only run beside the Section 7 run.

<details><summary>Check your reasoning</summary>Epoch 0 is the E0-calibrated frozen model, and both runs start from the same verified base weights and the same training split, so it is identical (0.0985 eV/atom, 0.1967 eV/Å in this repository's CPU check; the printed difference is 0). With only the 2,192 readout, E0 and scale/shift parameters training, the test errors were 0.0904 eV/atom and 0.2075 eV/Å (one to two minutes of training on CPU), against 0.00725 and 0.07803 for the Section 7 run: the readouts can rescale the energy, but the forces depend on the geometry-dependent features of the interaction blocks, which stay frozen. Training everything (`2`, all 8.2 M parameters) costs more time per epoch; whether it is better on 27 training structures is what your extra row shows — with one seeded split, a small difference either way is not evidence.</details>

## Interpretation and limits

The frozen foundation potential already predicts EMT forces to roughly a fifth of an eV/Å without ever having seen EMT — that is what a universal potential trained on Materials Project trajectories carries into a new system — but its energies sit several eV per atom away, because PBE and EMT put their zero in different places. Two per-element offsets remove that; a bounded fine-tuning of the readouts and the last interaction block then brings the held-out energy error an order of magnitude below the composition baseline and roughly halves the force error, in about a minute on CPU. That is the claim: the adaptation contract moves a foundation interatomic potential onto a different level of theory from a few dozen labelled structures, and the physics checks show the rebuilt, converted model is still an equivariant potential whose forces are the gradient of its energy.

The test split has 12 generated supercells of two elements, the metrics come from one seeded split with no dispersion estimate, and EMT is a toy level of theory chosen because it runs in the notebook. So a small error here says the contract works, not that the adapted model reproduces DFT for Cu–Al, transfers to other compositions, defects or surfaces, or that it is stable in molecular dynamics — none of which this repository exercises. Fine-tuning a foundation potential on a narrow dataset can also erode its behaviour elsewhere; upstream mitigates that with multi-head replay against the original training data, which is out of scope here.

**No uncertainty.** Every energy, force and stress this notebook prints is a point prediction: there is no per-prediction uncertainty, no committee or ensemble, and no out-of-domain flag. A structure unlike anything in the training data — another chemistry, a surface, a molecule in vacuum — is scored with the same silent confidence as one like the test split, so checking applicability is the user's job.

Three things to carry to real data. **Reference consistency:** every label must come from one code, one functional and one set of settings; mixing references is the most common way to get an unlearnable dataset. **Splits:** structures from one trajectory or one relaxation are near-duplicates — split by trajectory or by composition, never at random over frames; tag your frames with `group=<id>` and keep `BYOD_SPLIT = 'auto'`. **Baselines first:** if the composition baseline is already good, your labels vary with stoichiometry more than with geometry, and the force error is the number to read.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned source asset, audit and convert a pickled checkpoint into a code-free serving pair without executing anything outside the audited allow-lists, rebuild the model from the installed library, validate the demonstrated dataset contract, execute bounded fine-tuning, evaluate against two trivial baselines and the frozen model on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or physical validity beyond the checks shown.

## Troubleshooting

- **Section 1 stops with "needs a Linux x86_64 runtime".** The locked environment uses manylinux wheels; use Google Colab, Kaggle or a Linux Jupyter server.
- **Section 1 fails while downloading.** PyPI or the managed-Python download was interrupted: run the cell again (finished parts are reused). A size/SHA-256 refusal of the `uv` wheel that repeats means the download is being altered.
- **"The isolated environment's Python process exited".** Usually out of memory. Restart the session and choose **Run all**.
- **Section 3 fails with a digest or size mismatch.** A snapshot file was truncated or altered: delete the `weights/` folder in the runtime and run Section 3 again.
- **`this pipeline is already adapted`.** You called `pipe.adapt` yourself after an adaptation; call `pipe.reset_to_pretrained()` first (Section 7 does this for you).
- **BYOD errors.** Each names the failed rule: no file or more than one uploaded; no upload dialog outside Colab (set `BYOD_PATH`); a file that is not `.xyz`/`.extxyz`; a frame without `energy` and `forces`; fewer than 8 or more than 1,000 structures; fewer than 3 groups for a grouped split; a split with fewer than 4 training, 1 validation or 1 test structure; or a validation/test element missing from training. Fix the file or the field (`BYOD_SPLIT`, `VAL_FRACTION`, `TEST_FRACTION`, `SEED`) and **Run after** from Section 4.
- **Reload parity failed.** The adapter in the staging folder does not reproduce the model; the previous adapter is left in place. Re-run from Section 7.

## Glossary

- **Interatomic potential:** a function from atomic positions (and the cell) to a total energy; forces are minus its gradient.
- **Per-atom energy (eV/atom):** the total energy divided by the number of atoms, so structures of different sizes compare.
- **MAE / RMSE:** mean absolute error / root-mean-square error over structures (energy) or force components.
- **Equivariance:** rotating the input rotates the forces the same way and leaves the energy unchanged.
- **Level of theory:** the method that produced the labels (here EMT; MACE-MP-0 was trained on PBE DFT).
- **E0 calibration:** fitting one reference energy per element so the model's energy zero matches the labels'.
- **Composition baseline:** a per-element energy regression that cannot see geometry and predicts zero forces.
- **Group split:** a split that keeps related structures (one trajectory, one composition) on one side.
- **Adapter:** the changed tensors saved separately from the base model and overlaid on it at load time.
- **Reload parity:** the reloaded model reproducing the in-memory model's predictions.
- **Out-of-domain:** a structure unlike the training data; this notebook does not detect it.

## Conclusion (your notes)

Optional; not a required submission. Complete these sentences from your own run:

1. The frozen model's force error was ___ against ___ for zero forces, and its energy error was large because ___.
2. After calibration and fine-tuning the test energy MAE moved from ___ to ___ eV/atom.
3. Training only the readouts changed the force error from ___ to ___, because ___.
4. Before trusting an adapted potential on my own data, I would check ___.

## References

- Repository README: https://github.com/kurtvalcorza/mace-materials-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/mace-materials-pipeline/blob/main/MODEL_CARD.md
- Weights and conversion notes: https://github.com/kurtvalcorza/mace-materials-pipeline/blob/main/docs/WEIGHTS.md
- Hugging Face model repository: https://huggingface.co/mace-foundations/mace-mp-0 (revision `e291ace2bfae073c3ebc7ae2f9479a525989baa7`)
- Upstream release asset (byte-identical): https://github.com/ACEsuit/mace-mp/releases/tag/mace_mp_0b2
- Batatia et al., *A foundation model for atomistic materials chemistry*, arXiv:2401.00096 (2023): https://arxiv.org/abs/2401.00096
- Batatia et al., *MACE: Higher order equivariant message passing neural networks for fast and accurate force fields*, NeurIPS 2022: https://arxiv.org/abs/2206.07697
- ASE EMT calculator: https://wiki.fysik.dtu.dk/ase/ase/calculators/emt.html
- DIMER Notebook Specification 2.2 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)